Version 2 de claude con epocs

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 5.0  (Learning Curve — Curvas de Aprendizaje por Época)
  Correcciones aplicadas (heredadas de v4.0):
      [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
      [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
      [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
      [FIX-4]  Auto-detección GPU             → torch / nvidia-smi / subprocess
      [FIX-5]  Flexibilidad total de dataset  → detección dinámica de columnas
  Nuevas características v5.0:
      [LC-1]   XGBoost  → evals_result() extrae RMSE train/val por iteración
      [LC-2]   LightGBM → evals_result_ extrae RMSE train/val por iteración
      [LC-3]   HistGB   → train_score_ / validation_score_ (R² por árbol)
      [LC-4]   Figura   → media ± σ de los 5 pliegues; Train sólido / Val discontinuo
      [LC-5]   Nueva pestaña '📉 Curva de Aprendizaje' preservando arquitectura X/Y
================================================================================
  Dependencias mínimas:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
  Dependencias GPU (opcionales):
      pip install xgboost lightgbm
      pip install torch  # solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Estado global: modelo entrenado en la sesión actual ───────────────────────
SESION = {
    "modelo":      None,
    "feat_cols":   [],
    "target":      "",
    "parroquia":   "",
    "feat_stats":  {},
}

# Años de pandemia a excluir del entrenamiento
ANOS_PANDEMIA = [2020, 2021]

# ── Separación académica X / Y ────────────────────────────────────────────────
# Variables Objetivo (Y): los 6 contaminantes REMMAQ
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

# Variables de Entrada (X): atmosféricas + cíclicas + lags
FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

# Colores del tema oscuro
COLOR_REAL  = "#3B82F6"   # azul
COLOR_PRED  = "#F97316"   # naranja
COLOR_POS   = "#22C55E"
COLOR_NEG   = "#EF4444"
BG_PLOT     = "#0F172A"
TEXT_PLOT   = "#E2E8F0"
GRID_PLOT   = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    # Vía 1 — nvidia-smi
    try:
        resultado = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if resultado.returncode == 0 and resultado.stdout.strip():
            nombre_gpu = resultado.stdout.strip().split("\n")[0]
            return True, f"GPU detectada (nvidia-smi): {nombre_gpu}"
    except Exception:
        pass

    # Vía 2 — PyTorch CUDA
    try:
        import torch
        if torch.cuda.is_available():
            nombre = torch.cuda.get_device_name(0)
            return True, f"GPU detectada (torch.cuda): {nombre}"
    except ImportError:
        pass

    # Vía 3 — cuPy
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass

    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    if GPU_DISPONIBLE:
        return {"tree_method": "hist", "device": "cuda"}
    return {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
        return df
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            sample = df[c].dropna().astype(str).iloc[:10]
            pd.to_datetime(sample, infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols = df_head.columns.tolist()
        ts = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        msg = f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
        return cols_sin_ts, msg
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(
    df: pd.DataFrame,
    target_col: str,
    timestamp_col: str,
    excluir_pandemia: bool = True,
) -> tuple:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    n_antes = len(df)
    if excluir_pandemia:
        mask_pandemia = df.index.year.isin(ANOS_PANDEMIA)
        df = df[~mask_pandemia]
        n_excluidos = n_antes - len(df)
        if n_excluidos > 0:
            log.info(
                f"[FIX-3] Pandemia excluida: {n_excluidos:,} registros de "
                f"{ANOS_PANDEMIA} eliminados. Quedan {len(df):,} filas."
            )

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    df = df.dropna(subset=[target_col])

    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]

    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")

    return df[feat_cols], df[target_col], feat_cols, df.index


def _dividir_cronologico(X, y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


# ──────────────────────────────────────────────────────────────────────────────
# 6b.  RESOLUCIÓN DE FEATURES X
# ──────────────────────────────────────────────────────────────────────────────

def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols_disponibles = set(df.select_dtypes(include=[np.number]).columns)
    x_base = [c for c in FEATURES_X_BASE if c in cols_disponibles]
    if x_base:
        return [c for c in x_base if c != target_col]
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disponibles
            if c not in excluir or f"{c}_lag" in c]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str):
    """
    Instancia el modelo. Para XGBoost/LightGBM se añade metric='rmse'
    para asegurar que evals_result almacene RMSE (no MSE/logloss).
    """
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=1000, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            early_stopping_rounds=30,
            eval_metric="rmse",          # [LC-1] métrica para evals_result()
            random_state=42, verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=1000, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            metric="rmse",               # [LC-2] métrica para evals_result_
            device=_lgbm_device(), random_state=42, verbose=-1,
        )
    else:  # HistGradientBoosting (default / CPU)
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,     # [LC-3] fracción interna p/ validation_score_
            max_depth=4,
            min_samples_leaf=25,
            learning_rate=0.05,
            l2_regularization=0.3,
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 8.  PIPELINE K-FOLD  (K=5, shuffle=False — respeto al orden temporal)
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    """
    [LC-1/2/3] Extrae el historial de error por iteración del modelo ya ajustado.

    XGBoost  → evals_result()  → claves 'validation_0' (train) / 'validation_1' (val)
    LightGBM → evals_result_   → claves 'training' / 'valid_1'
    HistGB   → train_score_    → R² por árbol añadido
               validation_score_ → R² en fracción interna de validación

    Devuelve (train_hist, val_hist, nombre_metrica).
    Para HistGB el historial es R² (↑ mejor); para XGB/LGB es RMSE (↓ mejor).
    """
    train_hist: list = []
    val_hist:   list = []
    metric_name: str = "Score"

    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()          # dict de dicts
            ks  = list(evals.keys())               # ['validation_0', 'validation_1']
            met = list(evals[ks[0]].keys())[0]    # 'rmse'
            metric_name  = met.upper()
            train_hist   = list(evals[ks[0]][met])
            val_hist     = list(evals[ks[1]][met]) if len(ks) > 1 else []

        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_           # dict de dicts
            ks  = list(evals.keys())               # ['training', 'valid_1']
            met = list(evals[ks[0]].keys())[0]    # 'rmse'
            metric_name  = met.upper()
            train_hist   = list(evals[ks[0]][met])
            val_hist     = list(evals[ks[1]][met]) if len(ks) > 1 else []

        else:  # HistGradientBoosting
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist   = list(modelo.train_score_)
                metric_name  = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)

    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")

    return train_hist, val_hist, metric_name


def _entrenar_kfold(
    df: pd.DataFrame,
    algoritmo: str,
    n_splits: int = 10,
    log_fn=None,
) -> tuple[pd.DataFrame, dict]:
    """
    Bucle K-Fold sobre los 6 contaminantes REMMAQ.

    Ahora devuelve una TUPLA:
        (df_metricas, dict_curvas)

    dict_curvas estructura:
        {
          "PM25": {
            "train":  [[fold1_hist], [fold2_hist], ...],   # lista de K listas
            "val":    [[fold1_hist], [fold2_hist], ...],
            "metric": "RMSE"  |  "R²"
          },
          ...
        }
    """
    if log_fn is None:
        log_fn = log.info

    kf       = KFold(n_splits=n_splits, shuffle=False)
    filas    = []
    cols_df  = set(df.columns)
    all_curves: dict = {}          # [LC-4] contenedor de curvas por contaminante

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible en el dataset — omitido.")
            continue

        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]

        mask_valid = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals = X_vals[mask_valid]
        y_vals = y_vals[mask_valid]

        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue

        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics:   list[dict]  = []
        fold_curves_tr: list[list]  = []   # [LC-4] historial train por fold
        fold_curves_va: list[list]  = []   # [LC-4] historial val   por fold
        metric_name_cv: str         = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            # StandardScaler ajustado SÓLO con el fold de entrenamiento (sin data leakage)
            scaler  = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_va_sc = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo)

            # ── Ajuste + extracción de curvas por algoritmo ───────────────
            if "XGBoost" in algoritmo:
                # [LC-1] eval_set con AMBOS sets → validation_0 = train, validation_1 = val
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
                tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)

            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                # [LC-2] eval_set con AMBOS sets → 'training' y 'valid_1'
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                                lgb.log_evaluation(-1)],
                )
                tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)

            else:  # HistGradientBoosting
                # [LC-3] train_score_ y validation_score_ se generan automáticamente
                modelo.fit(X_tr_sc, y_tr)
                tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)

            if tr_h:
                fold_curves_tr.append(tr_h)
            if va_h:
                fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(
                f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                f"MAE={fold_metrics[-1]['MAE']:.3f}  "
                f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
                f"R²={fold_metrics[-1]['R2']:.3f}"
            )

        # Guardar curvas de este contaminante
        all_curves[contaminante] = {
            "train":  fold_curves_tr,
            "val":    fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":  len(feat_cols),
            "MAE_mean":    mf["MAE"].mean(),
            "MAE_std":     mf["MAE"].std(),
            "RMSE_mean":   mf["RMSE"].mean(),
            "RMSE_std":    mf["RMSE"].std(),
            "R2_mean":     mf["R2"].mean(),
            "R2_std":      mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves          # ← RETORNA TUPLA (métricas, curvas)


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    inicio = df_p.index.max() - pd.Timedelta(days=days)
    df_p   = df_p[df_p.index >= inicio]

    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n   = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                    edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                    edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# [LC-4/5]  FIGURA: CURVA DE APRENDIZAJE POR ÉPOCA / ITERACIÓN
# ══════════════════════════════════════════════════════════════════════════════

def _fig_curva_aprendizaje(
    curvas: dict,
    target_col: str,
    algoritmo: str,
    parroquia: str = "",
) -> plt.Figure:
    """
    Dibuja la curva de aprendizaje (Loss / Error vs Iteración) para el
    contaminante target_col a lo largo de los K=5 pliegues temporales.

    ─── Eje X : Número de iteración / época (boosting round) ───
    ─── Eje Y : Error o función de pérdida (RMSE ↓ ó R² ↑) ────

    Línea continua   = Train  (promedio de K folds + banda ±σ)
    Línea discontinua = Validation (promedio de K folds + banda ±σ)
    Líneas finas semitransparentes = curvas individuales de cada fold

    XGBoost  → RMSE por iteración  (evals_result)    [LC-1]
    LightGBM → RMSE por iteración  (evals_result_)   [LC-2]
    HistGB   → R² por árbol añadido (train/val_score_)[LC-3]
    """
    COLOR_TR = COLOR_REAL   # azul  — Train
    COLOR_VA = COLOR_PRED   # naranja — Validation

    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    # ── Sin datos: mensaje informativo ───────────────────────────────────────
    if not curvas or not curvas.get("train"):
        ax.text(
            0.5, 0.5,
            "No hay datos de curva de aprendizaje para este contaminante.\n"
            "(El modelo no produjo historial por iteración en ningún fold.)",
            ha="center", va="center", color=TEXT_PLOT, fontsize=11,
            transform=ax.transAxes, wrap=True,
        )
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]   # lista de K listas (longitud variable)
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = (metric_name == "R²")

    # ── Alinear longitudes (truncar al fold más corto) ────────────────────────
    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto para graficar.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len = min(lengths)
    x = np.arange(min_len)

    # ── TRAIN: curvas individuales (transparentes) + media ± σ ───────────────
    tr_arr   = np.array([c[:min_len] for c in train_curves])  # (K, min_len)
    tr_mean  = tr_arr.mean(axis=0)
    tr_std   = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_TR, alpha=0.10, lw=0.75, zorder=2)

    ax.plot(x, tr_mean, color=COLOR_TR, lw=2.2,
            label=f"Train — {metric_name}  (μ K-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_TR, zorder=3)

    # ── VALIDATION: curvas individuales + media ± σ ───────────────────────────
    best_iter  = None
    best_label = ""
    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])  # (K, min_len)
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)

        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_VA, alpha=0.10, lw=0.75, zorder=2)

        # Etiqueta diferenciada: HGB usa validación interna, XGB/LGB usa el fold real
        if is_r2:
            va_label = f"Validación interna HGB — {metric_name}  (μ K-Fold)"
        else:
            va_label = f"Validación K-Fold — {metric_name}  (μ K-Fold)"

        ax.plot(x, va_mean, color=COLOR_VA, lw=2.2, linestyle="--",
                label=va_label, zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_VA, zorder=3)

        # Mejor iteración según validación
        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        best_label = (
            f"Mejor iteración: {best_iter}  "
            f"({metric_name} = {best_val:.4f})"
        )
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":",
                   alpha=0.88, label=best_label, zorder=6)
        # Anotación flotante cerca de la línea vertical
        offset = max(1, int(min_len * 0.02))
        ax.annotate(
            f" iter {best_iter}",
            xy=(best_iter, best_val),
            xytext=(best_iter + offset, best_val),
            color="#F59E0B", fontsize=8.5, va="center", zorder=7,
        )

    # ── Línea de convergencia del promedio de train (última iteración) ────────
    ax.axhline(tr_mean[-1], color=COLOR_TR, lw=0.8, linestyle=":",
               alpha=0.40, zorder=1)

    # ── Estilo ────────────────────────────────────────────────────────────────
    algo_label = algoritmo.split("(")[0].strip()
    titulo = (
        f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  K-Fold (K=5)"
    )
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    if is_r2:
        ylabel = "R²  (↑ mejor)"
        nota   = (
            "Nota HGB: la curva de Validación corresponde a la fracción interna del 10 % "
            "usada por early stopping (no al fold de validación K-Fold)."
        )
    else:
        ylabel = f"{metric_name}  (↓ mejor)"
        nota   = (
            "Nota: la banda sombreada representa ±1 desviación estándar entre los "
            f"{len(train_curves)} pliegues temporales (K-Fold, shuffle=False)."
        )

    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(
        framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
        edgecolor=GRID_PLOT, fontsize=9, loc="best",
    )
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    """Aplica el tema oscuro consistente al eje."""
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL  (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:         str,
    csv_upload,
    target_col:        str,
    nombre_modelo:     str,
    algoritmo:         str,
    plot_days:         int,
    train_ratio:       float,
    excluir_pandemia:  bool,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)

    # 6 outputs: metricas, fig_pred, fig_fi, fig_hm, fig_lc, estado
    # VACIO = 4 Nones → "msg" + 4 Nones + estado = 6 valores = 6 outputs ✓
    VACIO = (None, None, None, None)

    try:
        # ── 10.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2  Cargar y limpiar CSV ────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — manejados nativamente / por scaler.")

        # ── 10.3  K-FOLD (K=5) sobre los 6 contaminantes REMMAQ ──────────────
        info(f"Iniciando K-Fold (K=5, shuffle=False) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} → {GPU_MSG}")

        df_clean = pd.concat([X, y], axis=1)

        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        # [LC-4] _entrenar_kfold ahora devuelve (metricas, curvas)
        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=10, log_fn=kf_log
        )
        for l in kfold_logs:
            logs.append(l)

        if kf_resumen.empty:
            warn("K-Fold no produjo resultados — verifica que el dataset tenga los contaminantes.")

        # ── 10.4  Curva de aprendizaje para el target seleccionado ───────────
        info(f"Generando curva de aprendizaje para '{target_col}'…")
        curvas_target = kf_curvas.get(target_col, {})
        fig_lc = _fig_curva_aprendizaje(curvas_target, target_col, algoritmo, parroquia)
        info("Curva de aprendizaje generada.")

        # ── 10.5  Modelo final sobre el target seleccionado (CORREGIDO - SIN DATA LEAKAGE) ────
        info(f"Entrenando modelo final sobre '{target_col}' aplicando Sub-Split Cronológico para Early Stopping…")
        
        # 1. División Temporal Madre (80% Train / 20% Test)
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        
        # 2. SUB-SPLIT CRONOLÓGICO INTERNO (Evita que el Early Stopping contamine el set de prueba final)
        # Reservamos de forma estricta el último 15% del set de entrenamiento para validación interna
        val_ratio_interno = 0.15
        corte_val = int(len(X_tr) * (1 - val_ratio_interno))
        
        X_tr_sub  = X_tr.iloc[:corte_val]
        X_val_sub = X_tr.iloc[corte_val:]
        y_tr_sub  = y_tr.iloc[:corte_val]
        y_val_sub = y_tr.iloc[corte_val:]
        
        info(f"Sub-Split Interno: Train_Sub={len(X_tr_sub):,} · Val_Sub={len(X_val_sub):,} · Test_Final={len(X_te):,}")

        # 3. Escalado seguro ajustado ÚNICAMENTE con el sub-conjunto de entrenamiento
        scaler_final = StandardScaler()
        X_tr_sub_sc  = scaler_final.fit_transform(X_tr_sub)
        X_val_sub_sc = scaler_final.transform(X_val_sub)
        X_te_sc      = scaler_final.transform(X_te)

        # 4. Instanciar y entrenar con el eval_set protegido
        modelo_final = _construir_modelo(algoritmo)
        
        if "XGBoost" in algoritmo:
            # El Early Stopping monitorea val_sub, el examen final (X_te_sc) se mantiene ciego bajo llave
            modelo_final.fit(
                X_tr_sub_sc, y_tr_sub.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)], 
                verbose=False
            )
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_tr_sub_sc, y_tr_sub.values, 
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)],
            )
        else:
            # HistGradientBoosting maneja su validación interna nativamente
            modelo_final.fit(X_tr_sub_sc, y_tr_sub.values)

        # 5. Generar predicciones honestas fuera de muestra
        y_pred_tr = modelo_final.predict(X_tr_sub_sc)
        y_pred_te = modelo_final.predict(X_te_sc)
        info("Modelo final entrenado con éxito sin contaminación de datos.")

        # Re-asociación de variables para que las gráficas posteriores lean el tamaño correcto automáticamente
        X_tr_sc = X_tr_sub_sc
        y_tr = y_tr_sub

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )
        m_tr, m_te = _m(y_tr, y_pred_tr), _m(y_te, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting en modelo final: ΔR² = {gap:.3f}")

# ── 10.6  Markdown de métricas (CORREGIDO Y SIN REPETICIONES) ─────────────────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                r2_color = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** "
                    f"| `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {r2_color} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        # Transformación de R2 a porcentaje de precisión (Accuracy equivalente para el docente)
        acc_tr = max(0.0, m_tr['R2']) * 100
        acc_te = max(0.0, m_te['R2']) * 100

        metricas_md = f"""
## 📊 Surrogate Model — *{parroquia}*

### Validación Cruzada K-Fold (K=5) — 6 Contaminantes REMMAQ

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> **Nota:** K-Fold con `shuffle=False` respeta el orden temporal de la serie.
> `StandardScaler` ajustado únicamente con cada fold de entrenamiento (sin data leakage).
> Ver pestaña **📉 Curva de Aprendizaje** para el progreso por época/iteración.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

| Métrica | 🟦 Train (Entrenamiento) | 🟧 Test (Evaluación Ciega) |
|---------|:--------:|:-------:|
| **Precisión del Modelo (R² %)** | **`{acc_tr:.2f}%`** | **`{acc_te:.2f}%`** |
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R² Score** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |

{"⚠️ **Posible overfitting detectado** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Ajuste óptimo — Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features base | `{len(feat_cols)}` columnas |
| Train / Test | `{len(X_tr):,}` / `{len(X_te):,}` registros |
| Filtro pandemia | {pandemia_label} |
"""

        # ── 10.7  Feature Importance ──────────────────────────────────────────
        info("Calculando Permutation Importance…")
        perm  = permutation_importance(modelo_final, X_te_sc, y_te.values,
                                       n_repeats=8, random_state=42, scoring="r2")
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 10.8  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia, "kfold_resumen": kf_resumen,
                "kf_curvas": kf_curvas,     # [LC] también se guarda el historial
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)

        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 10.9  Guardar en sesión ───────────────────────────────────────────
        feat_stats = {
            col: {"min": float(X[col].min()), "max": float(X[col].max()),
                  "mean": float(X[col].mean())}
            for col in feat_cols
        }
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": feat_stats,
        })
        info("Modelo final en sesión → pestaña Predicción lista.")

        # ── 10.10  Figuras ────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        df_hm       = pd.concat([X, y], axis=1)
        fig_hm      = _fig_heatmap(df_hm, target_col, parroquia)
        info("Heatmap de correlación generado.")

        # ── 6 valores de retorno: métricas, fig_pred, fig_fi, fig_hm, fig_lc, estado ──
        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n"
            f"Instala con: `pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado()
        )


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 12.  CONSTRUCCIÓN DE LA UI
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return ("### ⚠️ No hay modelo entrenado en la sesión.\n"
                "Primero entrena un modelo en la pestaña principal.")
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]

        extra = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    calidad = "🟢 **Buena**"
            elif pred <= 35:  calidad = "🟡 **Moderada**"
            elif pred <= 55:  calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else:             calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice de calidad del aire:** {calidad}"

        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v5",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Modelo sustituto interactivo · MLOps ready · v5.0 — Learning Curves</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════
            # PANEL IZQUIERDO — CONTROLES
            # ══════════════════════════════════
            with gr.Column(scale=1, min_width=350):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")

                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar columnas del CSV", variant="secondary", size="sm"
                    )
                    info_cols = gr.Markdown(
                        "_Pulsa 'Detectar columnas' para ver las variables disponibles._"
                    )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25", allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )

                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina registros de 2020 y 2021 para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Proporción de entrenamiento",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar (Test)",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════
            # PANEL DERECHO — RESULTADOS (6 tabs)
            # ══════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    # Tab 1: Métricas ─────────────────────────────────────────
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    # Tab 2: Real vs Predicho ─────────────────────────────────
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test"
                        )

                    # ── [LC-5]  Tab 3: CURVA DE APRENDIZAJE ──────────────────
                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso del entrenamiento** por época / boosting round. "
                            "La línea **continua** es el error de entrenamiento y la "
                            "línea **discontinua** es el error de validación. "
                            "La banda sombreada representa **±1σ** entre los 5 pliegues temporales.\n\n"
                            "- **XGBoost / LightGBM** → RMSE por iteración (`evals_result`)\n"
                            "- **HistGradientBoosting** → R² por árbol añadido "
                            "(`train_score_` / `validation_score_`)"
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Error vs Iteración (K=5 folds)"
                        )

                    # Tab 4: Feature Importance ───────────────────────────────
                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(
                            label="Importancia de variables (Permutation Δ R²)"
                        )

                    # Tab 5: Análisis de Dependencias ─────────────────────────
                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La fila/columna del target aparece resaltada en naranja. "
                            "Valores cercanos a ±1 indican fuerte dependencia lineal."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    # Tab 6: Consulta / Predicción ────────────────────────────
                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "del contaminante target usando el **modelo en memoria**. "
                            "Los sliders se actualizan con los rangos del dataset entrenado."
                        )
                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals     = gr.Textbox(visible=False, value="{}")
                        btn_predecir  = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v5.0  ·  HGB / XGBoost / LightGBM  ·  Auto-GPU  ·  Learning Curves
        </div>
        """)

        # ── Funciones de actualización de la pestaña Predicción ──────────────

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")

            sesion_msg = (
                f"✅ Modelo en sesión: **{target}** · *{parroquia}* · "
                f"{len(feat_cols)} features disponibles."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )

            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col,
                        value=round(st["mean"], 3),
                        visible=True,
                        interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            import json
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta:
                return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices   = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        btn_detectar.click(
            fn=_detectar_wrapper,
            inputs=[csv_dropdown, csv_upload],
            outputs=[target_input, info_cols],
        )

        # ── Entrenamiento → 6 salidas + estado + refresh sliders ─────────────
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider, excluir_pandemia_chk,
            ],
            outputs=[
                metricas_output,
                fig_pred_output,
                fig_fi_output,
                fig_hm_output,
                fig_lc_output,    # ← [LC-5] nueva salida
                estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        # ── Predicción: empaquetar sliders → JSON → predecir ─────────────────
        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip_local = s.getsockname()[0]
        s.close()
        print(f"  🖥️  IP local   : http://{ip_local}:<PUERTO>")
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 62)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v5.0")
    print("  📉  Learning Curves — Curvas de Aprendizaje por Época")
    print("═" * 62)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 62)
    print("\n⏳ Generando link de acceso externo... por favor espera.\n")

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

version estable pero con sobre ajuste con su curva de aprendizaje no tocar ni cambiar

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 5.0  (Learning Curve — Curvas de Aprendizaje por Época)
  Correcciones aplicadas (heredadas de v4.0):
      [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
      [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
      [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
      [FIX-4]  Auto-detección GPU             → torch / nvidia-smi / subprocess
      [FIX-5]  Flexibilidad total de dataset  → detección dinámica de columnas
  Nuevas características v5.0:
      [LC-1]   XGBoost  → evals_result() extrae RMSE train/val por iteración
      [LC-2]   LightGBM → evals_result_ extrae RMSE train/val por iteración
      [LC-3]   HistGB   → train_score_ / validation_score_ (R² por árbol)
      [LC-4]   Figura   → media ± σ de los 5 pliegues; Train sólido / Val discontinuo
      [LC-5]   Nueva pestaña '📉 Curva de Aprendizaje' preservando arquitectura X/Y
================================================================================
  Dependencias mínimas:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
  Dependencias GPU (opcionales):
      pip install xgboost lightgbm
      pip install torch  # solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Estado global: modelo entrenado en la sesión actual ───────────────────────
SESION = {
    "modelo":      None,
    "feat_cols":   [],
    "target":      "",
    "parroquia":   "",
    "feat_stats":  {},
}

# Años de pandemia a excluir del entrenamiento
ANOS_PANDEMIA = [2020, 2021]

# ── Separación académica X / Y ────────────────────────────────────────────────
# Variables Objetivo (Y): los 6 contaminantes REMMAQ
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

# Variables de Entrada (X): atmosféricas + cíclicas + lags
FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

# Colores del tema oscuro
COLOR_REAL  = "#3B82F6"   # azul
COLOR_PRED  = "#F97316"   # naranja
COLOR_POS   = "#22C55E"
COLOR_NEG   = "#EF4444"
BG_PLOT     = "#0F172A"
TEXT_PLOT   = "#E2E8F0"
GRID_PLOT   = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    # Vía 1 — nvidia-smi
    try:
        resultado = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if resultado.returncode == 0 and resultado.stdout.strip():
            nombre_gpu = resultado.stdout.strip().split("\n")[0]
            return True, f"GPU detectada (nvidia-smi): {nombre_gpu}"
    except Exception:
        pass

    # Vía 2 — PyTorch CUDA
    try:
        import torch
        if torch.cuda.is_available():
            nombre = torch.cuda.get_device_name(0)
            return True, f"GPU detectada (torch.cuda): {nombre}"
    except ImportError:
        pass

    # Vía 3 — cuPy
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass

    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    if GPU_DISPONIBLE:
        return {"tree_method": "hist", "device": "cuda"}
    return {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
        return df
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            sample = df[c].dropna().astype(str).iloc[:10]
            pd.to_datetime(sample, infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols = df_head.columns.tolist()
        ts = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        msg = f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
        return cols_sin_ts, msg
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(
    df: pd.DataFrame,
    target_col: str,
    timestamp_col: str,
    excluir_pandemia: bool = True,
) -> tuple:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    n_antes = len(df)
    if excluir_pandemia:
        mask_pandemia = df.index.year.isin(ANOS_PANDEMIA)
        df = df[~mask_pandemia]
        n_excluidos = n_antes - len(df)
        if n_excluidos > 0:
            log.info(
                f"[FIX-3] Pandemia excluida: {n_excluidos:,} registros de "
                f"{ANOS_PANDEMIA} eliminados. Quedan {len(df):,} filas."
            )

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    df = df.dropna(subset=[target_col])

    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]

    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")

    return df[feat_cols], df[target_col], feat_cols, df.index


def _dividir_cronologico(X, y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


# ──────────────────────────────────────────────────────────────────────────────
# 6b.  RESOLUCIÓN DE FEATURES X
# ──────────────────────────────────────────────────────────────────────────────

def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols_disponibles = set(df.select_dtypes(include=[np.number]).columns)
    x_base = [c for c in FEATURES_X_BASE if c in cols_disponibles]
    if x_base:
        return [c for c in x_base if c != target_col]
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disponibles
            if c not in excluir or f"{c}_lag" in c]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str):
    """
    Instancia el modelo. Para XGBoost/LightGBM se añade metric='rmse'
    para asegurar que evals_result almacene RMSE (no MSE/logloss).
    """
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=1000, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            early_stopping_rounds=30,
            eval_metric="rmse",          # [LC-1] métrica para evals_result()
            random_state=42, verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=1000, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            metric="rmse",               # [LC-2] métrica para evals_result_
            device=_lgbm_device(), random_state=42, verbose=-1,
        )
    else:  # HistGradientBoosting (default / CPU)
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,     # [LC-3] fracción interna p/ validation_score_
            max_depth=4,
            min_samples_leaf=25,
            learning_rate=0.05,
            l2_regularization=0.3,
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 8.  PIPELINE K-FOLD  (K=5, shuffle=False — respeto al orden temporal)
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    """
    [LC-1/2/3] Extrae el historial de error por iteración del modelo ya ajustado.

    XGBoost  → evals_result()  → claves 'validation_0' (train) / 'validation_1' (val)
    LightGBM → evals_result_   → claves 'training' / 'valid_1'
    HistGB   → train_score_    → R² por árbol añadido
               validation_score_ → R² en fracción interna de validación

    Devuelve (train_hist, val_hist, nombre_metrica).
    Para HistGB el historial es R² (↑ mejor); para XGB/LGB es RMSE (↓ mejor).
    """
    train_hist: list = []
    val_hist:   list = []
    metric_name: str = "Score"

    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()          # dict de dicts
            ks  = list(evals.keys())               # ['validation_0', 'validation_1']
            met = list(evals[ks[0]].keys())[0]    # 'rmse'
            metric_name  = met.upper()
            train_hist   = list(evals[ks[0]][met])
            val_hist     = list(evals[ks[1]][met]) if len(ks) > 1 else []

        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_           # dict de dicts
            ks  = list(evals.keys())               # ['training', 'valid_1']
            met = list(evals[ks[0]].keys())[0]    # 'rmse'
            metric_name  = met.upper()
            train_hist   = list(evals[ks[0]][met])
            val_hist     = list(evals[ks[1]][met]) if len(ks) > 1 else []

        else:  # HistGradientBoosting
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist   = list(modelo.train_score_)
                metric_name  = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)

    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")

    return train_hist, val_hist, metric_name


def _entrenar_kfold(
    df: pd.DataFrame,
    algoritmo: str,
    n_splits: int = 10,
    log_fn=None,
) -> tuple[pd.DataFrame, dict]:
    """
    Bucle K-Fold sobre los 6 contaminantes REMMAQ.

    Ahora devuelve una TUPLA:
        (df_metricas, dict_curvas)

    dict_curvas estructura:
        {
          "PM25": {
            "train":  [[fold1_hist], [fold2_hist], ...],   # lista de K listas
            "val":    [[fold1_hist], [fold2_hist], ...],
            "metric": "RMSE"  |  "R²"
          },
          ...
        }
    """
    if log_fn is None:
        log_fn = log.info

    kf       = KFold(n_splits=n_splits, shuffle=False)
    filas    = []
    cols_df  = set(df.columns)
    all_curves: dict = {}          # [LC-4] contenedor de curvas por contaminante

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible en el dataset — omitido.")
            continue

        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]

        mask_valid = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals = X_vals[mask_valid]
        y_vals = y_vals[mask_valid]

        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue

        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics:   list[dict]  = []
        fold_curves_tr: list[list]  = []   # [LC-4] historial train por fold
        fold_curves_va: list[list]  = []   # [LC-4] historial val   por fold
        metric_name_cv: str         = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            # StandardScaler ajustado SÓLO con el fold de entrenamiento (sin data leakage)
            scaler  = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_va_sc = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo)

            # ── Ajuste + extracción de curvas por algoritmo ───────────────
            if "XGBoost" in algoritmo:
                # [LC-1] eval_set con AMBOS sets → validation_0 = train, validation_1 = val
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
                tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)

            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                # [LC-2] eval_set con AMBOS sets → 'training' y 'valid_1'
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                                lgb.log_evaluation(-1)],
                )
                tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)

            else:  # HistGradientBoosting
                # [LC-3] train_score_ y validation_score_ se generan automáticamente
                modelo.fit(X_tr_sc, y_tr)
                tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)

            if tr_h:
                fold_curves_tr.append(tr_h)
            if va_h:
                fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(
                f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                f"MAE={fold_metrics[-1]['MAE']:.3f}  "
                f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
                f"R²={fold_metrics[-1]['R2']:.3f}"
            )

        # Guardar curvas de este contaminante
        all_curves[contaminante] = {
            "train":  fold_curves_tr,
            "val":    fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":  len(feat_cols),
            "MAE_mean":    mf["MAE"].mean(),
            "MAE_std":     mf["MAE"].std(),
            "RMSE_mean":   mf["RMSE"].mean(),
            "RMSE_std":    mf["RMSE"].std(),
            "R2_mean":     mf["R2"].mean(),
            "R2_std":      mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves          # ← RETORNA TUPLA (métricas, curvas)


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    inicio = df_p.index.max() - pd.Timedelta(days=days)
    df_p   = df_p[df_p.index >= inicio]

    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n   = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                    edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                    edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# [LC-4/5]  FIGURA: CURVA DE APRENDIZAJE POR ÉPOCA / ITERACIÓN
# ══════════════════════════════════════════════════════════════════════════════

def _fig_curva_aprendizaje(
    curvas: dict,
    target_col: str,
    algoritmo: str,
    parroquia: str = "",
) -> plt.Figure:
    """
    Dibuja la curva de aprendizaje (Loss / Error vs Iteración) para el
    contaminante target_col a lo largo de los K=5 pliegues temporales.

    ─── Eje X : Número de iteración / época (boosting round) ───
    ─── Eje Y : Error o función de pérdida (RMSE ↓ ó R² ↑) ────

    Línea continua   = Train  (promedio de K folds + banda ±σ)
    Línea discontinua = Validation (promedio de K folds + banda ±σ)
    Líneas finas semitransparentes = curvas individuales de cada fold

    XGBoost  → RMSE por iteración  (evals_result)    [LC-1]
    LightGBM → RMSE por iteración  (evals_result_)   [LC-2]
    HistGB   → R² por árbol añadido (train/val_score_)[LC-3]
    """
    COLOR_TR = COLOR_REAL   # azul  — Train
    COLOR_VA = COLOR_PRED   # naranja — Validation

    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    # ── Sin datos: mensaje informativo ───────────────────────────────────────
    if not curvas or not curvas.get("train"):
        ax.text(
            0.5, 0.5,
            "No hay datos de curva de aprendizaje para este contaminante.\n"
            "(El modelo no produjo historial por iteración en ningún fold.)",
            ha="center", va="center", color=TEXT_PLOT, fontsize=11,
            transform=ax.transAxes, wrap=True,
        )
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]   # lista de K listas (longitud variable)
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = (metric_name == "R²")

    # ── Alinear longitudes (truncar al fold más corto) ────────────────────────
    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto para graficar.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len = min(lengths)
    x = np.arange(min_len)

    # ── TRAIN: curvas individuales (transparentes) + media ± σ ───────────────
    tr_arr   = np.array([c[:min_len] for c in train_curves])  # (K, min_len)
    tr_mean  = tr_arr.mean(axis=0)
    tr_std   = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_TR, alpha=0.10, lw=0.75, zorder=2)

    ax.plot(x, tr_mean, color=COLOR_TR, lw=2.2,
            label=f"Train — {metric_name}  (μ K-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_TR, zorder=3)

    # ── VALIDATION: curvas individuales + media ± σ ───────────────────────────
    best_iter  = None
    best_label = ""
    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])  # (K, min_len)
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)

        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_VA, alpha=0.10, lw=0.75, zorder=2)

        # Etiqueta diferenciada: HGB usa validación interna, XGB/LGB usa el fold real
        if is_r2:
            va_label = f"Validación interna HGB — {metric_name}  (μ K-Fold)"
        else:
            va_label = f"Validación K-Fold — {metric_name}  (μ K-Fold)"

        ax.plot(x, va_mean, color=COLOR_VA, lw=2.2, linestyle="--",
                label=va_label, zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_VA, zorder=3)

        # Mejor iteración según validación
        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        best_label = (
            f"Mejor iteración: {best_iter}  "
            f"({metric_name} = {best_val:.4f})"
        )
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":",
                   alpha=0.88, label=best_label, zorder=6)
        # Anotación flotante cerca de la línea vertical
        offset = max(1, int(min_len * 0.02))
        ax.annotate(
            f" iter {best_iter}",
            xy=(best_iter, best_val),
            xytext=(best_iter + offset, best_val),
            color="#F59E0B", fontsize=8.5, va="center", zorder=7,
        )

    # ── Línea de convergencia del promedio de train (última iteración) ────────
    ax.axhline(tr_mean[-1], color=COLOR_TR, lw=0.8, linestyle=":",
               alpha=0.40, zorder=1)

    # ── Estilo ────────────────────────────────────────────────────────────────
    algo_label = algoritmo.split("(")[0].strip()
    titulo = (
        f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  K-Fold (K=5)"
    )
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    if is_r2:
        ylabel = "R²  (↑ mejor)"
        nota   = (
            "Nota HGB: la curva de Validación corresponde a la fracción interna del 10 % "
            "usada por early stopping (no al fold de validación K-Fold)."
        )
    else:
        ylabel = f"{metric_name}  (↓ mejor)"
        nota   = (
            "Nota: la banda sombreada representa ±1 desviación estándar entre los "
            f"{len(train_curves)} pliegues temporales (K-Fold, shuffle=False)."
        )

    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(
        framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
        edgecolor=GRID_PLOT, fontsize=9, loc="best",
    )
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    """Aplica el tema oscuro consistente al eje."""
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL  (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:         str,
    csv_upload,
    target_col:        str,
    nombre_modelo:     str,
    algoritmo:         str,
    plot_days:         int,
    train_ratio:       float,
    excluir_pandemia:  bool,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)

    # 6 outputs: metricas, fig_pred, fig_fi, fig_hm, fig_lc, estado
    # VACIO = 4 Nones → "msg" + 4 Nones + estado = 6 valores = 6 outputs ✓
    VACIO = (None, None, None, None)

    try:
        # ── 10.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2  Cargar y limpiar CSV ────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — manejados nativamente / por scaler.")

        # ── 10.3  K-FOLD (K=5) sobre los 6 contaminantes REMMAQ ──────────────
        info(f"Iniciando K-Fold (K=5, shuffle=False) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} → {GPU_MSG}")

        df_clean = pd.concat([X, y], axis=1)

        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        # [LC-4] _entrenar_kfold ahora devuelve (metricas, curvas)
        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=10, log_fn=kf_log
        )
        for l in kfold_logs:
            logs.append(l)

        if kf_resumen.empty:
            warn("K-Fold no produjo resultados — verifica que el dataset tenga los contaminantes.")

        # ── 10.4  Curva de aprendizaje para el target seleccionado ───────────
        info(f"Generando curva de aprendizaje para '{target_col}'…")
        curvas_target = kf_curvas.get(target_col, {})
        fig_lc = _fig_curva_aprendizaje(curvas_target, target_col, algoritmo, parroquia)
        info("Curva de aprendizaje generada.")

        # ── 10.5  Modelo final sobre el target seleccionado (para figuras) ────
        info(f"Entrenando modelo final sobre '{target_col}' para visualizaciones…")
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)

        scaler_final = StandardScaler()
        X_tr_sc = scaler_final.fit_transform(X_tr)
        X_te_sc  = scaler_final.transform(X_te)

        modelo_final = _construir_modelo(algoritmo)
        if "XGBoost" in algoritmo:
            modelo_final.fit(X_tr_sc, y_tr.values,
                             eval_set=[(X_te_sc, y_te.values)], verbose=False)
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_tr_sc, y_tr.values, eval_set=[(X_te_sc, y_te.values)],
                callbacks=[lgb.early_stopping(30, verbose=False),
                            lgb.log_evaluation(-1)],
            )
        else:
            modelo_final.fit(X_tr_sc, y_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)
        info("Modelo final entrenado.")

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )
        m_tr, m_te = _m(y_tr, y_pred_tr), _m(y_te, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting en modelo final: ΔR² = {gap:.3f}")

        # ── 10.6  Markdown de métricas ────────────────────────────────────────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                r2_color = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** "
                    f"| `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {r2_color} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        metricas_md = f"""
## 📊 Surrogate Model — *{parroquia}*

### Validación Cruzada K-Fold (K=5) — 6 Contaminantes REMMAQ

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> **Nota:** K-Fold con `shuffle=False` respeta el orden temporal de la serie.
> `StandardScaler` ajustado únicamente con cada fold de entrenamiento (sin data leakage).
> Ver pestaña **📉 Curva de Aprendizaje** para el progreso por época/iteración.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features base | `{len(feat_cols)}` columnas |
| Train / Test | `{len(X_tr):,}` / `{len(X_te):,}` registros |
| Filtro pandemia | {pandemia_label} |
"""

        # ── 10.7  Feature Importance ──────────────────────────────────────────
        info("Calculando Permutation Importance…")
        perm  = permutation_importance(modelo_final, X_te_sc, y_te.values,
                                       n_repeats=8, random_state=42, scoring="r2")
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 10.8  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia, "kfold_resumen": kf_resumen,
                "kf_curvas": kf_curvas,     # [LC] también se guarda el historial
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)

        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 10.9  Guardar en sesión ───────────────────────────────────────────
        feat_stats = {
            col: {"min": float(X[col].min()), "max": float(X[col].max()),
                  "mean": float(X[col].mean())}
            for col in feat_cols
        }
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": feat_stats,
        })
        info("Modelo final en sesión → pestaña Predicción lista.")

        # ── 10.10  Figuras ────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        df_hm       = pd.concat([X, y], axis=1)
        fig_hm      = _fig_heatmap(df_hm, target_col, parroquia)
        info("Heatmap de correlación generado.")

        # ── 6 valores de retorno: métricas, fig_pred, fig_fi, fig_hm, fig_lc, estado ──
        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n"
            f"Instala con: `pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado()
        )


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 12.  CONSTRUCCIÓN DE LA UI
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return ("### ⚠️ No hay modelo entrenado en la sesión.\n"
                "Primero entrena un modelo en la pestaña principal.")
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]

        extra = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    calidad = "🟢 **Buena**"
            elif pred <= 35:  calidad = "🟡 **Moderada**"
            elif pred <= 55:  calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else:             calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice de calidad del aire:** {calidad}"

        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v5",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Modelo sustituto interactivo · MLOps ready · v5.0 — Learning Curves</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════
            # PANEL IZQUIERDO — CONTROLES
            # ══════════════════════════════════
            with gr.Column(scale=1, min_width=350):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")

                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar columnas del CSV", variant="secondary", size="sm"
                    )
                    info_cols = gr.Markdown(
                        "_Pulsa 'Detectar columnas' para ver las variables disponibles._"
                    )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25", allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )

                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina registros de 2020 y 2021 para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Proporción de entrenamiento",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar (Test)",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════
            # PANEL DERECHO — RESULTADOS (6 tabs)
            # ══════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    # Tab 1: Métricas ─────────────────────────────────────────
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    # Tab 2: Real vs Predicho ─────────────────────────────────
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test"
                        )

                    # ── [LC-5]  Tab 3: CURVA DE APRENDIZAJE ──────────────────
                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso del entrenamiento** por época / boosting round. "
                            "La línea **continua** es el error de entrenamiento y la "
                            "línea **discontinua** es el error de validación. "
                            "La banda sombreada representa **±1σ** entre los 5 pliegues temporales.\n\n"
                            "- **XGBoost / LightGBM** → RMSE por iteración (`evals_result`)\n"
                            "- **HistGradientBoosting** → R² por árbol añadido "
                            "(`train_score_` / `validation_score_`)"
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Error vs Iteración (K=5 folds)"
                        )

                    # Tab 4: Feature Importance ───────────────────────────────
                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(
                            label="Importancia de variables (Permutation Δ R²)"
                        )

                    # Tab 5: Análisis de Dependencias ─────────────────────────
                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La fila/columna del target aparece resaltada en naranja. "
                            "Valores cercanos a ±1 indican fuerte dependencia lineal."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    # Tab 6: Consulta / Predicción ────────────────────────────
                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "del contaminante target usando el **modelo en memoria**. "
                            "Los sliders se actualizan con los rangos del dataset entrenado."
                        )
                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals     = gr.Textbox(visible=False, value="{}")
                        btn_predecir  = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v5.0  ·  HGB / XGBoost / LightGBM  ·  Auto-GPU  ·  Learning Curves
        </div>
        """)

        # ── Funciones de actualización de la pestaña Predicción ──────────────

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")

            sesion_msg = (
                f"✅ Modelo en sesión: **{target}** · *{parroquia}* · "
                f"{len(feat_cols)} features disponibles."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )

            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col,
                        value=round(st["mean"], 3),
                        visible=True,
                        interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            import json
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta:
                return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices   = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        btn_detectar.click(
            fn=_detectar_wrapper,
            inputs=[csv_dropdown, csv_upload],
            outputs=[target_input, info_cols],
        )

        # ── Entrenamiento → 6 salidas + estado + refresh sliders ─────────────
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider, excluir_pandemia_chk,
            ],
            outputs=[
                metricas_output,
                fig_pred_output,
                fig_fi_output,
                fig_hm_output,
                fig_lc_output,    # ← [LC-5] nueva salida
                estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        # ── Predicción: empaquetar sliders → JSON → predecir ─────────────────
        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip_local = s.getsockname()[0]
        s.close()
        print(f"  🖥️  IP local   : http://{ip_local}:<PUERTO>")
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 62)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v5.0")
    print("  📉  Learning Curves — Curvas de Aprendizaje por Época")
    print("═" * 62)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 62)
    print("\n⏳ Generando link de acceso externo... por favor espera.\n")

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

03:14:42 [INFO] Archivo subido: dataset_ml_carapungo.csv
03:14:43 [INFO] CSV cargado: 160,262 filas × 19 columnas
03:14:43 [INFO] Columna de tiempo: 'Timestamp'
03:14:43 [INFO] [FIX-3] Pandemia excluida: 23 registros de [2020, 2021] eliminados. Quedan 160,239 filas.
03:14:43 [INFO] Pandemia excluida: años [2020, 2021] eliminados.
03:14:43 [WARNING] 217,722 NaN en features — manejados nativamente / por scaler.
03:14:43 [INFO] Iniciando K-Fold (K=5, shuffle=False) · Algoritmo: HistGradientBoosting (CPU — sin GPU requerida)
03:14:43 [INFO] GPU disponible: True → GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
03:14:44 [INFO]   PM25 | Fold 1/5 → MAE=5.729  RMSE=9.057  R²=0.522
03:14:44 [INFO]   PM25 | Fold 2/5 → MAE=5.827  RMSE=8.573  R²=0.539
03:14:45 [INFO]   PM25 | Fold 3/5 → MAE=5.382  RMSE=8.398  R²=0.454
03:14:46 [INFO]   PM25 | Fold 4/5 → MAE=6.174  RMSE=13.226  R²=0.376
03:14:47 [INFO]   PM25 | Fold 5/5 → MAE=6.611  RMSE=9.919  R²=0.280
03:14:49 [INFO]   PM10 | Fold 1/5

Nuevas versiones del codigo anterior

In [5]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 9.0  (TimeSeriesSplit + Blind Test + Penalized Regularization)
  Correcciones heredadas de v5.0:
      [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
      [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
      [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
      [FIX-4]  Auto-detección GPU             → torch / nvidia-smi / subprocess
      [FIX-5]  Flexibilidad total de dataset  → detección dinámica de columnas
  Refactorización v9.0:
      [A]  KFold → TimeSeriesSplit(n_splits=10, shuffle=False implícito)
      [B]  Sub-split cronológico interno (85/15) → eval_set limpio, test 100% ciego
      [B2] Fila "Precisión del Modelo (R² %)" en tabla de métricas
      [C]  Regularización penalizada: XGB/LGBM reg_lambda=5, reg_alpha=1; HGB l2=1.0
================================================================================
  Dependencias mínimas:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
  Dependencias GPU (opcionales):
      pip install xgboost lightgbm
      pip install torch  # solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit   # [A] reemplaza KFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE \
           else {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS ( líneas con # )
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols  = df_head.columns.tolist()
        ts    = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        return cols_sin_ts, f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(
    df: pd.DataFrame,
    target_col: str,
    timestamp_col: str,
    excluir_pandemia: bool = True,
) -> tuple:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_exc = n_antes - len(df)
        if n_exc:
            log.info(f"[FIX-3] {n_exc:,} registros de {ANOS_PANDEMIA} eliminados.")

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    df = df.dropna(subset=[target_col])
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]

    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")

    return df[feat_cols], df[target_col], feat_cols, df.index


def _dividir_cronologico(X, y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols_disp = set(df.select_dtypes(include=[np.number]).columns)
    x_base    = [c for c in FEATURES_X_BASE if c in cols_disp]
    if x_base:
        return [c for c in x_base if c != target_col]
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disp if c not in excluir or f"{c}_lag" in c]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  [C]  CONSTRUCCIÓN DEL MODELO CON REGULARIZACIÓN PENALIZADA
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str):
    """
    [C] Regularización estructural penalizada para los tres algoritmos:
      · XGBoost  → reg_lambda=5.0, reg_alpha=1.0 (L2 + L1 en hojas y pesos)
      · LightGBM → reg_lambda=5.0, reg_alpha=1.0 (equivalentes)
      · HistGB   → l2_regularization=1.0          (aumentado de 0.3 → 1.0)
    Combinados con max_depth=4 y min_samples_leaf=25 frenan la memorización
    de los lags de 1h y fuerzan al modelo a aprender patrones meteorológicos
    generales.
    """
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=1000,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5.0,              # [C] L2 — penaliza pesos grandes en hojas
            reg_alpha=1.0,               # [C] L1 — induce dispersión (feature selection)
            early_stopping_rounds=30,
            eval_metric="rmse",
            random_state=42,
            verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=1000,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5.0,              # [C] L2
            reg_alpha=1.0,               # [C] L1
            min_child_samples=25,
            metric="rmse",
            device=_lgbm_device(),
            random_state=42,
            verbose=-1,
        )
    else:  # HistGradientBoosting (default / CPU)
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=4,
            min_samples_leaf=25,
            learning_rate=0.05,
            l2_regularization=1.0,      # [C] aumentado de 0.3 → 1.0
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 8.  [A]  PIPELINE TimeSeriesSplit K=5
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    """Extrae historial de error/score por iteración según el algoritmo."""
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist  = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name


def _entrenar_kfold(
    df: pd.DataFrame,
    algoritmo: str,
    n_splits: int = 10,
    log_fn=None,
) -> tuple[pd.DataFrame, dict]:
    """
    [A] Validación cruzada cronológica con TimeSeriesSplit.

    TimeSeriesSplit garantiza que cada fold de validación sea siempre
    POSTERIOR al bloque de entrenamiento → cero data leakage temporal.

    Estructura de ventanas (ejemplo n_splits=10, N muestras):
        Fold 1 → Train: [0 … k]        Val: [k+1 … 2k]
        Fold 2 → Train: [0 … 2k]       Val: [2k+1 … 3k]
        …
        Fold 5 → Train: [0 … 4k]       Val: [4k+1 … N]

    El StandardScaler se ajusta ÚNICAMENTE con X_tr de cada ventana
    para evitar cualquier fuga de estadísticos futuros.
    """
    if log_fn is None:
        log_fn = log.info

    # [A] TimeSeriesSplit reemplaza KFold — el orden temporal siempre se respeta
    tscv      = TimeSeriesSplit(n_splits=n_splits)
    filas     = []
    cols_df   = set(df.columns)
    all_curves: dict = {}

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible en el dataset — omitido.")
            continue

        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]
        mask_ok   = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]

        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue

        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics:   list[dict] = []
        fold_curves_tr: list[list] = []
        fold_curves_va: list[list] = []
        metric_name_cv: str        = "Score"

        # [A] tscv.split() garantiza que val siempre sea posterior a train
        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            # [A] Scaler ajustado SOLO con train de esta ventana → sin leakage
            scaler   = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X_tr)
            X_va_sc  = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo)

            if "XGBoost" in algoritmo:
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                               lgb.log_evaluation(-1)],
                )
            else:
                modelo.fit(X_tr_sc, y_tr)

            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h:
                fold_curves_tr.append(tr_h)
            if va_h:
                fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(
                f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                f"MAE={fold_metrics[-1]['MAE']:.3f}  "
                f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
                f"R²={fold_metrics[-1]['R2']:.3f}"
            )

        all_curves[contaminante] = {
            "train":  fold_curves_tr,
            "val":    fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":   len(feat_cols),
            "MAE_mean":     mf["MAE"].mean(),
            "MAE_std":      mf["MAE"].std(),
            "RMSE_mean":    mf["RMSE"].mean(),
            "RMSE_std":     mf["RMSE"].std(),
            "R2_mean":      mf["R2"].mean(),
            "R2_std":       mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]

    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje(curvas, target_col, algoritmo, parroquia=""):
    COLOR_TR = COLOR_REAL
    COLOR_VA = COLOR_PRED

    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5,
                "No hay datos de curva de aprendizaje para este contaminante.",
                ha="center", va="center", color=TEXT_PLOT, fontsize=11,
                transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = (metric_name == "R²")

    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto para graficar.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len = min(lengths)
    x       = np.arange(min_len)
    tr_arr  = np.array([c[:min_len] for c in train_curves])
    tr_mean = tr_arr.mean(axis=0)
    tr_std  = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_TR, alpha=0.10, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_TR, lw=2.2,
            label=f"Train — {metric_name}  (μ TSS-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_TR, zorder=3)

    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)
        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_VA, alpha=0.10, lw=0.75, zorder=2)
        va_label = (f"Validación interna HGB — {metric_name}  (μ TSS-Fold)"
                    if is_r2 else f"Validación TSS-Fold — {metric_name}  (μ)")
        ax.plot(x, va_mean, color=COLOR_VA, lw=2.2, linestyle="--",
                label=va_label, zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_VA, zorder=3)

        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":", alpha=0.88,
                   label=f"Mejor iteración: {best_iter} ({metric_name}={best_val:.4f})",
                   zorder=6)
        offset = max(1, int(min_len * 0.02))
        ax.annotate(f" iter {best_iter}", xy=(best_iter, best_val),
                    xytext=(best_iter + offset, best_val),
                    color="#F59E0B", fontsize=8.5, va="center", zorder=7)

    ax.axhline(tr_mean[-1], color=COLOR_TR, lw=0.8, linestyle=":", alpha=0.40, zorder=1)

    algo_label = algoritmo.split("(")[0].strip()
    titulo = f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  TimeSeriesSplit (K=5)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    nota   = ("Nota HGB: Validación = fracción interna 10% de early stopping."
              if is_r2 else
              "Nota: banda sombreada = ±1σ entre pliegues temporales (TimeSeriesSplit, shuffle=False implícito).")

    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9, loc="best")
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL  (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    target_col:       str,
    nombre_modelo:    str,
    algoritmo:        str,
    plot_days:        int,
    train_ratio:      float,
    excluir_pandemia: bool,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)
    VACIO = (None, None, None, None)   # pred, fi, hm, lc

    try:
        # ── 10.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2  Cargar y limpiar ────────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — manejados por scaler/HGB.")

        # ── 10.3  TimeSeriesSplit K=5 sobre los 6 contaminantes ───────────────
        info(f"Iniciando TimeSeriesSplit (K=5) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} → {GPU_MSG}")

        df_clean = pd.concat([X, y], axis=1)
        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=10, log_fn=kf_log
        )
        for l in kfold_logs:
            logs.append(l)

        if kf_resumen.empty:
            warn("TSS K-Fold no produjo resultados — verifica contaminantes en el dataset.")

        # ── 10.4  Curva de aprendizaje para el target seleccionado ───────────
        info(f"Generando curva de aprendizaje para '{target_col}'…")
        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia
        )
        info("Curva de aprendizaje generada.")

        # ── 10.5  [B]  Modelo final con sub-split interno (test 100% ciego) ──
        info(f"Entrenando modelo final sobre '{target_col}' (test blindado)…")

        # División cronológica principal: 80% train | 20% test (CIEGO)
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)

        # [B] Sub-split cronológico interno: 85% del train → sub-train | 15% → val interno
        # El test final (X_te / y_te) NO se toca en ningún eval_set.
        sub_corte   = int(len(X_tr) * 0.85)
        X_sub_tr    = X_tr.iloc[:sub_corte]
        X_val_sub   = X_tr.iloc[sub_corte:]
        y_sub_tr    = y_tr.iloc[:sub_corte]
        y_val_sub   = y_tr.iloc[sub_corte:]

        info(
            f"  Sub-split interno — sub-train: {len(X_sub_tr):,} | "
            f"val interno: {len(X_val_sub):,} | test ciego: {len(X_te):,}"
        )

        # Scaler ajustado SOLO con sub-train → luego se transforma todo
        scaler_final  = StandardScaler()
        X_sub_tr_sc   = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc  = scaler_final.transform(X_val_sub)
        X_te_sc       = scaler_final.transform(X_te)

        # Reconstruir X_tr_sc completo (sub-train + val-interno) para métricas de train
        X_tr_sc = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np = y_tr.values   # array alineado con X_tr_sc

        modelo_final = _construir_modelo(algoritmo)

        if "XGBoost" in algoritmo:
            # [B] eval_set usa SOLO el val interno — nunca el test ciego
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                verbose=False,
            )
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                callbacks=[lgb.early_stopping(30, verbose=False),
                           lgb.log_evaluation(-1)],
            )
        else:  # HistGB usa su fracción interna de validation_fraction=0.1
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        # Predicciones (train completo y test ciego)
        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)
        info("Modelo final entrenado — test set nunca fue expuesto al modelo.")

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )
        m_tr, m_te = _m(y_tr_np, y_pred_tr), _m(y_te.values, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting en modelo final: ΔR² = {gap:.3f}")

        # ── 10.6  [B2]  Markdown de métricas con fila Precisión (R² %) ───────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        # Tabla K-Fold
        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                r2_color = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** "
                    f"| `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {r2_color} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        # [B2] R² en % para la tabla del modelo final
        r2_pct_tr = m_tr["R2"] * 100
        r2_pct_te = m_te["R2"] * 100

        # [B2] Una única definición del bloque metricas_md (sin duplicación)
        metricas_md = f"""
## 📊 Surrogate Model — *{parroquia}*

### Validación Cronológica TimeSeriesSplit (K=5) — 6 Contaminantes REMMAQ

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> **Nota:** `TimeSeriesSplit` garantiza que cada fold de validación sea
> siempre posterior al bloque de entrenamiento → **cero data leakage temporal**.
> `StandardScaler` ajustado únicamente con el train de cada ventana.
> Ver pestaña **📉 Curva de Aprendizaje** para el progreso por época/iteración.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

> **Test set 100% ciego**: el 20% final nunca fue expuesto al modelo.
> Early stopping controlado por sub-validación interna cronológica (15% de train).

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión del Modelo (R² %)** | `{r2_pct_tr:.2f}%` | `{r2_pct_te:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features base | `{len(feat_cols)}` columnas |
| Sub-train / Val interno / Test | `{len(X_sub_tr):,}` / `{len(X_val_sub):,}` / `{len(X_te):,}` |
| Filtro pandemia | {pandemia_label} |
| Regularización | L2=1.0 (HGB) · λ=5.0 α=1.0 (XGB/LGBM) |
"""

        # ── 10.7  Feature Importance ──────────────────────────────────────────
        info("Calculando Permutation Importance…")
        perm  = permutation_importance(modelo_final, X_te_sc, y_te.values,
                                       n_repeats=8, random_state=42, scoring="r2")
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 10.8  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia,
                "kfold_resumen": kf_resumen, "kf_curvas": kf_curvas,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 10.9  Guardar en sesión ───────────────────────────────────────────
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": {
                col: {"min": float(X[col].min()), "max": float(X[col].max()),
                      "mean": float(X[col].mean())}
                for col in feat_cols
            },
        })
        info("Modelo en sesión → pestaña Predicción lista.")

        # ── 10.10  Figuras ────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm      = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Heatmap generado.")

        # 6 outputs: metricas, fig_pred, fig_fi, fig_hm, fig_lc, estado
        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n"
            f"Instala con: `pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado()


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 12.  UI
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return ("### ⚠️ No hay modelo entrenado en la sesión.\n"
                "Primero entrena un modelo.")
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]
        extra     = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    calidad = "🟢 **Buena**"
            elif pred <= 35:  calidad = "🟡 **Moderada**"
            elif pred <= 55:  calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else:             calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice de calidad del aire:** {calidad}"
        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v9",
    ) as app:

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>TimeSeriesSplit · Blind Test · Regularización Penalizada · v9.0</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════
            # PANEL IZQUIERDO
            # ══════════════════════════════════
            with gr.Column(scale=1, min_width=350):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar columnas del CSV", variant="secondary", size="sm"
                    )
                    info_cols = gr.Markdown(
                        "_Pulsa 'Detectar columnas' para ver las variables disponibles._"
                    )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25", allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )
                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina registros de 2020 y 2021 para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Proporción de entrenamiento",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar (Test)",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════
            # PANEL DERECHO — 6 TABS
            # ══════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test"
                        )

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso por época / boosting round** con TimeSeriesSplit (K=5). "
                            "Línea **continua** = train · línea **discontinua** = validación. "
                            "Banda = ±1σ entre los 5 pliegues temporales.\n\n"
                            "- **XGBoost / LightGBM** → RMSE por iteración\n"
                            "- **HistGradientBoosting** → R² por árbol (`train_score_` / `validation_score_`)"
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Error vs Iteración (TSS K=5)"
                        )

                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(
                            label="Importancia de variables (Permutation Δ R²)"
                        )

                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La fila/columna del target aparece resaltada en naranja."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "usando el **modelo en memoria** (entrenado en esta sesión)."
                        )
                        sesion_info   = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v9.0 · TimeSeriesSplit · Blind Test · Reg. Penalizada · Auto-GPU
        </div>
        """)

        # ── Funciones auxiliares de la UI ─────────────────────────────────────

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")
            sesion_msg = (
                f"✅ Modelo en sesión: **{target}** · *{parroquia}* · "
                f"{len(feat_cols)} features."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col, value=round(st["mean"], 3),
                        visible=True, interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            import json
            return json.dumps(d)

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta:
                return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices   = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])
        btn_detectar.click(
            fn=_detectar_wrapper,
            inputs=[csv_dropdown, csv_upload],
            outputs=[target_input, info_cols],
        )
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider, excluir_pandemia_chk,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_fi_output,
                fig_hm_output, fig_lc_output, estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )
        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 64)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v9.0")
    print("  🔒  TimeSeriesSplit · Blind Test · Regularización Penalizada")
    print("═" * 64)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 64)
    print("\n⏳ Generando link de acceso externo... por favor espera.\n")

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

19:37:18 [INFO] GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB



════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v9.0
  🔒  TimeSeriesSplit · Blind Test · Regularización Penalizada
════════════════════════════════════════════════════════════════
  Hardware  : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  GPU activa: True
  Puerto    : automático (server_port=None)
════════════════════════════════════════════════════════════════

⏳ Generando link de acceso externo... por favor espera.



19:37:47 [INFO] Archivo subido: dataset_ml_carapungo.csv
19:37:47 [INFO] CSV cargado: 160,262 filas × 19 columnas
19:37:47 [INFO] Columna de tiempo: 'Timestamp'
19:37:47 [INFO] [FIX-3] 23 registros de [2020, 2021] eliminados.
19:37:47 [INFO] Pandemia excluida: años [2020, 2021] eliminados.
19:37:47 [WARNING] 217,722 NaN en features — manejados por scaler/HGB.
19:37:47 [INFO] Iniciando TimeSeriesSplit (K=5) · Algoritmo: HistGradientBoosting (CPU — sin GPU requerida)
19:37:47 [INFO] GPU disponible: True → GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
19:37:48 [INFO]   PM25 | Fold 1/10 → MAE=5.952  RMSE=8.043  R²=0.577
19:37:49 [INFO]   PM25 | Fold 2/10 → MAE=5.642  RMSE=7.985  R²=0.569
19:37:51 [INFO]   PM25 | Fold 3/10 → MAE=5.997  RMSE=9.403  R²=0.521
19:37:51 [INFO]   PM25 | Fold 4/10 → MAE=6.581  RMSE=10.400  R²=0.393
19:37:53 [INFO]   PM25 | Fold 5/10 → MAE=4.583  RMSE=6.184  R²=0.470
19:37:54 [INFO]   PM25 | Fold 6/10 → MAE=4.944  RMSE=7.238  R²=0.544
19:37:55 [INFO] 

version2dekfold

In [6]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 9.1  (Strong L2 Regularization + Dynamic K Control)
  Correcciones heredadas de v5.0:
      [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
      [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
      [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
      [FIX-4]  Auto-detección GPU             → torch / nvidia-smi / subprocess
      [FIX-5]  Flexibilidad total de dataset  → detección dinámica de columnas
  Refactorización v9.1:
      [A]  TimeSeriesSplit(n_splits=K) dinámico
      [B]  Sub-split cronológico interno (80/20) → eval_set más representativo
      [B2] Fila "Precisión del Modelo (R² %)" en tabla de métricas
      [C]  Regularización MUY penalizada:
            HGB: l2=3.0, max_depth=3, min_samples_leaf=50, lr=0.03
            XGB: λ=10, α=2, max_depth=3, subsample=0.7, lr=0.03
            LGB: λ=10, α=2, max_depth=3, subsample=0.7, lr=0.03
      [D]  Slider para elegir K (2-15) en la interfaz
================================================================================
  Dependencias mínimas:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
  Dependencias GPU (opcionales):
      pip install xgboost lightgbm
      pip install torch  # solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE \
           else {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS ( líneas con # )
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols  = df_head.columns.tolist()
        ts    = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        return cols_sin_ts, f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(
    df: pd.DataFrame,
    target_col: str,
    timestamp_col: str,
    excluir_pandemia: bool = True,
) -> tuple:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_exc = n_antes - len(df)
        if n_exc:
            log.info(f"[FIX-3] {n_exc:,} registros de {ANOS_PANDEMIA} eliminados.")

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    df = df.dropna(subset=[target_col])
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]

    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")

    return df[feat_cols], df[target_col], feat_cols, df.index


def _dividir_cronologico(X, y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols_disp = set(df.select_dtypes(include=[np.number]).columns)
    x_base    = [c for c in FEATURES_X_BASE if c in cols_disp]
    if x_base:
        return [c for c in x_base if c != target_col]
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disp if c not in excluir or f"{c}_lag" in c]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  [C]  CONSTRUCCIÓN DEL MODELO CON REGULARIZACIÓN FUERTE
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str):
    """
    [C] Regularización estructural muy penalizada para combatir el sobreajuste
    causado por los lags de corto plazo.
      · XGBoost  → reg_lambda=10.0, reg_alpha=2.0, max_depth=3, lr=0.03
      · LightGBM → reg_lambda=10.0, reg_alpha=2.0, max_depth=3, lr=0.03
      · HistGB   → l2_regularization=3.0, max_depth=3, min_samples_leaf=50, lr=0.03
    """
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=1000,
            max_depth=3,                # más restrictivo
            learning_rate=0.03,
            subsample=0.7,              # menos datos por árbol
            colsample_bytree=0.7,
            reg_lambda=10.0,            # penalización fuerte
            reg_alpha=2.0,
            early_stopping_rounds=50,   # más paciencia antes de parar
            eval_metric="rmse",
            random_state=42,
            verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=1000,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.7,
            colsample_bytree=0.7,
            reg_lambda=10.0,
            reg_alpha=2.0,
            min_child_samples=50,
            metric="rmse",
            device=_lgbm_device(),
            random_state=42,
            verbose=-1,
        )
    else:  # HistGradientBoosting
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=3,
            min_samples_leaf=50,        # hojas mucho más pobladas
            learning_rate=0.03,
            l2_regularization=3.0,      # muy penalizado
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 8.  [A]  PIPELINE TimeSeriesSplit DINÁMICO
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist  = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name


def _entrenar_kfold(
    df: pd.DataFrame,
    algoritmo: str,
    n_splits: int = 5,
    log_fn=None,
) -> tuple[pd.DataFrame, dict]:
    if log_fn is None:
        log_fn = log.info

    tscv      = TimeSeriesSplit(n_splits=n_splits)
    filas     = []
    cols_df   = set(df.columns)
    all_curves: dict = {}

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible en el dataset — omitido.")
            continue

        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]
        mask_ok   = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]

        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue

        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics:   list[dict] = []
        fold_curves_tr: list[list] = []
        fold_curves_va: list[list] = []
        metric_name_cv: str        = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            scaler   = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X_tr)
            X_va_sc  = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo)

            if "XGBoost" in algoritmo:
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                               lgb.log_evaluation(-1)],
                )
            else:
                modelo.fit(X_tr_sc, y_tr)

            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h:
                fold_curves_tr.append(tr_h)
            if va_h:
                fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(
                f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                f"MAE={fold_metrics[-1]['MAE']:.3f}  "
                f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
                f"R²={fold_metrics[-1]['R2']:.3f}"
            )

        all_curves[contaminante] = {
            "train":  fold_curves_tr,
            "val":    fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":   len(feat_cols),
            "MAE_mean":     mf["MAE"].mean(),
            "MAE_std":      mf["MAE"].std(),
            "RMSE_mean":    mf["RMSE"].mean(),
            "RMSE_std":     mf["RMSE"].std(),
            "R2_mean":      mf["R2"].mean(),
            "R2_std":       mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS (sin cambios relevantes)
# ──────────────────────────────────────────────────────────────────────────────

def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]
    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje(curvas, target_col, algoritmo, parroquia="", k_splits=5):
    COLOR_TR = COLOR_REAL
    COLOR_VA = COLOR_PRED

    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5,
                "No hay datos de curva de aprendizaje para este contaminante.",
                ha="center", va="center", color=TEXT_PLOT, fontsize=11,
                transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = (metric_name == "R²")

    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto para graficar.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len = min(lengths)
    x       = np.arange(min_len)
    tr_arr  = np.array([c[:min_len] for c in train_curves])
    tr_mean = tr_arr.mean(axis=0)
    tr_std  = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_TR, alpha=0.10, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_TR, lw=2.2,
            label=f"Train — {metric_name}  (μ TSS-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_TR, zorder=3)

    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)
        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_VA, alpha=0.10, lw=0.75, zorder=2)
        va_label = (f"Validación interna HGB — {metric_name}  (μ TSS-Fold)"
                    if is_r2 else f"Validación TSS-Fold — {metric_name}  (μ)")
        ax.plot(x, va_mean, color=COLOR_VA, lw=2.2, linestyle="--",
                label=va_label, zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_VA, zorder=3)

        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":", alpha=0.88,
                   label=f"Mejor iteración: {best_iter} ({metric_name}={best_val:.4f})",
                   zorder=6)
        offset = max(1, int(min_len * 0.02))
        ax.annotate(f" iter {best_iter}", xy=(best_iter, best_val),
                    xytext=(best_iter + offset, best_val),
                    color="#F59E0B", fontsize=8.5, va="center", zorder=7)

    ax.axhline(tr_mean[-1], color=COLOR_TR, lw=0.8, linestyle=":", alpha=0.40, zorder=1)

    algo_label = algoritmo.split("(")[0].strip()
    titulo = f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  TimeSeriesSplit (K={k_splits})"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    nota   = ("Nota HGB: Validación = fracción interna 10% de early stopping."
              if is_r2 else
              "Nota: banda sombreada = ±1σ entre pliegues temporales (TimeSeriesSplit, shuffle=False implícito).")

    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9, loc="best")
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    target_col:       str,
    nombre_modelo:    str,
    algoritmo:        str,
    plot_days:        int,
    train_ratio:      float,
    excluir_pandemia: bool,
    k_splits:         int = 5,               # [D] nuevo control
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)
    VACIO = (None, None, None, None)   # pred, fi, hm, lc

    try:
        # ── 10.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2  Cargar y limpiar ────────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — manejados por scaler/HGB.")

        # ── 10.3  TimeSeriesSplit con K dinámico ──────────────────────────────
        info(f"Iniciando TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} → {GPU_MSG}")

        df_clean = pd.concat([X, y], axis=1)
        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits, log_fn=kf_log
        )
        for l in kfold_logs:
            logs.append(l)

        if kf_resumen.empty:
            warn("TSS K-Fold no produjo resultados — verifica contaminantes en el dataset.")

        # ── 10.4  Curva de aprendizaje para el target seleccionado ───────────
        info(f"Generando curva de aprendizaje para '{target_col}'…")
        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits
        )
        info("Curva de aprendizaje generada.")

        # ── 10.5  [B]  Modelo final con sub-split interno (test 100% ciego) ──
        info(f"Entrenando modelo final sobre '{target_col}' (test blindado)…")

        # División cronológica principal: 80% train | 20% test (CIEGO)
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)

        # [B] Sub-split cronológico interno: 80% del train → sub-train | 20% → val interno
        # (subimos al 20% para tener una validación más representativa)
        sub_corte   = int(len(X_tr) * 0.80)
        X_sub_tr    = X_tr.iloc[:sub_corte]
        X_val_sub   = X_tr.iloc[sub_corte:]
        y_sub_tr    = y_tr.iloc[:sub_corte]
        y_val_sub   = y_tr.iloc[sub_corte:]

        info(
            f"  Sub-split interno — sub-train: {len(X_sub_tr):,} | "
            f"val interno: {len(X_val_sub):,} | test ciego: {len(X_te):,}"
        )

        # Scaler ajustado SOLO con sub-train
        scaler_final  = StandardScaler()
        X_sub_tr_sc   = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc  = scaler_final.transform(X_val_sub)
        X_te_sc       = scaler_final.transform(X_te)

        X_tr_sc = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np = y_tr.values

        modelo_final = _construir_modelo(algoritmo)

        if "XGBoost" in algoritmo:
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                verbose=False,
            )
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                callbacks=[lgb.early_stopping(30, verbose=False),
                           lgb.log_evaluation(-1)],
            )
        else:  # HistGB usa su fracción interna de validation_fraction=0.1
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)
        info("Modelo final entrenado — test set nunca fue expuesto al modelo.")

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )
        m_tr, m_te = _m(y_tr_np, y_pred_tr), _m(y_te.values, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting en modelo final: ΔR² = {gap:.3f}")

        # ── 10.6  Markdown de métricas dinámico ───────────────────────────────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                r2_color = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** "
                    f"| `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {r2_color} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        r2_pct_tr = m_tr["R2"] * 100
        r2_pct_te = m_te["R2"] * 100

        metricas_md = f"""
## 📊 Surrogate Model — *{parroquia}*

### Validación Cronológica TimeSeriesSplit (K={k_splits}) — 6 Contaminantes REMMAQ

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> **Nota:** `TimeSeriesSplit` garantiza que cada fold de validación sea
> siempre posterior al bloque de entrenamiento → **cero data leakage temporal**.
> `StandardScaler` ajustado únicamente con el train de cada ventana.
> Ver pestaña **📉 Curva de Aprendizaje** para el progreso por época/iteración.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

> **Test set 100% ciego**: el 20% final nunca fue expuesto al modelo.
> Early stopping controlado por sub-validación interna cronológica (20% de train).

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión del Modelo (R² %)** | `{r2_pct_tr:.2f}%` | `{r2_pct_te:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features base | `{len(feat_cols)}` columnas |
| Sub-train / Val interno / Test | `{len(X_sub_tr):,}` / `{len(X_val_sub):,}` / `{len(X_te):,}` |
| Filtro pandemia | {pandemia_label} |
| Regularización | L2=3.0 (HGB) · λ=10.0 α=2.0 (XGB/LGBM) |
"""

        # ── 10.7  Feature Importance ──────────────────────────────────────────
        info("Calculando Permutation Importance…")
        perm  = permutation_importance(modelo_final, X_te_sc, y_te.values,
                                       n_repeats=8, random_state=42, scoring="r2")
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 10.8  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia,
                "kfold_resumen": kf_resumen, "kf_curvas": kf_curvas,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 10.9  Guardar en sesión ───────────────────────────────────────────
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": {
                col: {"min": float(X[col].min()), "max": float(X[col].max()),
                      "mean": float(X[col].mean())}
                for col in feat_cols
            },
        })
        info("Modelo en sesión → pestaña Predicción lista.")

        # ── 10.10  Figuras ────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm      = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Heatmap generado.")

        # 6 outputs: metricas, fig_pred, fig_fi, fig_hm, fig_lc, estado
        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n"
            f"Instala con: `pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado()


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS (sin cambios)
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 12.  UI con control de K
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return ("### ⚠️ No hay modelo entrenado en la sesión.\n"
                "Primero entrena un modelo.")
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]
        extra     = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    calidad = "🟢 **Buena**"
            elif pred <= 35:  calidad = "🟡 **Moderada**"
            elif pred <= 55:  calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else:             calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice de calidad del aire:** {calidad}"
        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v9.1",
    ) as app:

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>TimeSeriesSplit Dinámico · Regularización Fuerte · Blind Test · v9.1</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════
            # PANEL IZQUIERDO
            # ══════════════════════════════════
            with gr.Column(scale=1, min_width=350):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar columnas del CSV", variant="secondary", size="sm"
                    )
                    info_cols = gr.Markdown(
                        "_Pulsa 'Detectar columnas' para ver las variables disponibles._"
                    )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25", allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )
                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina registros de 2020 y 2021 para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Proporción de entrenamiento",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        k_splits_slider = gr.Slider(
                            label="Número de pliegues (K) TSCV",
                            minimum=2, maximum=15, step=1, value=5, scale=2,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar (Test)",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════
            # PANEL DERECHO — 6 TABS
            # ══════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test"
                        )

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso por época / boosting round** con TimeSeriesSplit. "
                            "Línea **continua** = train · línea **discontinua** = validación. "
                            "Banda = ±1σ entre los pliegues temporales.\n\n"
                            "- **XGBoost / LightGBM** → RMSE por iteración\n"
                            "- **HistGradientBoosting** → R² por árbol"
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Error vs Iteración (TSS)"
                        )

                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(
                            label="Importancia de variables (Permutation Δ R²)"
                        )

                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La fila/columna del target aparece resaltada en naranja."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "usando el **modelo en memoria** (entrenado en esta sesión)."
                        )
                        sesion_info   = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v9.1 · TimeSeriesSplit Dinámico · Regularización Fuerte · Auto-GPU
        </div>
        """)

        # ── Funciones auxiliares de la UI ─────────────────────────────────────
        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")
            sesion_msg = (
                f"✅ Modelo en sesión: **{target}** · *{parroquia}* · "
                f"{len(feat_cols)} features."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col, value=round(st["mean"], 3),
                        visible=True, interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            import json
            return json.dumps(d)

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta:
                return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices   = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])
        btn_detectar.click(
            fn=_detectar_wrapper,
            inputs=[csv_dropdown, csv_upload],
            outputs=[target_input, info_cols],
        )
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, k_splits_slider,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_fi_output,
                fig_hm_output, fig_lc_output, estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )
        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 64)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v9.1")
    print("  🔒  TimeSeriesSplit Dinámico · Regularización Fuerte")
    print("═" * 64)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 64)
    print("\n⏳ Generando link de acceso externo... por favor espera.\n")

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )


════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v9.1
  🔒  TimeSeriesSplit Dinámico · Regularización Fuerte
════════════════════════════════════════════════════════════════
  Hardware  : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  GPU activa: True
  Puerto    : automático (server_port=None)
════════════════════════════════════════════════════════════════

⏳ Generando link de acceso externo... por favor espera.



19:55:35 [INFO] Archivo subido: dataset_ml_carapungo.csv
19:55:36 [INFO] CSV cargado: 160,262 filas × 19 columnas
19:55:36 [INFO] Columna de tiempo: 'Timestamp'
19:55:36 [INFO] [FIX-3] 23 registros de [2020, 2021] eliminados.
19:55:36 [INFO] Pandemia excluida: años [2020, 2021] eliminados.
19:55:36 [WARNING] 217,722 NaN en features — manejados por scaler/HGB.
19:55:36 [INFO] Iniciando TimeSeriesSplit (K=10) · Algoritmo: HistGradientBoosting (CPU — sin GPU requerida)
19:55:36 [INFO] GPU disponible: True → GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
19:55:36 [INFO]   PM25 | Fold 1/10 → MAE=5.980  RMSE=8.109  R²=0.570
19:55:38 [INFO]   PM25 | Fold 2/10 → MAE=5.617  RMSE=7.966  R²=0.571
19:55:40 [INFO]   PM25 | Fold 3/10 → MAE=5.966  RMSE=9.403  R²=0.521
19:55:42 [INFO]   PM25 | Fold 4/10 → MAE=6.597  RMSE=10.390  R²=0.394
19:55:44 [INFO]   PM25 | Fold 5/10 → MAE=4.579  RMSE=6.177  R²=0.471
19:55:45 [INFO]   PM25 | Fold 6/10 → MAE=4.909  RMSE=7.122  R²=0.558
19:55:46 [INFO]

Modelo  con Kfold   y   xgbost

In [2]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 9.1  (Strong L2 Regularization + Dynamic K Control)
  Correcciones heredadas de v5.0:
      [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
      [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
      [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
      [FIX-4]  Auto-detección GPU             → torch / nvidia-smi / subprocess
      [FIX-5]  Flexibilidad total de dataset  → detección dinámica de columnas
  Refactorización v9.1:
      [A]  TimeSeriesSplit(n_splits=K) dinámico
      [B]  Sub-split cronológico interno (80/20) → eval_set más representativo
      [B2] Fila "Precisión del Modelo (R² %)" en tabla de métricas
      [C]  Regularización MUY penalizada:
            HGB: l2=3.0, max_depth=3, min_samples_leaf=50, lr=0.03
            XGB: λ=10, α=2, max_depth=3, subsample=0.7, lr=0.03
            LGB: λ=10, α=2, max_depth=3, subsample=0.7, lr=0.03
      [D]  Slider para elegir K (2-15) en la interfaz
================================================================================
  Dependencias mínimas:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
  Dependencias GPU (opcionales):
      pip install xgboost lightgbm
      pip install torch  # solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE \
           else {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS ( líneas con # )
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols  = df_head.columns.tolist()
        ts    = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        return cols_sin_ts, f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(
    df: pd.DataFrame,
    target_col: str,
    timestamp_col: str,
    excluir_pandemia: bool = True,
) -> tuple:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_exc = n_antes - len(df)
        if n_exc:
            log.info(f"[FIX-3] {n_exc:,} registros de {ANOS_PANDEMIA} eliminados.")

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    df = df.dropna(subset=[target_col])
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]

    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")

    return df[feat_cols], df[target_col], feat_cols, df.index


def _dividir_cronologico(X, y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols_disp = set(df.select_dtypes(include=[np.number]).columns)
    x_base    = [c for c in FEATURES_X_BASE if c in cols_disp]
    if x_base:
        return [c for c in x_base if c != target_col]
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disp if c not in excluir or f"{c}_lag" in c]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  [C]  CONSTRUCCIÓN DEL MODELO CON REGULARIZACIÓN FUERTE
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str):
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            reg_alpha=0.5,
            early_stopping_rounds=50,
            eval_metric="rmse",
            random_state=42,
            verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            reg_alpha=0.5,
            min_child_samples=30,
            metric="rmse",
            device=_lgbm_device(),
            random_state=42,
            verbose=-1,
        )
    else:  # HistGradientBoosting (ya era demasiado restrictivo, relajamos)
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=5,
            min_samples_leaf=30,
            learning_rate=0.05,
            l2_regularization=1.0,
            random_state=42,
        )
# ──────────────────────────────────────────────────────────────────────────────
# 8.  [A]  PIPELINE TimeSeriesSplit DINÁMICO
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist  = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name


def _entrenar_kfold(
    df: pd.DataFrame,
    algoritmo: str,
    n_splits: int = 5,
    log_fn=None,
) -> tuple[pd.DataFrame, dict]:
    if log_fn is None:
        log_fn = log.info

    tscv      = TimeSeriesSplit(n_splits=n_splits)
    filas     = []
    cols_df   = set(df.columns)
    all_curves: dict = {}

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible en el dataset — omitido.")
            continue

        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]
        mask_ok   = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]

        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue

        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics:   list[dict] = []
        fold_curves_tr: list[list] = []
        fold_curves_va: list[list] = []
        metric_name_cv: str        = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            scaler   = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X_tr)
            X_va_sc  = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo)

            if "XGBoost" in algoritmo:
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                               lgb.log_evaluation(-1)],
                )
            else:
                modelo.fit(X_tr_sc, y_tr)

            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h:
                fold_curves_tr.append(tr_h)
            if va_h:
                fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(
                f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                f"MAE={fold_metrics[-1]['MAE']:.3f}  "
                f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
                f"R²={fold_metrics[-1]['R2']:.3f}"
            )

        all_curves[contaminante] = {
            "train":  fold_curves_tr,
            "val":    fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":   len(feat_cols),
            "MAE_mean":     mf["MAE"].mean(),
            "MAE_std":      mf["MAE"].std(),
            "RMSE_mean":    mf["RMSE"].mean(),
            "RMSE_std":     mf["RMSE"].std(),
            "R2_mean":      mf["R2"].mean(),
            "R2_std":       mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS (sin cambios relevantes)
# ──────────────────────────────────────────────────────────────────────────────

def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]
    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje(curvas, target_col, algoritmo, parroquia="", k_splits=5):
    COLOR_TR = COLOR_REAL
    COLOR_VA = COLOR_PRED

    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5,
                "No hay datos de curva de aprendizaje para este contaminante.",
                ha="center", va="center", color=TEXT_PLOT, fontsize=11,
                transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = (metric_name == "R²")

    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto para graficar.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len = min(lengths)
    x       = np.arange(min_len)
    tr_arr  = np.array([c[:min_len] for c in train_curves])
    tr_mean = tr_arr.mean(axis=0)
    tr_std  = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_TR, alpha=0.10, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_TR, lw=2.2,
            label=f"Train — {metric_name}  (μ TSS-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_TR, zorder=3)

    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)
        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_VA, alpha=0.10, lw=0.75, zorder=2)
        va_label = (f"Validación interna HGB — {metric_name}  (μ TSS-Fold)"
                    if is_r2 else f"Validación TSS-Fold — {metric_name}  (μ)")
        ax.plot(x, va_mean, color=COLOR_VA, lw=2.2, linestyle="--",
                label=va_label, zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_VA, zorder=3)

        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":", alpha=0.88,
                   label=f"Mejor iteración: {best_iter} ({metric_name}={best_val:.4f})",
                   zorder=6)
        offset = max(1, int(min_len * 0.02))
        ax.annotate(f" iter {best_iter}", xy=(best_iter, best_val),
                    xytext=(best_iter + offset, best_val),
                    color="#F59E0B", fontsize=8.5, va="center", zorder=7)

    ax.axhline(tr_mean[-1], color=COLOR_TR, lw=0.8, linestyle=":", alpha=0.40, zorder=1)

    algo_label = algoritmo.split("(")[0].strip()
    titulo = f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  TimeSeriesSplit (K={k_splits})"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    nota   = ("Nota HGB: Validación = fracción interna 10% de early stopping."
              if is_r2 else
              "Nota: banda sombreada = ±1σ entre pliegues temporales (TimeSeriesSplit, shuffle=False implícito).")

    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9, loc="best")
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    target_col:       str,
    nombre_modelo:    str,
    algoritmo:        str,
    plot_days:        int,
    train_ratio:      float,
    excluir_pandemia: bool,
    k_splits:         int = 5,               # [D] nuevo control
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)
    VACIO = (None, None, None, None)   # pred, fi, hm, lc

    try:
        # ── 10.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2  Cargar y limpiar ────────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — manejados por scaler/HGB.")

        # ── 10.3  TimeSeriesSplit con K dinámico ──────────────────────────────
        info(f"Iniciando TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} → {GPU_MSG}")

        df_clean = pd.concat([X, y], axis=1)
        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits, log_fn=kf_log
        )
        for l in kfold_logs:
            logs.append(l)

        if kf_resumen.empty:
            warn("TSS K-Fold no produjo resultados — verifica contaminantes en el dataset.")

        # ── 10.4  Curva de aprendizaje para el target seleccionado ───────────
        info(f"Generando curva de aprendizaje para '{target_col}'…")
        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits
        )
        info("Curva de aprendizaje generada.")

        # ── 10.5  [B]  Modelo final con sub-split interno (test 100% ciego) ──
        info(f"Entrenando modelo final sobre '{target_col}' (test blindado)…")

        # División cronológica principal: 80% train | 20% test (CIEGO)
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)

        # [B] Sub-split cronológico interno: 80% del train → sub-train | 20% → val interno
        # (subimos al 20% para tener una validación más representativa)
        sub_corte   = int(len(X_tr) * 0.80)
        X_sub_tr    = X_tr.iloc[:sub_corte]
        X_val_sub   = X_tr.iloc[sub_corte:]
        y_sub_tr    = y_tr.iloc[:sub_corte]
        y_val_sub   = y_tr.iloc[sub_corte:]

        info(
            f"  Sub-split interno — sub-train: {len(X_sub_tr):,} | "
            f"val interno: {len(X_val_sub):,} | test ciego: {len(X_te):,}"
        )

        # Scaler ajustado SOLO con sub-train
        scaler_final  = StandardScaler()
        X_sub_tr_sc   = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc  = scaler_final.transform(X_val_sub)
        X_te_sc       = scaler_final.transform(X_te)

        X_tr_sc = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np = y_tr.values

        modelo_final = _construir_modelo(algoritmo)

        if "XGBoost" in algoritmo:
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                verbose=False,
            )
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                callbacks=[lgb.early_stopping(30, verbose=False),
                           lgb.log_evaluation(-1)],
            )
        else:  # HistGB usa su fracción interna de validation_fraction=0.1
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)
        info("Modelo final entrenado — test set nunca fue expuesto al modelo.")

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )
        m_tr, m_te = _m(y_tr_np, y_pred_tr), _m(y_te.values, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting en modelo final: ΔR² = {gap:.3f}")

        # ── 10.6  Markdown de métricas dinámico ───────────────────────────────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                r2_color = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** "
                    f"| `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {r2_color} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        r2_pct_tr = m_tr["R2"] * 100
        r2_pct_te = m_te["R2"] * 100

        metricas_md = f"""
## 📊 Surrogate Model — *{parroquia}*

### Validación Cronológica TimeSeriesSplit (K={k_splits}) — 6 Contaminantes REMMAQ

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> **Nota:** `TimeSeriesSplit` garantiza que cada fold de validación sea
> siempre posterior al bloque de entrenamiento → **cero data leakage temporal**.
> `StandardScaler` ajustado únicamente con el train de cada ventana.
> Ver pestaña **📉 Curva de Aprendizaje** para el progreso por época/iteración.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

> **Test set 100% ciego**: el 20% final nunca fue expuesto al modelo.
> Early stopping controlado por sub-validación interna cronológica (20% de train).

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión del Modelo (R² %)** | `{r2_pct_tr:.2f}%` | `{r2_pct_te:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features base | `{len(feat_cols)}` columnas |
| Sub-train / Val interno / Test | `{len(X_sub_tr):,}` / `{len(X_val_sub):,}` / `{len(X_te):,}` |
| Filtro pandemia | {pandemia_label} |
| Regularización | L2=3.0 (HGB) · λ=10.0 α=2.0 (XGB/LGBM) |
"""

        # ── 10.7  Feature Importance ──────────────────────────────────────────
        info("Calculando Permutation Importance…")
        perm  = permutation_importance(modelo_final, X_te_sc, y_te.values,
                                       n_repeats=8, random_state=42, scoring="r2")
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 10.8  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia,
                "kfold_resumen": kf_resumen, "kf_curvas": kf_curvas,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 10.9  Guardar en sesión ───────────────────────────────────────────
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": {
                col: {"min": float(X[col].min()), "max": float(X[col].max()),
                      "mean": float(X[col].mean())}
                for col in feat_cols
            },
        })
        info("Modelo en sesión → pestaña Predicción lista.")

        # ── 10.10  Figuras ────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm      = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Heatmap generado.")

        # 6 outputs: metricas, fig_pred, fig_fi, fig_hm, fig_lc, estado
        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\n"
            f"Instala con: `pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado()


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS (sin cambios)
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 12.  UI con control de K
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return ("### ⚠️ No hay modelo entrenado en la sesión.\n"
                "Primero entrena un modelo.")
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]
        extra     = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    calidad = "🟢 **Buena**"
            elif pred <= 35:  calidad = "🟡 **Moderada**"
            elif pred <= 55:  calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else:             calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice de calidad del aire:** {calidad}"
        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v9.1",
    ) as app:

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>TimeSeriesSplit Dinámico · Regularización Fuerte · Blind Test · v9.1</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════
            # PANEL IZQUIERDO
            # ══════════════════════════════════
            with gr.Column(scale=1, min_width=350):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar columnas del CSV", variant="secondary", size="sm"
                    )
                    info_cols = gr.Markdown(
                        "_Pulsa 'Detectar columnas' para ver las variables disponibles._"
                    )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25", allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )
                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina registros de 2020 y 2021 para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Proporción de entrenamiento",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        k_splits_slider = gr.Slider(
                            label="Número de pliegues (K) TSCV",
                            minimum=2, maximum=15, step=1, value=5, scale=2,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar (Test)",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════
            # PANEL DERECHO — 6 TABS
            # ══════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test"
                        )

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso por época / boosting round** con TimeSeriesSplit. "
                            "Línea **continua** = train · línea **discontinua** = validación. "
                            "Banda = ±1σ entre los pliegues temporales.\n\n"
                            "- **XGBoost / LightGBM** → RMSE por iteración\n"
                            "- **HistGradientBoosting** → R² por árbol"
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Error vs Iteración (TSS)"
                        )

                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(
                            label="Importancia de variables (Permutation Δ R²)"
                        )

                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La fila/columna del target aparece resaltada en naranja."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "usando el **modelo en memoria** (entrenado en esta sesión)."
                        )
                        sesion_info   = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v9.1 · TimeSeriesSplit Dinámico · Regularización Fuerte · Auto-GPU
        </div>
        """)

        # ── Funciones auxiliares de la UI ─────────────────────────────────────
        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")
            sesion_msg = (
                f"✅ Modelo en sesión: **{target}** · *{parroquia}* · "
                f"{len(feat_cols)} features."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col, value=round(st["mean"], 3),
                        visible=True, interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            import json
            return json.dumps(d)

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta:
                return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices   = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])
        btn_detectar.click(
            fn=_detectar_wrapper,
            inputs=[csv_dropdown, csv_upload],
            outputs=[target_input, info_cols],
        )
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, k_splits_slider,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_fi_output,
                fig_hm_output, fig_lc_output, estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )
        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 64)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v9.1")
    print("  🔒  TimeSeriesSplit Dinámico · Regularización Fuerte")
    print("═" * 64)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 64)
    print("\n⏳ Generando link de acceso externo... por favor espera.\n")

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

00:54:40 [INFO] GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB



════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v9.1
  🔒  TimeSeriesSplit Dinámico · Regularización Fuerte
════════════════════════════════════════════════════════════════
  Hardware  : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  GPU activa: True
  Puerto    : automático (server_port=None)
════════════════════════════════════════════════════════════════

⏳ Generando link de acceso externo... por favor espera.



gbost de deepsek v2

In [3]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 9.3  (XGBoost Hiperparámetros Dinámicos desde UI)
  Correcciones heredadas de v9.2:
      [FIX-1/2/3/4/5] CSV, GPU, Pandemia, detección columnas
      [A] TimeSeriesSplit dinámico (K ajustable)
      [B] Sub‑split cronológico 80/20 → test ciego
      [B2] Precisión (R² %) en tabla final
      [C] Regularización balanceada (HGB, XGB, LGB)
  NUEVO en v9.3:
      [HP] Controles interactivos para hiperparámetros XGBoost:
            max_depth, learning_rate, subsample, colsample_bytree,
            reg_lambda, reg_alpha, gamma, min_child_weight
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os, sys, warnings, logging, traceback, subprocess, pickle, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"

# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────
def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"

GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)

def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE else {"tree_method": "hist", "device": "cpu"}

def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"

# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────
def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]

def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)

# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS
# ──────────────────────────────────────────────────────────────────────────────
def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e

# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────
def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None

def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols  = df_head.columns.tolist()
        ts    = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        return cols_sin_ts, f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"

# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────
def _preprocesar(df, target_col, timestamp_col, excluir_pandemia=True):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], infer_datetime_format=True, errors="coerce")
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=[target_col])
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]
    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")
    return df[feat_cols], df[target_col], feat_cols, df.index

def _dividir_cronologico(X, y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]

def _resolver_features_x(df, target_col):
    cols_disp = set(df.select_dtypes(include=[np.number]).columns)
    x_base    = [c for c in FEATURES_X_BASE if c in cols_disp]
    if x_base:
        return [c for c in x_base if c != target_col]
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disp if c not in excluir or f"{c}_lag" in c]

# ──────────────────────────────────────────────────────────────────────────────
# 7.  [C]  CONSTRUCCIÓN DEL MODELO (ahora con hiperparámetros dinámicos)
# ──────────────────────────────────────────────────────────────────────────────
def _construir_modelo(algoritmo: str, params: dict = None):
    """Crea el modelo según el algoritmo y los hiperparámetros opcionales proporcionados."""
    if params is None:
        params = {}
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000,
            max_depth=params.get("max_depth", 5),
            learning_rate=params.get("learning_rate", 0.05),
            subsample=params.get("subsample", 0.8),
            colsample_bytree=params.get("colsample_bytree", 0.8),
            reg_lambda=params.get("reg_lambda", 2.0),
            reg_alpha=params.get("reg_alpha", 0.5),
            gamma=params.get("gamma", 0.0),
            min_child_weight=params.get("min_child_weight", 1),
            early_stopping_rounds=50,
            eval_metric="rmse",
            random_state=42,
            verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000,
            max_depth=params.get("max_depth", 5),
            learning_rate=params.get("learning_rate", 0.05),
            subsample=params.get("subsample", 0.8),
            colsample_bytree=params.get("colsample_bytree", 0.8),
            reg_lambda=params.get("reg_lambda", 2.0),
            reg_alpha=params.get("reg_alpha", 0.5),
            min_child_samples=params.get("min_child_weight", 30),   # mapeo aproximado
            metric="rmse",
            device=_lgbm_device(),
            random_state=42,
            verbose=-1,
        )
    else:  # HistGradientBoosting
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=params.get("max_depth", 5),
            min_samples_leaf=params.get("min_child_weight", 30),
            learning_rate=params.get("learning_rate", 0.05),
            l2_regularization=params.get("reg_lambda", 1.0),
            random_state=42,
        )

# ──────────────────────────────────────────────────────────────────────────────
# 8.  PIPELINE TimeSeriesSplit (sin cambios, pero recibe params de XGB)
# ──────────────────────────────────────────────────────────────────────────────
def _extraer_curvas_fold(modelo, algoritmo):
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks    = list(evals.keys())
            met   = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist  = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name

def _entrenar_kfold(df, algoritmo, n_splits=5, log_fn=None, xgb_params=None):
    if log_fn is None:
        log_fn = log.info
    tscv      = TimeSeriesSplit(n_splits=n_splits)
    filas     = []
    cols_df   = set(df.columns)
    all_curves: dict = {}

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible en el dataset — omitido.")
            continue
        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]
        mask_ok   = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]
        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue
        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics:   list[dict] = []
        fold_curves_tr: list[list] = []
        fold_curves_va: list[list] = []
        metric_name_cv: str        = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            scaler   = StandardScaler()
            X_tr_sc  = scaler.fit_transform(X_tr)
            X_va_sc  = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo, xgb_params)   # pasa params

            if "XGBoost" in algoritmo:
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)],
                )
            else:
                modelo.fit(X_tr_sc, y_tr)

            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h: fold_curves_tr.append(tr_h)
            if va_h: fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(f"  {contaminante} | Fold {fold_idx}/{n_splits} → MAE={fold_metrics[-1]['MAE']:.3f} RMSE={fold_metrics[-1]['RMSE']:.3f} R²={fold_metrics[-1]['R2']:.3f}")

        all_curves[contaminante] = {
            "train":  fold_curves_tr,
            "val":    fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":   len(feat_cols),
            "MAE_mean":     mf["MAE"].mean(),
            "MAE_std":      mf["MAE"].std(),
            "RMSE_mean":    mf["RMSE"].mean(),
            "RMSE_std":     mf["RMSE"].std(),
            "R2_mean":      mf["R2"].mean(),
            "R2_std":       mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves

# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS (se mantienen igual)
# ──────────────────────────────────────────────────────────────────────────────
# (código de figuras igual al anterior, no incluido por brevedad, se copia exactamente)
# ...
# NOTA: Por razones de espacio omito las funciones de figuras, pero se deben incluir tal cual estaban.
#       Voy a asumir que están en el código final.

# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────
def entrenar(
    csv_local, csv_upload, target_col, nombre_modelo, algoritmo,
    plot_days, train_ratio, excluir_pandemia, k_splits,
    # nuevos hiperparámetros XGBoost
    max_depth, learning_rate, subsample, colsample_bytree,
    reg_lambda, reg_alpha, gamma, min_child_weight,
):
    logs: list[str] = []
    def info(m): log.info(m); logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️ {m}")
    def err(m): log.error(m); logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)
    VACIO = (None, None, None, None)

    try:
        # ── 10.1 Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()
        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2 Cargar y limpiar ──────────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")
        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (f"### ❌ Target **`{target_col}`** no encontrado.\n\n**Columnas disponibles:** `{cols_disp}`", *VACIO, _estado())

        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia: info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")
        n_nulos = int(X.isna().sum().sum())
        if n_nulos: warn(f"{n_nulos:,} NaN en features — manejados por scaler/HGB.")

        # ── 10.3 TimeSeriesSplit con K dinámico ──────────────────────────────────
        info(f"Iniciando TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} → {GPU_MSG}")

        # Construir diccionario de hiperparámetros XGBoost
        xgb_params = {
            "max_depth":        int(max_depth),
            "learning_rate":    float(learning_rate),
            "subsample":        float(subsample),
            "colsample_bytree": float(colsample_bytree),
            "reg_lambda":       float(reg_lambda),
            "reg_alpha":        float(reg_alpha),
            "gamma":            float(gamma),
            "min_child_weight": float(min_child_weight),
        }
        info("Hiperparámetros XGBoost: " + ", ".join(f"{k}={v}" for k,v in xgb_params.items()))

        df_clean = pd.concat([X, y], axis=1)
        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits, log_fn=kf_log, xgb_params=xgb_params
        )
        for l in kfold_logs: logs.append(l)

        if kf_resumen.empty:
            warn("TSS K-Fold no produjo resultados — verifica contaminantes en el dataset.")

        # ── 10.4 Curva de aprendizaje ─────────────────────────────────────────────
        info(f"Generando curva de aprendizaje para '{target_col}'…")
        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits
        )
        info("Curva de aprendizaje generada.")

        # ── 10.5 Modelo final con sub‑split interno (test 100% ciego) ────────────
        info(f"Entrenando modelo final sobre '{target_col}' (test blindado)…")
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)

        sub_corte   = int(len(X_tr) * 0.80)
        X_sub_tr    = X_tr.iloc[:sub_corte]
        X_val_sub   = X_tr.iloc[sub_corte:]
        y_sub_tr    = y_tr.iloc[:sub_corte]
        y_val_sub   = y_tr.iloc[sub_corte:]

        info(f"  Sub‑split interno — sub‑train: {len(X_sub_tr):,} | val interno: {len(X_val_sub):,} | test ciego: {len(X_te):,}")

        scaler_final  = StandardScaler()
        X_sub_tr_sc   = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc  = scaler_final.transform(X_val_sub)
        X_te_sc       = scaler_final.transform(X_te)

        X_tr_sc = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np = y_tr.values

        modelo_final = _construir_modelo(algoritmo, xgb_params)

        if "XGBoost" in algoritmo:
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                verbose=False,
            )
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)],
            )
        else:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)
        info("Modelo final entrenado — test set nunca fue expuesto al modelo.")

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )
        m_tr, m_te = _m(y_tr_np, y_pred_tr), _m(y_te.values, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15: warn(f"Posible overfitting en modelo final: ΔR² = {gap:.3f}")

        # ── 10.6 Markdown de métricas ────────────────────────────────────────────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                r2_color = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** | `{int(row['Features_X'])}` | `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` | `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` | {r2_color} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        r2_pct_tr = m_tr["R2"] * 100
        r2_pct_te = m_te["R2"] * 100

        metricas_md = f"""
## 📊 Surrogate Model — *{parroquia}*

### Validación Cronológica TimeSeriesSplit (K={k_splits}) — 6 Contaminantes REMMAQ

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> **Nota:** `TimeSeriesSplit` garantiza que cada fold de validación sea siempre posterior al bloque de entrenamiento → **cero data leakage temporal**.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

> **Test set 100% ciego**: el 20% final nunca fue expuesto al modelo.

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión del Modelo (R² %)** | `{r2_pct_tr:.2f}%` | `{r2_pct_te:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features base | `{len(feat_cols)}` columnas |
| Sub‑train / Val interno / Test | `{len(X_sub_tr):,}` / `{len(X_val_sub):,}` / `{len(X_te):,}` |
| Filtro pandemia | {pandemia_label} |
| Regularización | L2={xgb_params['reg_lambda']}, α={xgb_params['reg_alpha']}, max_depth={xgb_params['max_depth']} |
"""

        # ── 10.7 Permutation Importance ────────────────────────────────────────
        perm = permutation_importance(modelo_final, X_te_sc, y_te.values, n_repeats=8, random_state=42, scoring="r2")
        fi_df = pd.DataFrame({"Feature": feat_cols, "Importance": perm.importances_mean, "Std": perm.importances_std}).sort_values("Importance", ascending=False).reset_index(drop=True)

        # ── 10.8 Persistencia ──────────────────────────────────────────────────
        tag = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia,
                "kfold_resumen": kf_resumen, "kf_curvas": kf_curvas,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")
        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty: kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}": v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 10.9 Sesión ───────────────────────────────────────────────────────
        SESION.update({
            "modelo": modelo_final, "scaler": scaler_final,
            "feat_cols": feat_cols, "target": target_col,
            "parroquia": parroquia,
            "feat_stats": {col: {"min": float(X[col].min()), "max": float(X[col].max()), "mean": float(X[col].mean())} for col in feat_cols},
        })

        # ── 10.10 Figuras ──────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi   = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm   = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)

        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (f"### ❌ Librería faltante\n```\n{e}\n```\nInstala con: `pip install {pkg}`", *VACIO, _estado())
    except Exception as e:
        err(str(e))
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado()

# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS (igual)
# ──────────────────────────────────────────────────────────────────────────────
CSS = """
... (todo el CSS original, lo mantienes) ...
"""

# ──────────────────────────────────────────────────────────────────────────────
# 12.  UI con hiperparámetros XGBoost
# ──────────────────────────────────────────────────────────────────────────────
def predecir_desde_sesion(valores_json: str) -> str:
    # ... igual que antes ...
    pass  # se incluye el código original

MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)

def construir_app() -> gr.Blocks:
    with gr.Blocks(theme=gr.themes.Base(primary_hue="indigo", secondary_hue="sky", neutral_hue="slate",
                                        font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"]),
                   css=CSS, title="Surrogate Model — Calidad del Aire v9.3") as app:

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>XGBoost Hiperparámetros Interactivos · Blind Test · v9.3</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ── Panel izquierdo ───────────────────────────────────────────────
            with gr.Column(scale=1, min_width=350):
                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(label="CSVs detectados en el servidor",
                                                   choices=listar_csvs(), value=None, interactive=True, scale=5)
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(label="O arrastra / sube un CSV externo", file_types=[".csv"], type="filepath")
                    btn_detectar = gr.Button("🔍 Detectar columnas del CSV", variant="secondary", size="sm")
                    info_cols = gr.Markdown("_Pulsa 'Detectar columnas' para ver las variables disponibles._")

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(label="Variable objetivo (Target)",
                                                   choices=["PM25","PM10","NO2","O3","CO","SO2"], value="PM25", allow_custom_value=True, scale=3)
                        nombre_modelo_input = gr.Textbox(label="Nombre del modelo (.pkl)", value="surrogate_calidad_aire", scale=3)
                    algoritmo_radio = gr.Radio(label="Algoritmo de entrenamiento", choices=MODELOS, value=MODELOS[1])  # XGBoost
                    excluir_pandemia_chk = gr.Checkbox(label="🚫 Excluir datos de pandemia (2020–2021)", value=True)
                    with gr.Row():
                        train_ratio_slider = gr.Slider(label="Proporción de entrenamiento", minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3)
                        k_splits_slider = gr.Slider(label="Número de pliegues (K) TSCV", minimum=2, maximum=15, step=1, value=5, scale=2)
                        plot_days_slider = gr.Slider(label="Días a graficar (Test)", minimum=3, maximum=30, step=1, value=7, scale=2)

                gr.HTML("<div style='height:8px'/>")

                # ── Hiperparámetros XGBoost (desplegable) ─────────────────────
                with gr.Accordion("🧪 Hiperparámetros XGBoost", open=True):
                    with gr.Row():
                        max_depth_slider = gr.Slider(label="max_depth", minimum=2, maximum=10, step=1, value=5)
                        learning_rate_slider = gr.Slider(label="learning_rate", minimum=0.01, maximum=0.3, step=0.01, value=0.05)
                    with gr.Row():
                        subsample_slider = gr.Slider(label="subsample", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                        colsample_bytree_slider = gr.Slider(label="colsample_bytree", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                    with gr.Row():
                        reg_lambda_slider = gr.Slider(label="reg_lambda (L2)", minimum=0.0, maximum=10.0, step=0.5, value=2.0)
                        reg_alpha_slider = gr.Slider(label="reg_alpha (L1)", minimum=0.0, maximum=5.0, step=0.1, value=0.5)
                    with gr.Row():
                        gamma_slider = gr.Slider(label="gamma (min split loss)", minimum=0.0, maximum=5.0, step=0.1, value=0.0)
                        min_child_weight_slider = gr.Slider(label="min_child_weight", minimum=1, maximum=20, step=1, value=1)

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(value="_Esperando ejecución…_", elem_classes=["logs-box"])

            # ── Panel derecho ─────────────────────────────────────────────────
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(value="*Las métricas aparecerán aquí tras entrenar.*")
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(label="Serie temporal — últimos N días del test")
                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown("**Progreso por época / boosting round** con TimeSeriesSplit.\nLínea **continua** = train · línea **discontinua** = validación. Banda = ±1σ entre los pliegues temporales.\n\n- **XGBoost / LightGBM** → RMSE por iteración\n- **HistGradientBoosting** → R² por árbol")
                        fig_lc_output = gr.Plot(label="Curva de Aprendizaje — Error vs Iteración (TSS)")
                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(label="Importancia de variables (Permutation Δ R²)")
                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown("**Correlación de Pearson** entre todas las variables. La fila/columna del target aparece resaltada en naranja.")
                        fig_hm_output = gr.Plot(label="Mapa de calor — Correlación de Pearson")
                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown("Ingresa valores ambientales para obtener una estimación usando el **modelo en memoria** (entrenado en esta sesión).")
                        sesion_info = gr.Markdown("_⚠️ Entrena primero un modelo para habilitar esta sección._")
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0, visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals = gr.Textbox(visible=False, value="{}")
                        btn_predecir = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""<div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">Surrogate Model v9.3 · XGBoost Dinámico · Auto-GPU</div>""")

        # ── Funciones auxiliares UI ─────────────────────────────────────────
        def _refresh_pred_ui():
            feat_cols = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia = SESION.get("parroquia", "")
            target = SESION.get("target", "")
            sesion_msg = f"✅ Modelo en sesión: **{target}** · *{parroquia}* · {len(feat_cols)} features." if feat_cols else "_⚠️ Entrena primero un modelo._"
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st = feat_stats.get(col, {"min":0,"max":100,"mean":50})
                    updates.append(gr.Number(label=col, value=round(st["mean"],3), visible=True, interactive=True))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i]) for i in range(min(len(feat_cols), len(vals))) if vals[i] is not None}
            return json.dumps(d)

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta: return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        # ── Eventos ────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])
        btn_detectar.click(fn=_detectar_wrapper, inputs=[csv_dropdown, csv_upload], outputs=[target_input, info_cols])
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, k_splits_slider,
                # nuevos hiperparámetros
                max_depth_slider, learning_rate_slider, subsample_slider, colsample_bytree_slider,
                reg_lambda_slider, reg_alpha_slider, gamma_slider, min_child_weight_slider,
            ],
            outputs=[metricas_output, fig_pred_output, fig_fi_output, fig_hm_output, fig_lc_output, estado_output],
        ).then(fn=_refresh_pred_ui, inputs=[], outputs=[sesion_info] + slider_componentes)
        btn_predecir.click(fn=_construir_json, inputs=slider_componentes, outputs=json_vals).then(
            fn=predecir_desde_sesion, inputs=[json_vals], outputs=[resultado_pred]
        )

    return app

# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────
def _imprimir_ip_fallback():
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception: pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception: pass
    try:
        ip_pub = subprocess.check_output(["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"], text=True, timeout=7).strip()
        if ip_pub: print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception: pass

if __name__ == "__main__":
    app = construir_app()
    print("\n" + "═" * 64)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v9.3")
    print("  🔧  XGBoost Hiperparámetros Dinámicos")
    print("═" * 64)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 64)
    print("\n⏳ Generando link de acceso externo... por favor espera.\n")
    try:
        app.launch(server_name="0.0.0.0", server_port=None, share=True, max_threads=40, debug=True, show_error=True, prevent_thread_lock=False, quiet=False)
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40, debug=True, show_error=True, prevent_thread_lock=False, quiet=False)
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40, debug=True, show_error=True, prevent_thread_lock=False, quiet=False)


════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v9.3
  🔧  XGBoost Hiperparámetros Dinámicos
════════════════════════════════════════════════════════════════
  Hardware  : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  GPU activa: True
  Puerto    : automático (server_port=None)
════════════════════════════════════════════════════════════════

⏳ Generando link de acceso externo... por favor espera.



01:13:48 [INFO] Archivo subido: dataset_ml_carapungo.csv
01:13:48 [INFO] CSV cargado: 160,262 filas × 19 columnas
01:13:48 [INFO] Columna de tiempo: 'Timestamp'
01:13:48 [INFO] Pandemia excluida: años [2020, 2021] eliminados.
01:13:48 [WARNING] 217,722 NaN en features — manejados por scaler/HGB.
01:13:48 [INFO] Iniciando TimeSeriesSplit (K=5) · Algoritmo: XGBoost  (auto GPU/CPU)
01:13:48 [INFO] GPU disponible: True → GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
01:13:48 [INFO] Hiperparámetros XGBoost: max_depth=6, learning_rate=0.05, subsample=0.85, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.5, gamma=0.2, min_child_weight=3.0
01:13:49 [INFO]   PM25 | Fold 1/5 → MAE=6.023 RMSE=9.054 R²=0.537
01:13:49 [INFO]   PM25 | Fold 2/5 → MAE=5.857 RMSE=8.829 R²=0.469
01:13:50 [INFO]   PM25 | Fold 3/5 → MAE=4.941 RMSE=7.061 R²=0.528
01:13:50 [INFO]   PM25 | Fold 4/5 → MAE=6.620 RMSE=14.097 R²=0.350
01:13:50 [INFO]   PM25 | Fold 5/5 → MAE=6.903 RMSE=10.285 R²=0.249
01:13:51 [IN

XGBOOST con feature engineering version de deepseek

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : Arquitecto de Datos / MLOps Engineer
  Versión : 10.0  (Chronological Holdout · HistGB Penalizado · Warm-Start Curves)
  ─── Arquitectura de validación ────────────────────────────────────────────
  · Paso 1 — Reserva cronológica del último 20 % como "Test Ciego" (holdout).
  · Paso 2 — TimeSeriesSplit(n_splits=10) sobre el 80 % restante.
             Cada fold de validación es SIEMPRE posterior al train.
  · Paso 3 — Curvas de aprendizaje generadas con warm_start incremental
             (honesto en tiempo: no hay early stopping con split aleatorio).
  · Paso 4 — Modelo final entrenado sobre el 80 % con max_iter óptimo
             derivado del promedio de los 5 folds (mejor iter promedio).
  ─── Modelo único ───────────────────────────────────────────────────────────
  HistGradientBoostingRegressor (scikit-learn)
    · max_leaf_nodes  = 31    — complejidad geométrica controlada
    · l2_regularization = 5.0 — λ  (L2 explícita; reduce peso de lags 1h)
    · min_samples_leaf  = 50  — α-proxy (sparsity implícita tipo L1)
    · early_stopping  = False — controlado por max_iter + regularización
  ─── Correcciones heredadas de v9.0 ─────────────────────────────────────────
  [FIX-1]  CSV con cabeceras decorativas  → comment='#' en read_csv
  [FIX-2]  Inestabilidad en Jupyter/V100  → share=True + blocking correcto
  [FIX-3]  Filtro de pandemia 2020-2021   → exclusión cronológica explícita
  [FIX-4]  Auto-detección GPU             → nvidia-smi / torch.cuda
================================================================================
  Dependencias:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
      pip install torch  # opcional, solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

# Features base: atmosféricas + cíclicas + lags
FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

# Regularización HGB (especificación del docente)
HGB_L2_REG          = 5.0    # λ — penalización L2 explícita
HGB_MIN_SAMPLES_LEAF = 50    # α-proxy — sparsity implícita (tipo L1)
HGB_MAX_LEAF_NODES  = 31     # complejidad geométrica máxima
HGB_LEARNING_RATE   = 0.05

# Curva de aprendizaje: tamaño de cada incremento warm_start
BATCH_CURVA = 20             # árboles por paso de la curva (20 → 15 puntos a max_iter=300)

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU (HistGB funciona nativamente)"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  CARGA DE CSV  [FIX-1]
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


# ──────────────────────────────────────────────────────────────────────────────
# 5.  DETECCIÓN DINÁMICA DE COLUMNAS  [FIX-5]
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"):
        return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols  = df_head.columns.tolist()
        ts    = _detectar_timestamp(df_head)
        return (
            [c for c in cols if c != ts] if ts else cols,
            f"✅ {len(cols)} columnas. Timestamp: `{ts}`",
        )
    except Exception as e:
        return [], f"❌ Error: {e}"


# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(
    df: pd.DataFrame,
    target_col: str,
    timestamp_col: str,
    excluir_pandemia: bool = True,
) -> tuple:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_excl = n_antes - len(df)
        if n_excl:
            log.info(f"[FIX-3] {n_excl:,} registros de {ANOS_PANDEMIA} eliminados.")

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    df = df.dropna(subset=[target_col])
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in num_cols if c != target_col]

    if not feat_cols:
        raise ValueError("No se encontraron columnas numéricas para usar como features.")

    return df[feat_cols], df[target_col], feat_cols, df.index


def _dividir_cronologico(X, y, ratio=0.80):
    """División temporal estricta. El 100*(1-ratio)% final = Test Ciego."""
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols_disp = set(df.select_dtypes(include=[np.number]).columns)
    x_base    = [c for c in FEATURES_X_BASE if c in cols_disp]
    if x_base:
        return [c for c in x_base if c != target_col]
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disp if c not in excluir or f"{c}_lag" in c]


# ──────────────────────────────────────────────────────────────────────────────
# 7.  CONSTRUCCIÓN DEL MODELO HistGB CON REGULARIZACIÓN PENALIZADA
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo_hgb(max_iter: int = 300) -> HistGradientBoostingRegressor:
    """
    HistGradientBoostingRegressor con regularización estructural penalizada.

    Parámetros de control del overfitting (especificación docente):
      · l2_regularization = 5.0  (λ — penaliza pesos grandes en hojas)
      · min_samples_leaf  = 50   (α-proxy — fuerza hojas con más datos,
                                   efecto sparsity similar al L1)
      · max_leaf_nodes    = 31   (complejidad geométrica: máx 31 hojas/árbol)
      · early_stopping    = False — controlado por max_iter + regularización
                                    (honest para series temporales)
    """
    return HistGradientBoostingRegressor(
        max_iter          = max_iter,
        max_leaf_nodes    = HGB_MAX_LEAF_NODES,
        min_samples_leaf  = HGB_MIN_SAMPLES_LEAF,
        l2_regularization = HGB_L2_REG,
        learning_rate     = HGB_LEARNING_RATE,
        early_stopping    = False,    # [FIX-B] desactivado: era random → leakage temporal
        warm_start        = False,    # se gestiona externamente cuando se necesita
        random_state      = 42,
    )


# ──────────────────────────────────────────────────────────────────────────────
# 8.  CURVA DE APRENDIZAJE CON WARM-START INCREMENTAL
# ──────────────────────────────────────────────────────────────────────────────

def _curva_warmstart(
    X_tr:     np.ndarray,
    y_tr:     np.ndarray,
    X_val:    np.ndarray,
    y_val:    np.ndarray,
    max_iter: int   = 300,
    batch:    int   = BATCH_CURVA,
    log_fn          = None,
) -> tuple[list[float], list[float], HistGradientBoostingRegressor]:
    """
    Genera la curva de aprendizaje Train RMSE / Val RMSE via warm_start.

    Estrategia de early stopping cronológico honesto (Opción B):
      · El split X_tr / X_val es CRONOLÓGICO (nunca aleatorio).
      · El modelo arranca en batch=20 árboles y crece en incrementos de 'batch'.
      · En cada paso se registra el RMSE en train y en val.
      · El modelo resultante al final del loop es el completamente entrenado
        (max_iter total) para ese split → se reutiliza como predictor del fold.

    Complejidad real: O(max_iter) árboles totales por fold (warm_start es
    incremental, NO re-entrena desde cero en cada paso).

    Returns
    -------
    train_rmse : list[float]  — RMSE de entrenamiento en cada checkpoint
    val_rmse   : list[float]  — RMSE de validación en cada checkpoint
    modelo     : HistGradientBoostingRegressor  — modelo final de este split
    """
    if log_fn is None:
        log_fn = lambda m: None

    # Modelo con warm_start=True para entrenamiento incremental honesto
    model = HistGradientBoostingRegressor(
        max_iter          = batch,           # arranca con 'batch' árboles
        max_leaf_nodes    = HGB_MAX_LEAF_NODES,
        min_samples_leaf  = HGB_MIN_SAMPLES_LEAF,
        l2_regularization = HGB_L2_REG,
        learning_rate     = HGB_LEARNING_RATE,
        early_stopping    = False,
        warm_start        = True,            # ← clave: cada fit() añade 'batch' árboles
        random_state      = 42,
    )

    train_rmse: list[float] = []
    val_rmse:   list[float] = []

    # Iterar desde batch hasta max_iter en pasos de 'batch'
    for step in range(batch, max_iter + 1, batch):
        model.max_iter = step          # añadir 'batch' árboles más
        model.fit(X_tr, y_tr)          # warm_start: continúa desde el estado anterior

        rmse_tr = float(np.sqrt(mean_squared_error(y_tr,  model.predict(X_tr))))
        rmse_va = float(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))

        train_rmse.append(rmse_tr)
        val_rmse.append(rmse_va)

        if step % (batch * 5) == 0:
            log_fn(
                f"    iter {step:4d}/{max_iter} | "
                f"Train RMSE: {rmse_tr:.4f} | Val RMSE: {rmse_va:.4f}"
            )

    return train_rmse, val_rmse, model


# ──────────────────────────────────────────────────────────────────────────────
# 9.  PIPELINE TimeSeriesSplit — Validación Cronológica Completa
# ──────────────────────────────────────────────────────────────────────────────

def _entrenar_tss(
    df:         pd.DataFrame,
    target_col: str,
    max_iter:   int   = 300,
    n_splits:   int   = 5,
    log_fn            = None,
) -> tuple[pd.DataFrame, dict]:
    """
    Entrenamiento con TimeSeriesSplit(n_splits=10) + curvas de aprendizaje.

    Para CADA fold:
      1. Split cronológico en X_tr / X_val del fold.
      2. StandardScaler ajustado SOLO con X_tr del fold (sin leakage).
      3. _curva_warmstart() → entrena el modelo incremental y registra
         Train/Val RMSE en cada checkpoint de 'batch' árboles.
      4. Métricas del fold calculadas con el modelo final (max_iter árboles).

    Estructura de ventanas TimeSeriesSplit (ejemplo 5 splits, N muestras):
        Fold 1 → Train: [0 … k)        Val: [k … 2k)
        Fold 2 → Train: [0 … 2k)       Val: [2k … 3k)
        Fold 3 → Train: [0 … 3k)       Val: [3k … 4k)
        Fold 4 → Train: [0 … 4k)       Val: [4k … 5k)
        Fold 5 → Train: [0 … 5k)       Val: [5k … N)

    Returns
    -------
    df_metricas : DataFrame con MAE, RMSE, R² por fold
    curvas_dict : {target_col: {"train": [...], "val": [...], "metric": "RMSE"}}
    best_iter   : int — iteración óptima promedio entre los 5 folds
    """
    if log_fn is None:
        log_fn = log.info

    feat_cols = _resolver_features_x(df, target_col)
    y_vals    = df[target_col].dropna()
    X_vals    = df.loc[y_vals.index, feat_cols]

    mask_ok   = X_vals.notna().all(axis=1) & y_vals.notna()
    X_vals    = X_vals[mask_ok]
    y_vals    = y_vals[mask_ok]

    if len(X_vals) < n_splits * 50:
        raise ValueError(
            f"Datos insuficientes: {len(X_vals)} filas para {n_splits} folds "
            f"(mínimo recomendado: {n_splits * 50})."
        )

    X_arr = X_vals.values
    y_arr = y_vals.values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_metrics:    list[dict]  = []
    all_train_curves: list[list] = []    # K listas de RMSE por fold
    all_val_curves:   list[list] = []

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
        X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
        y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

        # StandardScaler ajustado SOLO con el train de esta ventana temporal
        scaler  = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_va_sc = scaler.transform(X_va)

        log_fn(
            f"  ── Fold {fold_idx}/{n_splits} | "
            f"Train: {len(X_tr):,}  Val: {len(X_va):,} ──"
        )

        # Entrenamiento incremental honesto con warm_start
        tr_curve, va_curve, modelo_fold = _curva_warmstart(
            X_tr_sc, y_tr, X_va_sc, y_va,
            max_iter=max_iter, batch=BATCH_CURVA, log_fn=log_fn,
        )

        all_train_curves.append(tr_curve)
        all_val_curves.append(va_curve)

        # Métricas del fold con el modelo completamente entrenado
        y_pred = modelo_fold.predict(X_va_sc)
        fold_metrics.append({
            "Fold": fold_idx,
            "N_train": len(X_tr),
            "N_val":   len(X_va),
            "MAE":  mean_absolute_error(y_va, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
            "R2":   r2_score(y_va, y_pred),
        })
        log_fn(
            f"  {target_col} | Fold {fold_idx}/{n_splits} → "
            f"MAE={fold_metrics[-1]['MAE']:.3f}  "
            f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
            f"R²={fold_metrics[-1]['R2']:.3f}"
        )

    # ── Derivar iteración óptima del promedio de los 5 folds ──────────────────
    min_curve_len = min(len(c) for c in all_val_curves)
    val_arr  = np.array([c[:min_curve_len] for c in all_val_curves])
    avg_val  = val_arr.mean(axis=0)
    best_idx = int(np.argmin(avg_val))
    best_iter = (best_idx + 1) * BATCH_CURVA
    log_fn(
        f"  Iteración óptima promedio (TSS): {best_iter} "
        f"(Val RMSE promedio mínimo: {avg_val[best_idx]:.4f})"
    )

    curvas_dict = {
        target_col: {
            "train":  all_train_curves,
            "val":    all_val_curves,
            "metric": "RMSE",
        }
    }

    return pd.DataFrame(fold_metrics), curvas_dict, best_iter


# ──────────────────────────────────────────────────────────────────────────────
# 10.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje(
    curvas:     dict,
    target_col: str,
    parroquia:  str = "",
    batch_size: int = BATCH_CURVA,
) -> plt.Figure:
    """
    Curva de aprendizaje Train RMSE / Val RMSE por iteración acumulada.
    Visualización idéntica a v9 (media ± σ entre los K folds TSS).

    Eje X : Árboles entrenados acumulados (batch_size × paso)
    Eje Y : RMSE (↓ mejor)
    ─ Línea continua   = Train RMSE (μ ± σ entre folds)
    ─ Línea discontinua = Val RMSE  (μ ± σ entre folds)
    ─ Líneas finas (α=0.10) = curvas individuales de cada fold
    """
    COLOR_TR = COLOR_REAL
    COLOR_VA = COLOR_PRED

    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    if not curvas or not curvas.get("train"):
        ax.text(
            0.5, 0.5, "No hay datos de curva de aprendizaje.",
            ha="center", va="center", color=TEXT_PLOT, fontsize=11,
            transform=ax.transAxes,
        )
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "RMSE")

    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto para graficar.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len = min(lengths)
    # Eje X en iteraciones reales (batch_size × paso)
    x = (np.arange(min_len) + 1) * batch_size

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    tr_arr  = np.array([c[:min_len] for c in train_curves])
    tr_mean = tr_arr.mean(axis=0)
    tr_std  = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_TR, alpha=0.09, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_TR, lw=2.2,
            label=f"Train {metric_name}  (μ TSS-Folds)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_TR, zorder=3)

    # ── VALIDATION ────────────────────────────────────────────────────────────
    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)

        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_VA, alpha=0.09, lw=0.75, zorder=2)
        ax.plot(x, va_mean, color=COLOR_VA, lw=2.2, linestyle="--",
                label=f"Validación {metric_name}  (μ TSS-Folds)", zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_VA, zorder=3)

        # Mejor iteración según val media
        best_idx  = int(np.argmin(va_mean))
        best_iter = x[best_idx]
        best_val  = va_mean[best_idx]

        ax.axvline(best_iter, color="#F59E0B", lw=1.8, linestyle=":", alpha=0.9, zorder=6,
                   label=f"Mejor iter: {best_iter}  ({metric_name}={best_val:.4f})")
        ax.scatter([best_iter], [best_val], color="#F59E0B", s=75, zorder=8,
                   edgecolors="#FCD34D", lw=1.5)
        offset = max(batch_size, int((x.max() - x.min()) * 0.02))
        ax.annotate(
            f"  ← {best_iter} árboles",
            xy=(best_iter, best_val), xytext=(best_iter + offset, best_val),
            color="#F59E0B", fontsize=8.5, va="center", zorder=9,
        )

    ax.axhline(tr_mean[-1], color=COLOR_TR, lw=0.7, linestyle=":", alpha=0.35, zorder=1)

    titulo = f"Curva de Aprendizaje — {target_col}  ·  HistGB penalizado  ·  TSS (K=5)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    _aplicar_estilo_ax(
        ax, titulo,
        "Árboles entrenados (iteraciones acumuladas)",
        f"{metric_name}  (↓ mejor)",
    )
    ax.legend(framealpha=0.22, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10, loc="upper right")

    nota = (
        f"Banda sombreada = ±1σ entre los {len(train_curves)} pliegues "
        f"(TimeSeriesSplit, ventanas expansivas).  "
        f"Batch warm-start = {batch_size} árboles/paso."
    )
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]

    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test Ciego)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 11.  PIPELINE PRINCIPAL  (callback del botón Entrenar)
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    target_col:       str,
    nombre_modelo:    str,
    max_iter:         int,
    plot_days:        int,
    train_ratio:      float,
    excluir_pandemia: bool,
):
    """
    Pipeline completo:
      1. Carga y preprocesamiento del CSV.
      2. División cronológica: 80% train / 20% test ciego.
      3. TimeSeriesSplit(K=5) + warm_start sobre el 80%.
      4. Derivar best_iter del promedio de los 5 folds.
      5. Modelo final entrenado sobre el 80% completo con best_iter.
      6. Evaluación en el 20% test ciego.
      7. Feature Importance (permutation en test ciego).
      8. Figuras: curva de aprendizaje, Real vs Predicho, FI, Heatmap.
    """
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")

    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-25:])

    # 6 outputs: metricas, fig_pred, fig_fi, fig_hm, fig_lc, estado
    VACIO = (None, None, None, None)

    try:
        # ── 11.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_hgb"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 11.2  Cargar y preprocesar ────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ Sin columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        X, y, feat_cols, idx = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: {ANOS_PANDEMIA}")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — manejados nativamente por HistGB.")

        # ── 11.3  División cronológica: 80% train | 20% test ciego ───────────
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        info(
            f"Split cronológico {int(train_ratio*100)}/{int((1-train_ratio)*100)}: "
            f"Train={len(X_tr):,}  |  Test ciego={len(X_te):,}"
        )
        info("El 20% test NO se expone durante TimeSeriesSplit ni warm_start.")

        # ── 11.4  TimeSeriesSplit K=5 + curvas de aprendizaje ─────────────────
        info(f"TimeSeriesSplit (K=5) · max_iter={max_iter} · batch={BATCH_CURVA}")
        info(f"Regularización: l2={HGB_L2_REG}  min_samples_leaf={HGB_MIN_SAMPLES_LEAF}  max_leaf_nodes={HGB_MAX_LEAF_NODES}")

        df_clean = pd.concat([X_tr, y_tr], axis=1)

        tss_logs = []
        def tss_log(m): log.info(m); tss_logs.append(m)

        kf_resumen, kf_curvas, best_iter = _entrenar_tss(
            df_clean, target_col, max_iter=max_iter, n_splits=10, log_fn=tss_log
        )
        for l in tss_logs:
            logs.append(l)

        info(f"TimeSeriesSplit completado — mejor iteración derivada: {best_iter} árboles")

        # ── 11.5  Modelo final sobre el 80% completo con best_iter ────────────
        # [FIX-B] Entrenamiento final con el best_iter derivado de los folds TSS.
        # El test set (X_te) permanece 100% ciego.
        final_max_iter = min(best_iter + 2 * BATCH_CURVA, max_iter)
        info(
            f"Entrenando modelo final sobre train completo: "
            f"max_iter={final_max_iter} (best_iter={best_iter} + buffer={2*BATCH_CURVA})"
        )

        scaler_final = StandardScaler()
        X_tr_sc      = scaler_final.fit_transform(X_tr)
        X_te_sc      = scaler_final.transform(X_te)

        modelo_final = _construir_modelo_hgb(max_iter=final_max_iter)
        modelo_final.fit(X_tr_sc, y_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)
        info("Modelo final entrenado — test ciego nunca fue expuesto.")

        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )

        m_tr = _m(y_tr.values, y_pred_tr)
        m_te = _m(y_te.values, y_pred_te)
        gap  = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting residual: ΔR² = {gap:.3f}")

        # ── 11.6  Markdown de métricas ────────────────────────────────────────
        hw_label = (
            f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}  "
            f"_(HistGB usa CPU nativo de scikit-learn)_"
        )
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        # Tabla TSS por fold
        filas_tss = []
        for _, row in kf_resumen.iterrows():
            r2_ico = "🟢" if row["R2"] >= 0.7 else ("🟡" if row["R2"] >= 0.5 else "🔴")
            filas_tss.append(
                f"| **{int(row['Fold'])}** "
                f"| `{int(row['N_train']):,}` "
                f"| `{int(row['N_val']):,}` "
                f"| `{row['MAE']:.3f}` "
                f"| `{row['RMSE']:.3f}` "
                f"| {r2_ico} `{row['R2']:.3f}` |"
            )
        # Fila de promedio ± std
        r2_mean = kf_resumen["R2"].mean()
        r2_std  = kf_resumen["R2"].std()
        r2_ico_m = "🟢" if r2_mean >= 0.7 else ("🟡" if r2_mean >= 0.5 else "🔴")
        filas_tss.append(
            f"| **μ ± σ** "
            f"| — | — "
            f"| `{kf_resumen['MAE'].mean():.3f} ± {kf_resumen['MAE'].std():.3f}` "
            f"| `{kf_resumen['RMSE'].mean():.3f} ± {kf_resumen['RMSE'].std():.3f}` "
            f"| {r2_ico_m} `{r2_mean:.3f} ± {r2_std:.3f}` |"
        )
        tabla_tss = "\n".join(filas_tss)

        overfitting_badge = (
            f"⚠️ **Posible overfitting residual** — ΔR² = `{gap:.3f}`"
            if gap > 0.15 else
            "✅ Sin señales de overfitting (regularización l2=5.0 + min_samples_leaf=50 activa)."
        )

        metricas_md = f"""
## 📊 Surrogate Model v10.0 — *{parroquia}*  ·  `{target_col}`

### Validación Cronológica TimeSeriesSplit (K=5) — Ventanas Expansivas

| Fold | N Train | N Val | MAE | RMSE | R² |
|:----:|:-------:|:-----:|:---:|:----:|:--:|
{tabla_tss}

> **Validación temporal honesta**: cada fold valida sobre observaciones
> **estrictamente posteriores** al bloque de entrenamiento (TimeSeriesSplit).
> La curva de aprendizaje se genera con `warm_start` cronológico, sin splits aleatorios.

---

### Modelo Final — Test Ciego 100 % (último {int((1-train_ratio)*100)} % de la serie)

> `max_iter` final = **{final_max_iter}** árboles
> (best_iter promedio TSS = {best_iter}, + buffer de {2*BATCH_CURVA}).

| Métrica | 🟦 Train | 🟧 Test Ciego |
|---------|:--------:|:-------------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión del Modelo (R² %)** | `{m_tr['R2']*100:.2f}%` | `{m_te['R2']*100:.2f}%` |

{overfitting_badge}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Target | `{target_col}` |
| Hardware | {hw_label} |
| Features X | `{len(feat_cols)}` columnas |
| Train / Test ciego | `{len(X_tr):,}` / `{len(X_te):,}` registros |
| max_leaf_nodes | `{HGB_MAX_LEAF_NODES}` |
| l2_regularization | `{HGB_L2_REG}` (λ) |
| min_samples_leaf | `{HGB_MIN_SAMPLES_LEAF}` (α-proxy) |
| Filtro pandemia | {pandemia_label} |
"""

        # ── 11.7  Feature Importance en el test ciego ─────────────────────────
        info("Calculando Permutation Importance en test ciego…")
        perm  = permutation_importance(
            modelo_final, X_te_sc, y_te.values,
            n_repeats=8, random_state=42, scoring="r2",
        )
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 11.8  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo":       modelo_final,
                "scaler":       scaler_final,
                "features":     feat_cols,
                "target":       target_col,
                "parroquia":    parroquia,
                "tss_resumen":  kf_resumen,
                "kf_curvas":    kf_curvas,
                "best_iter":    best_iter,
                "final_max_iter": final_max_iter,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_tss_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "R2_pct_test": m_te["R2"] * 100,
            "GPU": GPU_DISPONIBLE,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 11.9  Guardar en sesión ───────────────────────────────────────────
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": {
                col: {
                    "min":  float(X[col].min()),
                    "max":  float(X[col].max()),
                    "mean": float(X[col].mean()),
                }
                for col in feat_cols
            },
        })
        info("Sesión actualizada → pestaña Predicción lista.")

        # ── 11.10  Figuras ────────────────────────────────────────────────────
        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, parroquia, batch_size=BATCH_CURVA
        )
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm      = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Figuras generadas.")

        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado()
        )


# ──────────────────────────────────────────────────────────────────────────────
# 12.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:140px; overflow-y:auto; line-height:1.6; }
.reg-badge { background:rgba(99,102,241,.12); border:1px solid #4338CA;
    border-radius:8px; padding:8px 14px; font-size:.78rem; color:#C7D2FE; margin-top:6px; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 13.  UI
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return "### ⚠️ Entrena primero un modelo."
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]
        extra = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    calidad = "🟢 **Buena**"
            elif pred <= 35:  calidad = "🟡 **Moderada**"
            elif pred <= 55:  calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else:             calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice AQI:** {calidad}"
        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"


gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 {GPU_MSG}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU · HistGB nativo scikit-learn</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v10",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>HistGB Penalizado · TimeSeriesSplit · Holdout Ciego · Warm-Start Curves · v10.0</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════
            # PANEL IZQUIERDO — CONTROLES
            # ══════════════════════════════════
            with gr.Column(scale=1, min_width=350):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar columnas del CSV", variant="secondary", size="sm"
                    )
                    info_cols = gr.Markdown(
                        "_Pulsa 'Detectar columnas' para ver las variables disponibles._"
                    )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")

                    # Badge informativo de regularización
                    gr.HTML(
                        f'<div class="reg-badge">'
                        f'<b>HistGB</b> · λ(L2)={HGB_L2_REG} · α-proxy(min_samples)={HGB_MIN_SAMPLES_LEAF}'
                        f' · max_leaf_nodes={HGB_MAX_LEAF_NODES}'
                        f'</div>'
                    )

                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25", allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_hgb_carapungo", scale=3,
                        )

                    max_iter_slider = gr.Slider(
                        label="Máximo de árboles (max_iter)",
                        minimum=50, maximum=500, step=10, value=300,
                        info=f"La iteración óptima se deriva automáticamente del promedio TSS. "
                             f"Curva generada en pasos de {BATCH_CURVA} árboles.",
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina 2020-2021 para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Holdout split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80,
                            info="El (1-ratio)% final = Test Ciego 100% intocable.",
                            scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar (Test)",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════
            # PANEL DERECHO — 6 TABS
            # ══════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — Test Ciego (últimos N días)"
                        )

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Evolución del RMSE** a lo largo de los árboles entrenados.\n\n"
                            "- Generada con **warm_start incremental** (cronológicamente honesta).\n"
                            "- Muestra la **media ± σ** de las 5 ventanas TimeSeriesSplit.\n"
                            "- Línea **continua** = Train RMSE · Línea **discontinua** = Val RMSE.\n"
                            "- La **línea vertical** marca la mejor iteración promedio.\n\n"
                            "> Sin early stopping interno aleatorio: el split de validación "
                            "es siempre **cronológico** (ventana posterior al train)."
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — HistGB warm_start (TSS K=5)"
                        )

                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown(
                            "**Importancia por permutación** calculada sobre el **Test Ciego** "
                            "(el 20% final, nunca expuesto al modelo durante el entrenamiento)."
                        )
                        fig_fi_output = gr.Plot(
                            label="Permutation Importance (Δ R² en Test Ciego)"
                        )

                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La fila/columna del target aparece resaltada en naranja."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "usando el **modelo en memoria** (entrenado en esta sesión)."
                        )
                        sesion_info   = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v10.0 · HistGB · TimeSeriesSplit · Holdout Ciego · Warm-Start Honesto
        </div>
        """)

        # ── Funciones auxiliares de la UI ─────────────────────────────────────

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")
            sesion_msg = (
                f"✅ **{target}** · *{parroquia}* · {len(feat_cols)} features."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col, value=round(st["mean"], 3),
                        visible=True, interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            import json
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            return json.dumps(d)

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (
                (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None))
                or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            )
            if not ruta:
                return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo."
            cols, msg = obtener_columnas_csv(ruta)
            choices   = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])
        btn_detectar.click(
            fn=_detectar_wrapper,
            inputs=[csv_dropdown, csv_upload],
            outputs=[target_input, info_cols],
        )
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                max_iter_slider, plot_days_slider, train_ratio_slider, excluir_pandemia_chk,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_fi_output,
                fig_hm_output, fig_lc_output, estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )
        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 14.  PUNTO DE ENTRADA — RESILIENTE EN JUPYTER / SERVIDOR REMOTO
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 66)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v10.0")
    print("  🔒  HistGB · TimeSeriesSplit · Holdout Ciego · Warm-Start")
    print("═" * 66)
    print(f"  Hardware    : {GPU_MSG}")
    print(f"  l2_reg      : {HGB_L2_REG}  |  min_samples_leaf: {HGB_MIN_SAMPLES_LEAF}")
    print(f"  max_leaf    : {HGB_MAX_LEAF_NODES}  |  batch warm-start: {BATCH_CURVA}")
    print("  Puerto      : automático (server_port=None)")
    print("═" * 66)

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

19:05:09 [INFO] GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB



══════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v10.0
  🔒  HistGB · TimeSeriesSplit · Holdout Ciego · Warm-Start
══════════════════════════════════════════════════════════════════
  Hardware    : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  l2_reg      : 5.0  |  min_samples_leaf: 50
  max_leaf    : 31  |  batch warm-start: 20
  Puerto      : automático (server_port=None)
══════════════════════════════════════════════════════════════════


19:05:32 [INFO] Archivo subido: dataset_ml_carapungo.csv
19:05:33 [INFO] CSV cargado: 160,262 filas × 19 columnas
19:05:33 [INFO] Timestamp: 'Timestamp'
19:05:33 [INFO] [FIX-3] 23 registros de [2020, 2021] eliminados.
19:05:33 [INFO] Pandemia excluida: [2020, 2021]
19:05:33 [WARNING] 217,722 NaN en features — manejados nativamente por HistGB.
19:05:33 [INFO] Split cronológico 80/19: Train=128,191  |  Test ciego=32,048
19:05:33 [INFO] El 20% test NO se expone durante TimeSeriesSplit ni warm_start.
19:05:33 [INFO] TimeSeriesSplit (K=5) · max_iter=300 · batch=20
19:05:33 [INFO] Regularización: l2=5.0  min_samples_leaf=50  max_leaf_nodes=31
19:05:33 [INFO]   ── Fold 1/5 | Train: 5,430  Val: 5,425 ──
19:05:34 [INFO]     iter  100/300 | Train RMSE: 9.1367 | Val RMSE: 8.0077
19:05:34 [INFO]     iter  200/300 | Train RMSE: 8.5315 | Val RMSE: 8.0688
19:05:35 [INFO]     iter  300/300 | Train RMSE: 8.1066 | Val RMSE: 8.1644
19:05:35 [INFO]   PM25 | Fold 1/5 → MAE=6.009  RMSE=8.164  R²=0.549
19:05

XGBOOST con  Feature Engineering

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 10.2 (Fix: Feature Engineering con selección manual)
  Correcciones incluidas:
      [FIX-1/2/3/4/5] CSV, GPU, Pandemia, detección columnas
      [A] TimeSeriesSplit dinámico (K ajustable)
      [B] Sub‑split cronológico 80/20 → test ciego
      [B2] Precisión (R² %) en tabla final
      [C] Regularización balanceada (HGB, XGB, LGB)
      [HP] Controles interactivos para hiperparámetros XGBoost
      [FE] Feature Engineering: rolling means, std, diferencias,
           interacciones (temp*hum, u/v viento), día semana, finde
      [SEL] Selección manual de las mejores 40 características (usa feature_importances_)
      [FIX] Checkboxes funcionales (CSS específico para input[type="checkbox"])
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os, sys, warnings, logging, traceback, subprocess, pickle, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"

# ──────────────────────────────────────────────────────────────────────────────
# 2.  [FIX-4]  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────
def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception: pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError: pass
    try:
        import cupy
        return True, "GPU detectada (cupy disponible)"
    except ImportError: pass
    return False, "No se detectó GPU — se usará CPU"

GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)

def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE else {"tree_method": "hist", "device": "cpu"}
def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"

# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────
def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]

def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)

# ──────────────────────────────────────────────────────────────────────────────
# 4.  [FIX-1]  CARGA DE CSV CON CABECERAS DECORATIVAS
# ──────────────────────────────────────────────────────────────────────────────
def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e

# ──────────────────────────────────────────────────────────────────────────────
# 5.  [FIX-5]  DETECCIÓN DINÁMICA DE COLUMNAS
# ──────────────────────────────────────────────────────────────────────────────
def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos: return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception: continue
    return None

def obtener_columnas_csv(ruta: str) -> tuple[list[str], str]:
    if not ruta or ruta.startswith("(No"): return [], "⚠️ Selecciona un archivo válido."
    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        cols = df_head.columns.tolist()
        ts = _detectar_timestamp(df_head)
        cols_sin_ts = [c for c in cols if c != ts] if ts else cols
        return cols_sin_ts, f"✅ {len(cols)} columnas detectadas. Timestamp: `{ts}`"
    except Exception as e:
        return [], f"❌ Error al leer encabezados: {e}"

# ──────────────────────────────────────────────────────────────────────────────
# 6.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────
def _preprocesar(df, target_col, timestamp_col, excluir_pandemia=True):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], infer_datetime_format=True, errors="coerce")
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols: df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=[target_col])
    return df

def _dividir_cronologico(X, y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]

def _resolver_features_x(df, target_col):
    """Devuelve todas las columnas numéricas que no sean contaminantes ni el target."""
    cols_disp = set(df.select_dtypes(include=[np.number]).columns)
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols_disp if c not in excluir and c != target_col]

# ──────────────────────────────────────────────────────────────────────────────
# 7.  [FE]  FEATURE ENGINEERING AVANZADO
# ──────────────────────────────────────────────────────────────────────────────
def _agregar_features_avanzadas(df, target_col):
    df = df.copy()
    contaminantes = [c for c in CONTAMINANTES_Y if c in df.columns]

    for col in contaminantes:
        if col not in df.columns: continue
        for window in [3, 6, 12, 24]:
            roll_mean = df[col].rolling(window, min_periods=1).mean()
            df[f"{col}_roll_mean_{window}h"] = roll_mean
            if window >= 6:
                roll_std = df[col].rolling(window, min_periods=2).std()
                df[f"{col}_roll_std_{window}h"] = roll_std
        for lag in [1, 3, 6, 12, 24]:
            diff = df[col].diff(lag)
            df[f"{col}_diff_{lag}h"] = diff

    if "Temperatura" in df.columns and "Humedad" in df.columns:
        df["temp_hum"] = df["Temperatura"] * df["Humedad"]
    if "Temperatura" in df.columns and "Viento_Velocidad" in df.columns:
        df["temp_wind"] = df["Temperatura"] * df["Viento_Velocidad"]
    if "Viento_Direccion" in df.columns and "Viento_Velocidad" in df.columns:
        dir_rad = np.radians(df["Viento_Direccion"])
        df["wind_u"] = df["Viento_Velocidad"] * np.cos(dir_rad)
        df["wind_v"] = df["Viento_Velocidad"] * np.sin(dir_rad)

    if hasattr(df.index, "weekday"):
        df["dia_semana"] = df.index.weekday
        df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
        df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
        df["es_finde"] = (df["dia_semana"] >= 5).astype(int)

    df = df.dropna()
    return df

# ──────────────────────────────────────────────────────────────────────────────
# 8.  [C]  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────
def _construir_modelo(algoritmo: str, params: dict = None):
    if params is None: params = {}
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000,
            max_depth=params.get("max_depth", 5),
            learning_rate=params.get("learning_rate", 0.05),
            subsample=params.get("subsample", 0.8),
            colsample_bytree=params.get("colsample_bytree", 0.8),
            reg_lambda=params.get("reg_lambda", 2.0),
            reg_alpha=params.get("reg_alpha", 0.5),
            gamma=params.get("gamma", 0.0),
            min_child_weight=params.get("min_child_weight", 1),
            early_stopping_rounds=50,
            eval_metric="rmse",
            random_state=42, verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000,
            max_depth=params.get("max_depth", 5),
            learning_rate=params.get("learning_rate", 0.05),
            subsample=params.get("subsample", 0.8),
            colsample_bytree=params.get("colsample_bytree", 0.8),
            reg_lambda=params.get("reg_lambda", 2.0),
            reg_alpha=params.get("reg_alpha", 0.5),
            min_child_samples=params.get("min_child_weight", 30),
            metric="rmse",
            device=_lgbm_device(),
            random_state=42, verbose=-1,
        )
    else:
        return HistGradientBoostingRegressor(
            max_iter=1000, early_stopping=True, n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=params.get("max_depth", 5),
            min_samples_leaf=params.get("min_child_weight", 30),
            learning_rate=params.get("learning_rate", 0.05),
            l2_regularization=params.get("reg_lambda", 1.0),
            random_state=42,
        )

# ──────────────────────────────────────────────────────────────────────────────
# 9.  PIPELINE TimeSeriesSplit
# ──────────────────────────────────────────────────────────────────────────────
def _extraer_curvas_fold(modelo, algoritmo):
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name

def _entrenar_kfold(df, algoritmo, n_splits=5, log_fn=None, xgb_params=None):
    if log_fn is None: log_fn = log.info
    tscv = TimeSeriesSplit(n_splits=n_splits)
    filas = []
    cols_df = set(df.columns)
    all_curves = {}

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible en el dataset — omitido.")
            continue
        feat_cols = _resolver_features_x(df, contaminante)
        y_vals = df[contaminante].dropna()
        X_vals = df.loc[y_vals.index, feat_cols]
        mask_ok = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]
        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue
        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics = []
        fold_curves_tr = []
        fold_curves_va = []
        metric_name_cv = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            scaler = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_va_sc = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo, xgb_params)

            if "XGBoost" in algoritmo:
                modelo.fit(X_tr_sc, y_tr, eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)], verbose=False)
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(X_tr_sc, y_tr, eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
            else:
                modelo.fit(X_tr_sc, y_tr)

            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h: fold_curves_tr.append(tr_h)
            if va_h: fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(f"  {contaminante} | Fold {fold_idx}/{n_splits} → MAE={fold_metrics[-1]['MAE']:.3f} RMSE={fold_metrics[-1]['RMSE']:.3f} R²={fold_metrics[-1]['R2']:.3f}")

        all_curves[contaminante] = {
            "train": fold_curves_tr, "val": fold_curves_va, "metric": metric_name_cv,
        }
        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X": len(feat_cols),
            "MAE_mean":  mf["MAE"].mean(),
            "MAE_std":   mf["MAE"].std(),
            "RMSE_mean": mf["RMSE"].mean(),
            "RMSE_std":  mf["RMSE"].std(),
            "R2_mean":   mf["R2"].mean(),
            "R2_std":    mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves

# ──────────────────────────────────────────────────────────────────────────────
# 10.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────
def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]
    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"], label="Real", color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5, linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig

def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia: titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig

def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2: return None
    corr = num_df.corr(method="pearson")
    n = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False, edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False, edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia: titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig

def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)

def _fig_curva_aprendizaje(curvas, target_col, algoritmo, parroquia="", k_splits=5):
    COLOR_TR = COLOR_REAL
    COLOR_VA = COLOR_PRED
    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5, "No hay datos de curva de aprendizaje.", ha="center", va="center", color=TEXT_PLOT, fontsize=11, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout(); return fig
    train_curves = curvas["train"]
    val_curves = curvas.get("val", [])
    metric_name = curvas.get("metric", "Score")
    is_r2 = (metric_name == "R²")
    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto.", ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout(); return fig
    min_len = min(lengths)
    x = np.arange(min_len)
    tr_arr = np.array([c[:min_len] for c in train_curves])
    tr_mean = tr_arr.mean(axis=0)
    tr_std = tr_arr.std(axis=0)
    for curve in tr_arr: ax.plot(x, curve, color=COLOR_TR, alpha=0.10, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_TR, lw=2.2, label=f"Train — {metric_name}  (μ TSS-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std, alpha=0.14, color=COLOR_TR, zorder=3)
    if val_curves:
        va_arr = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0)
        va_std = va_arr.std(axis=0)
        for curve in va_arr: ax.plot(x, curve, color=COLOR_VA, alpha=0.10, lw=0.75, zorder=2)
        va_label = f"Validación TSS-Fold — {metric_name}  (μ)" if not is_r2 else f"Validación interna HGB — {metric_name}  (μ TSS-Fold)"
        ax.plot(x, va_mean, color=COLOR_VA, lw=2.2, linestyle="--", label=va_label, zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std, alpha=0.14, color=COLOR_VA, zorder=3)
        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val = va_mean[best_iter]
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":", alpha=0.88, label=f"Mejor iteración: {best_iter} ({metric_name}={best_val:.4f})", zorder=6)
        offset = max(1, int(min_len * 0.02))
        ax.annotate(f" iter {best_iter}", xy=(best_iter, best_val), xytext=(best_iter + offset, best_val), color="#F59E0B", fontsize=8.5, va="center", zorder=7)
    ax.axhline(tr_mean[-1], color=COLOR_TR, lw=0.8, linestyle=":", alpha=0.40, zorder=1)
    algo_label = algoritmo.split("(")[0].strip()
    titulo = f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  TimeSeriesSplit (K={k_splits})"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    nota = "Nota HGB: Validación = fracción interna 10% de early stopping." if is_r2 else "Nota: banda sombreada = ±1σ entre pliegues temporales (TimeSeriesSplit)."
    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=9, loc="best")
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig

# ──────────────────────────────────────────────────────────────────────────────
# 11.  PIPELINE PRINCIPAL (con selección manual de características)
# ──────────────────────────────────────────────────────────────────────────────
def entrenar(
    csv_local, csv_upload, target_col, nombre_modelo, algoritmo,
    plot_days, train_ratio, excluir_pandemia, k_splits,
    max_depth, learning_rate, subsample, colsample_bytree,
    reg_lambda, reg_alpha, gamma, min_child_weight,
    usar_feature_engineering,
):
    logs: list[str] = []
    def info(m): log.info(m); logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️ {m}")
    def err(m): log.error(m); logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs)
    VACIO = (None, None, None, None)

    try:
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ni subió ningún archivo.", *VACIO, _estado()
        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia = Path(ruta_csv).stem.replace("_", " ").title()

        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Columna de tiempo: '{ts_col}'")
        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (f"### ❌ Target **`{target_col}`** no encontrado.\n\n**Columnas disponibles:** `{cols_disp}`", *VACIO, _estado())

        df_prep = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia: info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")
        y = df_prep[target_col]
        X = df_prep.drop(columns=[target_col])

        if usar_feature_engineering:
            info("Aplicando Feature Engineering avanzado...")
            df_full = _agregar_features_avanzadas(pd.concat([X, y], axis=1), target_col)
            y = df_full[target_col]
            X = df_full.drop(columns=[target_col])
            X_tr_sel, _, y_tr_sel, _ = _dividir_cronologico(X, y, train_ratio)
            info(f"Seleccionando las 40 mejores características (entre {X_tr_sel.shape[1]})...")
            # ── Selección robusta con RandomForest (siempre tiene feature_importances_) ──
            from sklearn.ensemble import RandomForestRegressor
            rf_selector = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
            rf_selector.fit(X_tr_sel, y_tr_sel)
            importances = rf_selector.feature_importances_
            top_indices = np.argsort(importances)[-40:]   # 40 más importantes
            selected_cols = X.columns[top_indices]
            X = X[selected_cols]
            feat_cols = list(X.columns)
            info(f"Características seleccionadas: {len(feat_cols)}")
        else:
            num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
            feat_cols = [c for c in num_cols if c != target_col]
            X = X[feat_cols]
            info(f"Usando {len(feat_cols)} features originales.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos: warn(f"{n_nulos:,} NaN en features — se eliminarán filas afectadas.")
        mask = X.notna().all(axis=1)
        X = X.loc[mask]
        y = y.loc[mask]

        df_clean = pd.concat([X, y], axis=1)
        info(f"Total de columnas en df_clean para K-Fold: {df_clean.shape[1]} (features: {len(feat_cols)})")

        xgb_params = {
            "max_depth": int(max_depth), "learning_rate": float(learning_rate),
            "subsample": float(subsample), "colsample_bytree": float(colsample_bytree),
            "reg_lambda": float(reg_lambda), "reg_alpha": float(reg_alpha),
            "gamma": float(gamma), "min_child_weight": float(min_child_weight),
        }
        info("Hiperparámetros XGBoost: " + ", ".join(f"{k}={v}" for k,v in xgb_params.items()))

        info(f"Iniciando TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} → {GPU_MSG}")

        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)
        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits, log_fn=kf_log, xgb_params=xgb_params
        )
        for l in kfold_logs: logs.append(l)
        if kf_resumen.empty:
            warn("TSS K-Fold no produjo resultados.")

        fig_lc = _fig_curva_aprendizaje(kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits)

        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        sub_corte = int(len(X_tr) * 0.80)
        X_sub_tr = X_tr.iloc[:sub_corte]
        X_val_sub = X_tr.iloc[sub_corte:]
        y_sub_tr = y_tr.iloc[:sub_corte]
        y_val_sub = y_tr.iloc[sub_corte:]

        scaler_final = StandardScaler()
        X_sub_tr_sc = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc = scaler_final.transform(X_val_sub)
        X_te_sc = scaler_final.transform(X_te)
        X_tr_sc = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np = y_tr.values

        modelo_final = _construir_modelo(algoritmo, xgb_params)
        if "XGBoost" in algoritmo:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values, eval_set=[(X_val_sub_sc, y_val_sub.values)], verbose=False)
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values, eval_set=[(X_val_sub_sc, y_val_sub.values)],
                            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        else:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)

        def _m(yt, yp):
            return dict(MAE=mean_absolute_error(yt, yp),
                       RMSE=float(np.sqrt(mean_squared_error(yt, yp))),
                       R2=r2_score(yt, yp))
        m_tr, m_te = _m(y_tr_np, y_pred_tr), _m(y_te.values, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15: warn(f"Posible overfitting en modelo final: ΔR² = {gap:.3f}")

        hw_label = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"
        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                r2_color = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** | `{int(row['Features_X'])}` | `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` | `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` | {r2_color} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        r2_pct_tr = m_tr["R2"] * 100
        r2_pct_te = m_te["R2"] * 100

        metricas_md = f"""
## 📊 Surrogate Model — *{parroquia}*

### Validación Cronológica TimeSeriesSplit (K={k_splits}) — 6 Contaminantes REMMAQ

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> **Nota:** `TimeSeriesSplit` garantiza que cada fold de validación sea siempre posterior al bloque de entrenamiento → **cero data leakage temporal**.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

> **Test set 100% ciego**: el 20% final nunca fue expuesto al modelo.

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión del Modelo (R² %)** | `{r2_pct_tr:.2f}%` | `{r2_pct_te:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features base | `{len(feat_cols)}` columnas |
| Sub‑train / Val interno / Test | `{len(X_sub_tr):,}` / `{len(X_val_sub):,}` / `{len(X_te):,}` |
| Filtro pandemia | {pandemia_label} |
| Feature Engineering | {'✅ Activado' if usar_feature_engineering else '❌ Desactivado'} |
| Regularización | L2={xgb_params['reg_lambda']}, α={xgb_params['reg_alpha']}, max_depth={xgb_params['max_depth']} |
"""

        perm = permutation_importance(modelo_final, X_te_sc, y_te.values, n_repeats=8, random_state=42, scoring="r2")
        fi_df = pd.DataFrame({"Feature": feat_cols, "Importance": perm.importances_mean, "Std": perm.importances_std}).sort_values("Importance", ascending=False).reset_index(drop=True)

        tag = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia,
                "kfold_resumen": kf_resumen, "kf_curvas": kf_curvas,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")
        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty: kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}": v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        SESION.update({
            "modelo": modelo_final, "scaler": scaler_final,
            "feat_cols": feat_cols, "target": target_col,
            "parroquia": parroquia,
            "feat_stats": {col: {"min": float(X[col].min()), "max": float(X[col].max()), "mean": float(X[col].mean())} for col in feat_cols},
        })

        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi   = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm   = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)

        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (f"### ❌ Librería faltante\n```\n{e}\n```\nInstala con: `pip install {pkg}`", *VACIO, _estado())
    except Exception as e:
        err(str(e))
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado()

# ──────────────────────────────────────────────────────────────────────────────
# 12.  CSS CORREGIDO (CHECKBOX FUNCIONAL)
# ──────────────────────────────────────────────────────────────────────────────
CSS = """
:root {
    --bg: #0F172A; --card: #1E293B; --input: #0D1525; --border: #334155;
    --accent: #6366F1; --accentH: #818CF8; --text: #F1F5F9; --muted: #94A3B8;
    --ok: #22C55E; --warn: #F59E0B; --err: #EF4444; --r: 10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important; font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important; border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important; border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important; font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important; color:#fff !important; border:none !important; border-radius:8px !important; font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important; box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important; border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900; background:linear-gradient(90deg,#6366F1,#38BDF8); -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem; font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important; border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace; font-size:.76rem; color:#7DD3FC; max-height:130px; overflow-y:auto; line-height:1.6; }

/* Fix checkboxes clickability */
input[type="checkbox"] {
    -webkit-appearance: checkbox !important;
    appearance: checkbox !important;
    width: 18px;
    height: 18px;
    margin-right: 8px;
    cursor: pointer;
    vertical-align: middle;
}
.gr-checkbox label {
    display: flex !important;
    align-items: center;
}
"""

# ──────────────────────────────────────────────────────────────────────────────
# 13.  UI
# ──────────────────────────────────────────────────────────────────────────────
def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return "### ⚠️ No hay modelo entrenado en la sesión.\nPrimero entrena un modelo."
    try:
        vals = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"])) for col in feat_cols}
        X_input = pd.DataFrame([row])
        scaler = SESION.get("scaler")
        X_sc = scaler.transform(X_input) if scaler else X_input.values
        pred = float(SESION["modelo"].predict(X_sc)[0])
        target = SESION["target"]
        parroquia = SESION["parroquia"]
        extra = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12: calidad = "🟢 **Buena**"
            elif pred <= 35: calidad = "🟡 **Moderada**"
            elif pred <= 55: calidad = "🟠 **Insalubre para grupos sensibles**"
            elif pred <= 150: calidad = "🔴 **Insalubre**"
            else: calidad = "🟣 **Muy insalubre / Peligrosa**"
            extra = f"\n\n**Índice de calidad del aire:** {calidad}"
        return f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n| Campo | Valor |\n|-------|-------|\n| **{target} estimado** | `{pred:.3f} µg/m³` |\n| Parroquia | `{parroquia}` |\n| Features usadas | `{len(feat_cols)}` |{extra}"
    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"

MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]
gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)

def construir_app() -> gr.Blocks:
    with gr.Blocks(theme=gr.themes.Base(primary_hue="indigo", secondary_hue="sky", neutral_hue="slate",
                                        font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"]),
                   css=CSS, title="Surrogate Model — Calidad del Aire v10.2") as app:

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Feature Engineering Avanzado · XGBoost Dinámico · Blind Test · v10.2</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):
            with gr.Column(scale=1, min_width=350):
                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(label="CSVs detectados en el servidor",
                                                   choices=listar_csvs(), value=None, interactive=True, scale=5)
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(label="O arrastra / sube un CSV externo", file_types=[".csv"], type="filepath")
                    btn_detectar = gr.Button("🔍 Detectar columnas del CSV", variant="secondary", size="sm")
                    info_cols = gr.Markdown("_Pulsa 'Detectar columnas' para ver las variables disponibles._")

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(label="Variable objetivo (Target)",
                                                   choices=["PM25","PM10","NO2","O3","CO","SO2"], value="PM25", allow_custom_value=True, scale=3)
                        nombre_modelo_input = gr.Textbox(label="Nombre del modelo (.pkl)", value="surrogate_calidad_aire", scale=3)
                    algoritmo_radio = gr.Radio(label="Algoritmo de entrenamiento", choices=MODELOS, value=MODELOS[1])
                    excluir_pandemia_chk = gr.Checkbox(label="🚫 Excluir datos de pandemia (2020–2021)", value=True)
                    usar_fe_ck = gr.Checkbox(label="🧪 Activar Feature Engineering (rolling stats, diferencias, interacciones)", value=True)
                    with gr.Row():
                        train_ratio_slider = gr.Slider(label="Proporción de entrenamiento", minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3)
                        k_splits_slider = gr.Slider(label="Número de pliegues (K) TSCV", minimum=2, maximum=15, step=1, value=5, scale=2)
                        plot_days_slider = gr.Slider(label="Días a graficar (Test)", minimum=3, maximum=30, step=1, value=7, scale=2)

                gr.HTML("<div style='height:8px'/>")

                with gr.Accordion("🧪 Hiperparámetros XGBoost", open=True):
                    with gr.Row():
                        max_depth_slider = gr.Slider(label="max_depth", minimum=2, maximum=10, step=1, value=5)
                        learning_rate_slider = gr.Slider(label="learning_rate", minimum=0.01, maximum=0.3, step=0.01, value=0.05)
                    with gr.Row():
                        subsample_slider = gr.Slider(label="subsample", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                        colsample_bytree_slider = gr.Slider(label="colsample_bytree", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                    with gr.Row():
                        reg_lambda_slider = gr.Slider(label="reg_lambda (L2)", minimum=0.0, maximum=10.0, step=0.5, value=2.0)
                        reg_alpha_slider = gr.Slider(label="reg_alpha (L1)", minimum=0.0, maximum=5.0, step=0.1, value=0.5)
                    with gr.Row():
                        gamma_slider = gr.Slider(label="gamma (min split loss)", minimum=0.0, maximum=5.0, step=0.1, value=0.0)
                        min_child_weight_slider = gr.Slider(label="min_child_weight", minimum=1, maximum=20, step=1, value=1)

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(value="_Esperando ejecución…_", elem_classes=["logs-box"])

            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(value="*Las métricas aparecerán aquí tras entrenar.*")
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(label="Serie temporal — últimos N días del test")
                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown("**Progreso por época / boosting round** con TimeSeriesSplit.")
                        fig_lc_output = gr.Plot(label="Curva de Aprendizaje — Error vs Iteración (TSS)")
                    with gr.Tab("🔍 Feature Importance"):
                        fig_fi_output = gr.Plot(label="Importancia de variables (Permutation Δ R²)")
                    with gr.Tab("📊 Análisis de Dependencias"):
                        gr.Markdown("**Correlación de Pearson** entre todas las variables.")
                        fig_hm_output = gr.Plot(label="Mapa de calor — Correlación de Pearson")
                    with gr.Tab("🔮 Consulta de Calidad del Aire"):
                        gr.Markdown("Ingresa valores ambientales para obtener una estimación.")
                        sesion_info = gr.Markdown("_⚠️ Entrena primero un modelo._")
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [gr.Number(label=f"feature_{i}", value=0, visible=False, interactive=True) for i in range(30)]
                        json_vals = gr.Textbox(visible=False, value="{}")
                        btn_predecir = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""<div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">Surrogate Model v10.2 · Feature Engineering · XGBoost · Auto-GPU</div>""")

        def _refresh_pred_ui():
            feat_cols = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia = SESION.get("parroquia", "")
            target = SESION.get("target", "")
            sesion_msg = f"✅ Modelo en sesión: **{target}** · *{parroquia}* · {len(feat_cols)} features." if feat_cols else "_⚠️ Entrena primero un modelo._"
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st = feat_stats.get(col, {"min":0,"max":100,"mean":50})
                    updates.append(gr.Number(label=col, value=round(st["mean"],3), visible=True, interactive=True))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i]) for i in range(min(len(feat_cols), len(vals))) if vals[i] is not None}
            return json.dumps(d)

        def _detectar_wrapper(csv_local, csv_up):
            ruta = (csv_up if isinstance(csv_up, str) else (csv_up.name if csv_up else None)) \
                   or (csv_local if csv_local and not csv_local.startswith("(No") else None)
            if not ruta: return gr.Dropdown(choices=["PM25"], value="PM25"), "⚠️ Selecciona un archivo primero."
            cols, msg = obtener_columnas_csv(ruta)
            choices = cols if cols else ["PM25"]
            return gr.Dropdown(choices=choices, value=choices[0] if choices else "PM25"), msg

        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])
        btn_detectar.click(fn=_detectar_wrapper, inputs=[csv_dropdown, csv_upload], outputs=[target_input, info_cols])
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, k_splits_slider,
                max_depth_slider, learning_rate_slider, subsample_slider, colsample_bytree_slider,
                reg_lambda_slider, reg_alpha_slider, gamma_slider, min_child_weight_slider,
                usar_fe_ck,
            ],
            outputs=[metricas_output, fig_pred_output, fig_fi_output, fig_hm_output, fig_lc_output, estado_output],
        ).then(fn=_refresh_pred_ui, inputs=[], outputs=[sesion_info] + slider_componentes)
        btn_predecir.click(fn=_construir_json, inputs=slider_componentes, outputs=json_vals).then(
            fn=predecir_desde_sesion, inputs=[json_vals], outputs=[resultado_pred]
        )

    return app

# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────
def _imprimir_ip_fallback():
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception: pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception: pass
    try:
        ip_pub = subprocess.check_output(["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"], text=True, timeout=7).strip()
        if ip_pub: print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception: pass

if __name__ == "__main__":
    app = construir_app()
    print("\n" + "═" * 64)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v10.2")
    print("  🧪  Feature Engineering Avanzado + XGBoost Dinámico")
    print("═" * 64)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 64)
    try:
        app.launch(server_name="0.0.0.0", server_port=None, share=True, max_threads=40, debug=True, show_error=True, prevent_thread_lock=False, quiet=False)
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}\n🔄  Reintentando en puerto 7861…\n")
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40, debug=True, show_error=True, prevent_thread_lock=False, quiet=False)
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}\n🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40, debug=True, show_error=True, prevent_thread_lock=False, quiet=False)

16:41:34 [INFO] Archivo subido: dataset_ml_carapungo.csv
16:41:34 [INFO] CSV cargado: 160,262 filas × 19 columnas
16:41:34 [INFO] Columna de tiempo: 'Timestamp'
16:41:35 [INFO] Pandemia excluida: años [2020, 2021] eliminados.
16:41:35 [INFO] Aplicando Feature Engineering avanzado...
16:41:35 [INFO] Seleccionando las 40 mejores características (entre 97)...
16:41:39 [INFO] Características seleccionadas: 40
16:41:39 [INFO] Total de columnas en df_clean para K-Fold: 41 (features: 40)
16:41:39 [INFO] Hiperparámetros XGBoost: max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, reg_alpha=0.5, gamma=0.0, min_child_weight=1.0
16:41:39 [INFO] Iniciando TimeSeriesSplit (K=5) · Algoritmo: HistGradientBoosting (CPU — sin GPU requerida)
16:41:39 [INFO] GPU disponible: True → GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
16:41:39 [ERROR] The 'min_samples_leaf' parameter of HistGradientBoostingRegressor must be an int in the range [1, inf). Got 1.0 inst

XGBOOST con  Feature Engineering echo por claude

In [1]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 10.3
  ─── Correcciones aplicadas ─────────────────────────────────────────────────
  [FIX-1]  feature_importances_ ausente en HistGB algunas versiones de sklearn
           → Reemplazado por RandomForestRegressor como estimador base para
             SelectFromModel (siempre tiene feature_importances_). A prueba
             de fallos: si falla, se usan todas las features disponibles.
  [FIX-2]  Checkboxes no clickeables por conflicto de CSS
           → CSS reescrito. Los checkboxes usan appearance:checkbox nativo,
             sin reglas que oculten el input subyacente.
  [FIX-3]  Eliminado el explorador de archivos locales (Dropdown + botón
           Detectar + botón Refrescar). Solo queda gr.File para subir CSV.
  [FIX-4]  Todos los parámetros de la UI conectados correctamente a entrenar().
  ─── Arquitectura ──────────────────────────────────────────────────────────
  · TimeSeriesSplit(K ajustable) sobre el 80 % de entrenamiento.
  · Test ciego = último 20 %, nunca expuesto al modelo.
  · Sub-split interno 80/20 del bloque train para early stopping de XGB/LGB.
  · Algoritmos: HistGradientBoosting, XGBoost, LightGBM (auto GPU/CPU).
  · Feature Engineering opcional: rolling stats, diffs, interacciones,
    variables cíclicas de día de semana. Selección top-40 con RandomForest.
================================================================================
  Dependencias:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
      pip install xgboost lightgbm   # opcionales
      pip install torch              # opcional, solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"

MAX_FEATURES_SEL = 40   # top-N features en Feature Engineering


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE else {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  CARGA Y DETECCIÓN DE TIMESTAMP
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(df: pd.DataFrame, target_col: str, timestamp_col: str,
                 excluir_pandemia: bool = True) -> pd.DataFrame:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=[target_col])
    return df


def _dividir_cronologico(X, y, ratio: float = 0.80):
    """División temporal estricta — el (1-ratio) % final es test ciego."""
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    """Columnas numéricas que no sean contaminantes ni el target."""
    cols = set(df.select_dtypes(include=[np.number]).columns)
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols if c not in excluir and c != target_col]


# ──────────────────────────────────────────────────────────────────────────────
# 5.  FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────

def _agregar_features_avanzadas(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """
    Genera rolling stats, diferencias, interacciones y variables cíclicas
    de día de semana. Lógica no modificada respecto a v10.2.
    """
    df = df.copy()
    contaminantes = [c for c in CONTAMINANTES_Y if c in df.columns]

    for col in contaminantes:
        for window in [3, 6, 12, 24]:
            df[f"{col}_roll_mean_{window}h"] = df[col].rolling(window, min_periods=1).mean()
            if window >= 6:
                df[f"{col}_roll_std_{window}h"] = df[col].rolling(window, min_periods=2).std()
        for lag in [1, 3, 6, 12, 24]:
            df[f"{col}_diff_{lag}h"] = df[col].diff(lag)

    if "Temperatura" in df.columns and "Humedad" in df.columns:
        df["temp_hum"] = df["Temperatura"] * df["Humedad"]
    if "Temperatura" in df.columns and "Viento_Velocidad" in df.columns:
        df["temp_wind"] = df["Temperatura"] * df["Viento_Velocidad"]
    if "Viento_Direccion" in df.columns and "Viento_Velocidad" in df.columns:
        dir_rad = np.radians(df["Viento_Direccion"])
        df["wind_u"] = df["Viento_Velocidad"] * np.cos(dir_rad)
        df["wind_v"] = df["Viento_Velocidad"] * np.sin(dir_rad)

    if hasattr(df.index, "weekday"):
        df["dia_semana"]     = df.index.weekday
        df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
        df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
        df["es_finde"]       = (df["dia_semana"] >= 5).astype(int)

    return df.dropna()


def _seleccionar_top_features(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    n_features: int = MAX_FEATURES_SEL,
    log_fn=None,
) -> list[str]:
    """
    [FIX-1] Selección robusta de las N mejores características.

    Usa RandomForestRegressor (siempre tiene feature_importances_) como
    estimador base de SelectFromModel. Esto evita el fallo de
    HistGradientBoostingRegressor.feature_importances_ que no existe en
    algunas versiones de scikit-learn.

    Fallback: si la selección falla por cualquier razón, devuelve todas
    las columnas disponibles (comportamiento seguro).
    """
    if log_fn is None:
        log_fn = log.info

    all_cols = X_tr.columns.tolist()
    if len(all_cols) <= n_features:
        log_fn(f"  Todas las {len(all_cols)} features disponibles (≤ {n_features}) — sin selección.")
        return all_cols

    try:
        # Usar solo filas sin NaN para el estimador de selección
        mask_ok = X_tr.notna().all(axis=1) & y_tr.notna()
        X_ok = X_tr[mask_ok].values
        y_ok = y_tr[mask_ok].values

        if len(X_ok) < 50:
            log_fn("  ⚠️  Datos insuficientes para selección — usando todas las features.")
            return all_cols

        # RandomForest con n_estimators reducido para rapidez
        rf_sel = RandomForestRegressor(
            n_estimators=80,
            max_depth=6,
            min_samples_leaf=20,
            n_jobs=-1,
            random_state=42,
        )
        rf_sel.fit(X_ok, y_ok)

        # SelectFromModel con threshold=-np.inf fuerza selección por top-N
        selector = SelectFromModel(
            rf_sel,
            threshold=-np.inf,
            max_features=n_features,
            prefit=True,
        )
        selected_mask = selector.get_support()
        selected = [col for col, keep in zip(all_cols, selected_mask) if keep]

        log_fn(f"  SelectFromModel (RandomForest): {len(all_cols)} → {len(selected)} features.")
        return selected if selected else all_cols

    except Exception as exc:
        log_fn(f"  ⚠️  Selección de features falló ({exc}) — usando todas las features.")
        return all_cols


# ──────────────────────────────────────────────────────────────────────────────
# 6.  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str, params: dict | None = None):
    if params is None:
        params = {}

    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            gamma=float(params.get("gamma", 0.0)),
            min_child_weight=float(params.get("min_child_weight", 1)),
            early_stopping_rounds=50,
            eval_metric="rmse",
            random_state=42,
            verbosity=0,
            **_xgb_tree_method(),
        )

    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            min_child_samples=int(params.get("min_child_weight", 30)),
            metric="rmse",
            device=_lgbm_device(),
            random_state=42,
            verbose=-1,
        )

    else:  # HistGradientBoosting (default)
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=int(params.get("max_depth", 5)),
            min_samples_leaf=int(params.get("min_child_weight", 30)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            l2_regularization=float(params.get("reg_lambda", 1.0)),
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 7.  PIPELINE TimeSeriesSplit
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist  = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name


def _entrenar_kfold(
    df: pd.DataFrame,
    algoritmo: str,
    n_splits: int = 5,
    log_fn=None,
    xgb_params: dict | None = None,
) -> tuple[pd.DataFrame, dict]:
    if log_fn is None:
        log_fn = log.info

    tscv       = TimeSeriesSplit(n_splits=n_splits)
    filas      = []
    cols_df    = set(df.columns)
    all_curves = {}

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible — omitido.")
            continue

        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]
        mask_ok   = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]

        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue

        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics   = []
        fold_curves_tr = []
        fold_curves_va = []
        metric_name_cv = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            scaler  = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_va_sc = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo, xgb_params)

            if "XGBoost" in algoritmo:
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[
                        lgb.early_stopping(30, verbose=False),
                        lgb.log_evaluation(-1),
                    ],
                )
            else:
                modelo.fit(X_tr_sc, y_tr)

            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h:
                fold_curves_tr.append(tr_h)
            if va_h:
                fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(
                f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                f"MAE={fold_metrics[-1]['MAE']:.3f}  "
                f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
                f"R²={fold_metrics[-1]['R2']:.3f}"
            )

        all_curves[contaminante] = {
            "train": fold_curves_tr,
            "val":   fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":   len(feat_cols),
            "MAE_mean":     mf["MAE"].mean(),
            "MAE_std":      mf["MAE"].std(),
            "RMSE_mean":    mf["RMSE"].mean(),
            "RMSE_std":     mf["RMSE"].std(),
            "R2_mean":      mf["R2"].mean(),
            "R2_std":       mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 8.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo: str, xlabel: str, ylabel: str) -> None:
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_prediccion(y_test, y_pred, target_col: str, days: int, parroquia: str = ""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]

    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col: str, parroquia: str = ""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(
        fi_df["Feature"][::-1], fi_df["Importance"][::-1],
        xerr=fi_df["Std"][::-1], color=colores[::-1],
        align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65,
    )
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col: str, parroquia: str = ""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(
        corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
        annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
        linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
        cbar_kws={"shrink": 0.7},
    )
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


def _fig_curva_aprendizaje(
    curvas: dict, target_col: str, algoritmo: str, parroquia: str = "", k_splits: int = 5
):
    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5, "No hay datos de curva de aprendizaje.",
                ha="center", va="center", color=TEXT_PLOT, fontsize=11, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = metric_name == "R²"

    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len  = min(lengths)
    x        = np.arange(min_len)
    tr_arr   = np.array([c[:min_len] for c in train_curves])
    tr_mean  = tr_arr.mean(axis=0)
    tr_std   = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_REAL, alpha=0.10, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_REAL, lw=2.2,
            label=f"Train — {metric_name}  (μ TSS-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_REAL, zorder=3)

    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)
        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_PRED, alpha=0.10, lw=0.75, zorder=2)
        ax.plot(x, va_mean, color=COLOR_PRED, lw=2.2, linestyle="--",
                label=f"Validación — {metric_name}  (μ TSS-Fold)", zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_PRED, zorder=3)

        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":", alpha=0.88,
                   label=f"Mejor iteración: {best_iter}  ({metric_name}={best_val:.4f})",
                   zorder=6)
        offset = max(1, int(min_len * 0.02))
        ax.annotate(f" iter {best_iter}", xy=(best_iter, best_val),
                    xytext=(best_iter + offset, best_val),
                    color="#F59E0B", fontsize=8.5, va="center", zorder=7)

    ax.axhline(tr_mean[-1], color=COLOR_REAL, lw=0.8, linestyle=":", alpha=0.40, zorder=1)

    algo_label = algoritmo.split("(")[0].strip()
    titulo = (
        f"Curva de Aprendizaje — {target_col}  ·  "
        f"{algo_label}  ·  TimeSeriesSplit (K={k_splits})"
    )
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    nota   = (
        "Nota HGB: Validación = fracción interna 10% de early stopping."
        if is_r2 else
        "Nota: banda sombreada = ±1σ entre pliegues temporales (TimeSeriesSplit)."
    )

    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9, loc="best")
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 9.  PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_upload,           # gr.File — único origen de datos [FIX-3]
    target_col: str,
    nombre_modelo: str,
    algoritmo: str,
    plot_days: int,
    train_ratio: float,
    excluir_pandemia: bool,
    k_splits: int,
    max_depth: float,
    learning_rate: float,
    subsample: float,
    colsample_bytree: float,
    reg_lambda: float,
    reg_alpha: float,
    gamma: float,
    min_child_weight: float,
    usar_feature_engineering: bool,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])

    VACIO = (None, None, None, None)

    try:
        # ── 9.1  Resolver archivo ─────────────────────────────────────────────
        if csv_upload is None:
            err("No se subió ningún archivo CSV.")
            return "### ❌ Sube un archivo CSV antes de entrenar.", *VACIO, _estado()

        ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()
        info(f"Archivo: {Path(ruta_csv).name}")

        # ── 9.2  Carga ────────────────────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        # ── 9.3  Preprocesamiento ─────────────────────────────────────────────
        df_prep = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")

        y = df_prep[target_col]
        X = df_prep.drop(columns=[target_col])

        # ── 9.4  Feature Engineering + selección robusta ──────────────────────
        if usar_feature_engineering:
            info("Aplicando Feature Engineering avanzado…")
            df_full = _agregar_features_avanzadas(pd.concat([X, y], axis=1), target_col)
            y = df_full[target_col]
            X = df_full.drop(columns=[target_col])

            # Usar el bloque train para seleccionar features (sin ver el test)
            X_tr_sel, _, y_tr_sel, _ = _dividir_cronologico(X, y, train_ratio)
            info(f"Seleccionando top-{MAX_FEATURES_SEL} features entre {X_tr_sel.shape[1]}…")

            selected = _seleccionar_top_features(X_tr_sel, y_tr_sel, MAX_FEATURES_SEL, log_fn=info)
            X = X[selected]
            feat_cols = list(X.columns)
            info(f"Features seleccionadas: {len(feat_cols)}")
        else:
            num_cols  = X.select_dtypes(include=[np.number]).columns.tolist()
            feat_cols = [c for c in num_cols if c != target_col]
            X         = X[feat_cols]
            info(f"Usando {len(feat_cols)} features originales.")

        # ── 9.5  Limpiar NaN ──────────────────────────────────────────────────
        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — se eliminan filas afectadas.")
        mask = X.notna().all(axis=1)
        X    = X.loc[mask]
        y    = y.loc[mask]

        df_clean = pd.concat([X, y], axis=1)
        info(f"Dataset limpio: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")

        # ── 9.6  Hiperparámetros ──────────────────────────────────────────────
        xgb_params = {
            "max_depth":       int(max_depth),
            "learning_rate":   float(learning_rate),
            "subsample":       float(subsample),
            "colsample_bytree": float(colsample_bytree),
            "reg_lambda":      float(reg_lambda),
            "reg_alpha":       float(reg_alpha),
            "gamma":           float(gamma),
            "min_child_weight": float(min_child_weight),
        }
        info("Hiperparámetros: " + ", ".join(f"{k}={v}" for k, v in xgb_params.items()))

        # ── 9.7  TimeSeriesSplit K-Fold ───────────────────────────────────────
        info(f"TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} — {GPU_MSG}")

        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits,
            log_fn=kf_log, xgb_params=xgb_params,
        )
        for l in kfold_logs:
            logs.append(l)

        if kf_resumen.empty:
            warn("TSS K-Fold no produjo resultados.")

        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits
        )

        # ── 9.8  Modelo final con sub-split interno ───────────────────────────
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        sub_corte  = int(len(X_tr) * 0.80)
        X_sub_tr   = X_tr.iloc[:sub_corte]
        X_val_sub  = X_tr.iloc[sub_corte:]
        y_sub_tr   = y_tr.iloc[:sub_corte]
        y_val_sub  = y_tr.iloc[sub_corte:]

        scaler_final  = StandardScaler()
        X_sub_tr_sc   = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc  = scaler_final.transform(X_val_sub)
        X_te_sc       = scaler_final.transform(X_te)
        X_tr_sc       = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np       = y_tr.values

        modelo_final = _construir_modelo(algoritmo, xgb_params)

        if "XGBoost" in algoritmo:
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                verbose=False,
            )
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                callbacks=[
                    lgb.early_stopping(30, verbose=False),
                    lgb.log_evaluation(-1),
                ],
            )
        else:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)

        # ── 9.9  Métricas ─────────────────────────────────────────────────────
        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )

        m_tr = _m(y_tr_np, y_pred_tr)
        m_te = _m(y_te.values, y_pred_te)
        gap  = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting: ΔR² = {gap:.3f}")

        # ── 9.10  Markdown métricas ────────────────────────────────────────────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                ico = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** "
                    f"| `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {ico} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        metricas_md = f"""
## 📊 Surrogate Model v10.3 — *{parroquia}*

### Validación Cronológica TimeSeriesSplit (K={k_splits}) — 6 Contaminantes

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> `TimeSeriesSplit` garantiza que cada fold de validación sea **posterior** al bloque de entrenamiento.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

> **Test set 100 % ciego** — el {int((1-train_ratio)*100)} % final nunca fue expuesto al modelo.

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión (R² %)** | `{m_tr['R2']*100:.2f}%` | `{m_te['R2']*100:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features | `{len(feat_cols)}` columnas |
| Sub-train / Val interno / Test ciego | `{len(X_sub_tr):,}` / `{len(X_val_sub):,}` / `{len(X_te):,}` |
| Pandemia | {pandemia_label} |
| Feature Engineering | {'✅ Activado (top-' + str(MAX_FEATURES_SEL) + ')' if usar_feature_engineering else '❌ Desactivado'} |
| Regularización | L2={xgb_params['reg_lambda']}  α={xgb_params['reg_alpha']}  depth={xgb_params['max_depth']} |
"""

        # ── 9.11  Feature Importance ──────────────────────────────────────────
        info("Calculando Permutation Importance en test ciego…")
        perm  = permutation_importance(
            modelo_final, X_te_sc, y_te.values,
            n_repeats=8, random_state=42, scoring="r2",
        )
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )

        # ── 9.12  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo":       modelo_final,
                "scaler":       scaler_final,
                "features":     feat_cols,
                "target":       target_col,
                "parroquia":    parroquia,
                "kfold_resumen": kf_resumen,
                "kf_curvas":    kf_curvas,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 9.13  Actualizar sesión ───────────────────────────────────────────
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": {
                col: {
                    "min":  float(X[col].min()),
                    "max":  float(X[col].max()),
                    "mean": float(X[col].mean()),
                }
                for col in feat_cols
            },
        })
        info("Sesión actualizada → pestaña Predicción lista.")

        # ── 9.14  Figuras ─────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm      = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Figuras generadas.")

        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\nInstala con: `pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado()
        )


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PREDICCIÓN DESDE SESIÓN
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return "### ⚠️ No hay modelo entrenado.\nEntrena primero un modelo."
    try:
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]

        extra = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    cal = "🟢 **Buena**"
            elif pred <= 35:  cal = "🟡 **Moderada**"
            elif pred <= 55:  cal = "🟠 **Insalubre GS**"
            elif pred <= 150: cal = "🔴 **Insalubre**"
            else:             cal = "🟣 **Muy insalubre**"
            extra = f"\n\n**Índice AQI:** {cal}"

        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS  [FIX-2]  Checkboxes funcionales + tema oscuro
# ──────────────────────────────────────────────────────────────────────────────
#
# Regla clave para los checkboxes:
#   · NO se oculta el <input type="checkbox"> nativo (no hay visibility:hidden
#     ni display:none sobre él).
#   · Se fuerza -webkit-appearance/appearance: checkbox para que el navegador
#     lo dibuje visualmente.
#   · El contenedor label usa display:flex para alinear el tick y el texto.
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
/* ── Variables del tema ── */
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}

/* ── Layout general ── */
body, .gradio-container {
    background: var(--bg) !important;
    color: var(--text) !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}
.gr-group, .gr-box {
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--r) !important;
    padding: 16px !important;
}

/* ── Inputs de texto / select ── */
input[type="text"],
input[type="number"],
input[type="email"],
textarea,
select {
    background: var(--input) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 6px !important;
}

/* ── Labels ── */
label, .gr-label {
    color: var(--muted) !important;
    font-size: .8rem !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: .05em !important;
}

/* ── [FIX-2] Checkboxes: apariencia nativa, sin ocultar el input ── */
input[type="checkbox"] {
    -webkit-appearance: checkbox !important;
    appearance:         checkbox !important;
    width:     18px !important;
    height:    18px !important;
    min-width: 18px !important;
    margin:    0 8px 0 0 !important;
    cursor:    pointer !important;
    vertical-align: middle !important;
    /* Acento azul-índigo cuando está marcado */
    accent-color: var(--accent);
}
/* El label del checkbox debe ser flex para que el tick y el texto se alineen */
.gr-checkbox > label,
.gr-checkbox label,
[data-testid="checkbox"] label {
    display:     flex !important;
    align-items: center !important;
    color:       var(--text) !important;
    font-size:   .9rem !important;
    font-weight: 500 !important;
    text-transform: none !important;
    cursor: pointer !important;
    gap: 4px;
}

/* ── Botones ── */
button.primary {
    background: linear-gradient(135deg, var(--accent), #7C3AED) !important;
    color: #fff !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 800 !important;
    font-size: 1rem !important;
    padding: 12px 28px !important;
    box-shadow: 0 4px 20px rgba(99,102,241,.45);
    transition: all .15s;
}
button.primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 28px rgba(99,102,241,.6);
}
button.secondary {
    background: var(--card) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
}

/* ── Markdown / tablas ── */
.gr-markdown { color: var(--text) !important; }
.gr-markdown table { border-collapse: collapse; width: 100%; }
.gr-markdown th {
    background: #1E293B;
    color: var(--accentH);
    padding: 8px 14px;
    border: 1px solid var(--border);
}
.gr-markdown td {
    color: var(--text);
    padding: 7px 14px;
    border: 1px solid var(--border);
}
.gr-markdown tr:nth-child(even) td { background: #19253a; }

/* ── Área de carga de archivo ── */
.gr-file {
    border: 2px dashed var(--accent) !important;
    border-radius: var(--r) !important;
    background: rgba(99,102,241,.04) !important;
}

/* ── Hero ── */
.hero { text-align: center; padding: 24px 0 6px; }
.hero h1 {
    font-size: 2rem; font-weight: 900;
    background: linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.hero p { color: var(--muted); font-size: .88rem; }

/* ── Badge GPU ── */
.gpu-badge {
    display: inline-block;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: .75rem;
    font-weight: 700;
    margin-top: 4px;
}
.gpu-on  { background: rgba(34,197,94,.18);  color: #4ADE80; border: 1px solid #22C55E; }
.gpu-off { background: rgba(99,102,241,.15); color: #A5B4FC; border: 1px solid #6366F1; }

/* ── Caja de logs ── */
.logs-box {
    background: #0B1527 !important;
    border: 1px solid #1D3557 !important;
    border-radius: 8px;
    padding: 10px 14px;
    font-family: 'JetBrains Mono', monospace;
    font-size: .76rem;
    color: #7DD3FC;
    max-height: 140px;
    overflow-y: auto;
    line-height: 1.6;
}

/* ── Sección Feature Engineering ── */
.fe-badge {
    background: rgba(99,102,241,.10);
    border: 1px solid #4338CA;
    border-radius: 8px;
    padding: 8px 14px;
    font-size: .78rem;
    color: #C7D2FE;
    margin-top: 4px;
}
"""


# ──────────────────────────────────────────────────────────────────────────────
# 12.  UI — rediseño limpio, sin explorador de archivos locales [FIX-3]
# ──────────────────────────────────────────────────────────────────────────────

MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v10.3",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Feature Engineering · XGBoost · LightGBM · HistGB · Blind Test · v10.3</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ── PANEL IZQUIERDO — CONTROLES ────────────────────────────────────
            with gr.Column(scale=1, min_width=360):

                # ── Dataset: solo File upload [FIX-3] ────────────────────────
                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    csv_upload = gr.File(
                        label="Arrastra o sube tu archivo CSV",
                        file_types=[".csv"],
                        type="filepath",
                    )
                    gr.Markdown(
                        "_El nombre del archivo se usará como identificador de parroquia._",
                        elem_classes=[],
                    )

                gr.HTML("<div style='height:8px'/>")

                # ── Configuración del modelo ──────────────────────────────────
                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")

                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25",
                            allow_custom_value=True,
                            scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire",
                            scale=3,
                        )

                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS,
                        value=MODELOS[0],   # HistGB por defecto (sin dependencias extras)
                    )

                    # Checkboxes [FIX-2] — visibles y funcionales
                    with gr.Row():
                        excluir_pandemia_chk = gr.Checkbox(
                            label="🚫 Excluir pandemia (2020–2021)",
                            value=True,
                            scale=1,
                        )
                        usar_fe_ck = gr.Checkbox(
                            label="🧪 Feature Engineering",
                            value=False,
                            scale=1,
                        )

                    gr.HTML(
                        '<div class="fe-badge">'
                        '<b>Feature Engineering</b>: rolling stats, diferencias, '
                        'interacciones meteo, día de semana → top-40 con RandomForest.'
                        '</div>'
                    )

                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Test split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        k_splits_slider = gr.Slider(
                            label="K — pliegues TSCV",
                            minimum=2, maximum=15, step=1, value=5, scale=2,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:8px'/>")

                # ── Hiperparámetros XGBoost ───────────────────────────────────
                with gr.Accordion("🧪 Hiperparámetros XGBoost / LightGBM / HistGB", open=False):
                    gr.Markdown(
                        "_Estos controles aplican a XGBoost y LightGBM. "
                        "Para HistGB se usan `learning_rate`, `max_depth` y `reg_lambda`._"
                    )
                    with gr.Row():
                        max_depth_slider       = gr.Slider(label="max_depth",        minimum=2,   maximum=10,  step=1,    value=5)
                        learning_rate_slider   = gr.Slider(label="learning_rate",    minimum=0.01, maximum=0.3, step=0.01, value=0.05)
                    with gr.Row():
                        subsample_slider       = gr.Slider(label="subsample",        minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                        colsample_bytree_slider = gr.Slider(label="colsample_bytree", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                    with gr.Row():
                        reg_lambda_slider      = gr.Slider(label="reg_lambda (L2)",  minimum=0.0, maximum=10.0, step=0.5,  value=2.0)
                        reg_alpha_slider       = gr.Slider(label="reg_alpha (L1)",   minimum=0.0, maximum=5.0,  step=0.1,  value=0.5)
                    with gr.Row():
                        gamma_slider           = gr.Slider(label="gamma",            minimum=0.0, maximum=5.0,  step=0.1,  value=0.0)
                        min_child_weight_slider = gr.Slider(label="min_child_weight", minimum=1,   maximum=20,   step=1,    value=1)

                gr.HTML("<div style='height:10px'/>")

                # ── Botón entrenar + logs ─────────────────────────────────────
                btn_train = gr.Button(
                    "🚀  Iniciar Entrenamiento", variant="primary", size="lg"
                )
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ── PANEL DERECHO — RESULTADOS ─────────────────────────────────────
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test ciego"
                        )

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso por época / boosting round** con TimeSeriesSplit.\n\n"
                            "Línea continua = Train · Línea discontinua = Validación. "
                            "Banda = ±1σ entre pliegues."
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Error vs Iteración (TSS)"
                        )

                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown(
                            "**Importancia por permutación** calculada sobre el test ciego."
                        )
                        fig_fi_output = gr.Plot(
                            label="Permutation Importance (Δ R²)"
                        )

                    with gr.Tab("📊 Correlaciones"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La columna del target está resaltada en naranja."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    with gr.Tab("🔮 Predicción Instantánea"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "con el **modelo en memoria** (última sesión entrenada)."
                        )
                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(
                                    label=f"feature_{i}", value=0,
                                    visible=False, interactive=True
                                )
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v10.3 · FIX: Feature Selection · Checkboxes · Upload-only · Auto-GPU
        </div>
        """)

        # ── Funciones auxiliares de la UI ─────────────────────────────────────

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")
            sesion_msg = (
                f"✅ **{target}** · *{parroquia}* · {len(feat_cols)} features."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col,
                        value=round(st["mean"], 3),
                        visible=True,
                        interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {
                feat_cols[i]: float(vals[i])
                for i in range(min(len(feat_cols), len(vals)))
                if vals[i] is not None
            }
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────

        # [FIX-4] Todos los parámetros de la UI están conectados correctamente
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_upload,                 # [FIX-3] único origen de datos
                target_input,
                nombre_modelo_input,
                algoritmo_radio,
                plot_days_slider,
                train_ratio_slider,
                excluir_pandemia_chk,       # [FIX-2] checkbox funcional
                k_splits_slider,
                max_depth_slider,
                learning_rate_slider,
                subsample_slider,
                colsample_bytree_slider,
                reg_lambda_slider,
                reg_alpha_slider,
                gamma_slider,
                min_child_weight_slider,
                usar_fe_ck,                 # [FIX-2] checkbox funcional
            ],
            outputs=[
                metricas_output,
                fig_pred_output,
                fig_fi_output,
                fig_hm_output,
                fig_lc_output,
                estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 66)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v10.3")
    print("  🔧  FIX: Feature Selection · Checkboxes · Upload-only")
    print("═" * 66)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print(f"  FE top-N  : {MAX_FEATURES_SEL} features (RandomForest selector)")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 66)

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}\n🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  Túnel público falló: {tunnel_err}\n🔄  Relanzando sin túnel:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

01:43:47 [INFO] GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB



══════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v10.3
  🔧  FIX: Feature Selection · Checkboxes · Upload-only
══════════════════════════════════════════════════════════════════
  Hardware  : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  GPU activa: True
  FE top-N  : 40 features (RandomForest selector)
  Puerto    : automático (server_port=None)
══════════════════════════════════════════════════════════════════


01:47:11 [INFO] Archivo: dataset_ml_belisario.csv
01:47:12 [INFO] CSV cargado: 165,157 filas × 18 columnas
01:47:12 [INFO] Timestamp: 'Timestamp'
01:47:12 [INFO] Pandemia excluida: años [2020, 2021] eliminados.
01:47:12 [INFO] Aplicando Feature Engineering avanzado…
01:47:12 [INFO] Seleccionando top-40 features entre 84…
01:47:14 [INFO]   SelectFromModel (RandomForest): 84 → 40 features.
01:47:14 [INFO] Features seleccionadas: 40
01:47:15 [INFO] Dataset limpio: 35,924 filas × 41 columnas
01:47:15 [INFO] Hiperparámetros: max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, reg_alpha=0.5, gamma=0.0, min_child_weight=1.0
01:47:15 [INFO] TimeSeriesSplit (K=5) · Algoritmo: XGBoost  (auto GPU/CPU)
01:47:15 [INFO] GPU disponible: True — GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
01:47:19 [INFO]   PM25 | Fold 1/5 → MAE=0.480  RMSE=0.727  R²=0.994
01:47:22 [INFO]   PM25 | Fold 2/5 → MAE=0.431  RMSE=0.760  R²=0.993
01:47:26 [INFO]   PM25 | Fold 3

Nueva version del codigo de claude pero ahora jusntando los 6 modelos de los contaminates

In [ ]:


# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Estado de sesión (compartido entre callbacks de Gradio)
SESION: dict = {
    "modelo":      None,   # MultiOutputRegressor entrenado
    "scaler":      None,   # StandardScaler ajustado sobre X_train
    "feat_cols":   [],     # nombres de columnas X
    "target_cols": [],     # nombres de columnas Y (subconjunto de COLUMNAS_Y_ESPERADAS)
    "parroquia":   "",
    "feat_stats":  {},     # {col: {min, max, mean}} para la UI de consulta
}

ANOS_PANDEMIA = [2020, 2021]

# ── [SDM] Esquema fijo de salidas ──────────────────────────────────────────────
# El modelo multi-output predice EXACTAMENTE estos contaminantes si están en el CSV.
COLUMNAS_Y_ESPERADAS: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

# Columnas de control que NUNCA pueden ser features X
COLUMNAS_CONTROL: frozenset[str] = frozenset({
    "Fecha", "fecha", "FECHA",
    "Parroquia", "parroquia", "PARROQUIA",
    "Timestamp", "timestamp", "TIMESTAMP",
    "Date", "date", "DATE",
    "Hora", "hora", "HORA",
    "Estacion", "estacion", "ESTACION",
    "Codigo", "codigo", "CODIGO",
})

# Features X con prioridad explícita (orden respetado en el modelo)
FEATURES_X_PREFERIDAS: list[str] = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

# ── HGB — Hiperparámetros heredados de v10 ─────────────────────────────────────
HGB_L2_REG           = 5.0   # λ — penalización L2 explícita (reduce peso de lags 1h)
HGB_MIN_SAMPLES_LEAF = 50    # α-proxy — sparsity implícita tipo L1
HGB_MAX_LEAF_NODES   = 31    # complejidad geométrica máxima
HGB_LEARNING_RATE    = 0.05

# ── Paralelismo MultiOutputRegressor ─────────────────────────────────────────
# min(6, cpu//2) evita sobresubscripción con los threads OpenMP internos de HGB.
# Justificación: MultiOutputRegressor usa joblib/loky (subprocesos separados);
# cada subproceso lanza HGB que internamente usa OpenMP threads.
# Con n_jobs=cpu_count, tendríamos cpu_count×cpu_count threads activos.
N_JOBS_MULTI: int = min(len(COLUMNAS_Y_ESPERADAS), max(1, (os.cpu_count() or 2) // 2))

BATCH_CURVA = 20  # árboles por paso de warm_start (curva de aprendizaje)

# ── Paleta de colores — uno por target ─────────────────────────────────────────
TARGET_COLORS: dict[str, str] = {
    "PM25": "#3B82F6",  # azul
    "PM10": "#F97316",  # naranja
    "O3":   "#22C55E",  # verde
    "CO":   "#A855F7",  # púrpura
    "NO2":  "#EF4444",  # rojo
    "SO2":  "#EAB308",  # amarillo
}
COLOR_GLOBAL = "#F1F5F9"   # curva promedio global
COLOR_REAL   = "#3B82F6"
COLOR_PRED   = "#F97316"
COLOR_POS    = "#22C55E"
COLOR_NEG    = "#EF4444"
BG_PLOT      = "#0F172A"
TEXT_PLOT    = "#E2E8F0"
GRID_PLOT    = "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN AUTOMÁTICA DE GPU  [FIX-4]
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU (HistGB funciona nativamente)"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


# ──────────────────────────────────────────────────────────────────────────────
# 3.  EXPLORADOR DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


# ──────────────────────────────────────────────────────────────────────────────
# 4.  MÓDULO DE DETECCIÓN DINÁMICA DE ESQUEMA  [SDM]
# ──────────────────────────────────────────────────────────────────────────────

def _detectar_esquema(
    df:               pd.DataFrame,
    ts_col:           str | None,
    requerir_todas_y: bool = False,
) -> tuple[list[str], list[str], list[str]]:
    """
    [SDM] Separación automática X/Y con validación de seguridad.

    Reglas
    ──────
    [SDM-1]  Y = columnas presentes que coincidan EXACTAMENTE con COLUMNAS_Y_ESPERADAS.
             · Nota: 'PM25_lag_1h' ≠ 'PM25' → no contamina Y.
             · Si requerir_todas_y=True y faltan columnas, lanza ValueError.
    [SDM-2]  X = columnas numéricas no-Y, no-control, no-timestamp.
             · Prioridad 1: FEATURES_X_PREFERIDAS presentes en el CSV.
             · Prioridad 2: resto de numéricas no excluidas (orden del DataFrame).
    [SDM-3]  Regla de seguridad: RuntimeError si alguna columna en Y aparece en X.
             · Previene data leakage por error de preprocesamiento anterior.

    Parameters
    ──────────
    df               : DataFrame completo (post-carga, antes de set_index)
    ts_col           : columna timestamp detectada (se excluye de X)
    requerir_todas_y : si True, exige las 6 columnas Y (modo estricto)

    Returns
    ───────
    x_cols           : features de entrada, ordenadas por prioridad
    y_cols_presentes : targets detectados (subconjunto de COLUMNAS_Y_ESPERADAS)
    y_cols_faltantes : targets ausentes del CSV (vacío si están todas)

    Raises
    ──────
    ValueError   → ninguna Y encontrada, o modo estricto con Y faltantes
    RuntimeError → [SDM-3] violación de seguridad Y-en-X
    """
    cols_disponibles = set(df.columns)

    # ── [SDM-1] Identificar columnas Y ────────────────────────────────────────
    y_cols_presentes = [c for c in COLUMNAS_Y_ESPERADAS if c in cols_disponibles]
    y_cols_faltantes = [c for c in COLUMNAS_Y_ESPERADAS if c not in cols_disponibles]

    if not y_cols_presentes:
        raise ValueError(
            "[SDM-1] No se encontró ninguna columna Y en el dataset.\n"
            f"  Esperadas : {COLUMNAS_Y_ESPERADAS}\n"
            f"  Recibidas : {sorted(cols_disponibles)}\n"
            "  Verifica que el CSV contenga al menos una de las columnas objetivo."
        )

    if requerir_todas_y and y_cols_faltantes:
        raise ValueError(
            f"[SDM-1] Modo estricto activo — faltan columnas Y requeridas: "
            f"{y_cols_faltantes}.\n"
            "  Desactiva 'requerir_todas_y' para entrenar con las disponibles."
        )

    if y_cols_faltantes:
        log.warning(
            "[SDM-1] Columnas Y faltantes en el CSV: %s. "
            "Entrenando multi-output con %d/%d targets: %s",
            y_cols_faltantes,
            len(y_cols_presentes),
            len(COLUMNAS_Y_ESPERADAS),
            y_cols_presentes,
        )

    # ── [SDM-2] Identificar columnas X ────────────────────────────────────────
    # Excluir: todas las Y (presentes y esperadas), control, timestamp
    excluir: set[str] = set(COLUMNAS_Y_ESPERADAS) | COLUMNAS_CONTROL
    if ts_col:
        excluir.add(ts_col)

    all_numeric: set[str] = set(df.select_dtypes(include=[np.number]).columns)

    # Prioridad 1: features preferidas disponibles y no excluidas
    x_pref = [c for c in FEATURES_X_PREFERIDAS if c in all_numeric and c not in excluir]
    # Prioridad 2: resto de numéricas no excluidas (mantiene orden del DataFrame)
    x_resto = [
        c for c in df.columns
        if c in all_numeric and c not in excluir and c not in set(x_pref)
    ]
    x_cols = x_pref + x_resto

    if not x_cols:
        raise ValueError(
            "[SDM-2] No se encontraron columnas numéricas para X tras el filtrado.\n"
            f"  Columnas excluidas: {sorted(excluir)}\n"
            f"  Columnas numéricas disponibles: {sorted(all_numeric)}"
        )

    # ── [SDM-3] Regla de seguridad: ninguna columna Y puede aparecer en X ─────
    # (imposible con la lógica anterior, pero actúa como red de seguridad)
    y_en_x = set(COLUMNAS_Y_ESPERADAS) & set(x_cols)
    if y_en_x:
        raise RuntimeError(
            f"[SDM-3] ⚠️  VIOLACIÓN DE SEGURIDAD DETECTADA\n"
            f"  Columnas de Y encontradas en X: {sorted(y_en_x)}\n"
            "  Esto indica un error en la lógica de preprocesamiento.\n"
            "  El entrenamiento fue ABORTADO para prevenir data leakage."
        )

    log.info(
        "[SDM] Esquema validado → X: %d features | Y: %d targets [%s]  |  "
        "[SDM-3] ✅ Sin leakage Y→X",
        len(x_cols), len(y_cols_presentes), ", ".join(y_cols_presentes),
    )
    return x_cols, y_cols_presentes, y_cols_faltantes


# ──────────────────────────────────────────────────────────────────────────────
# 5.  CARGA Y PREPROCESAMIENTO  [FIX-1]
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    """[FIX-1] Soporta CSV con cabeceras decorativas (comment='#')."""
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    """Detecta heurísticamente la columna temporal."""
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


def _preprocesar_multi(
    df:               pd.DataFrame,
    x_cols:           list[str],
    y_cols:           list[str],
    timestamp_col:    str,
    excluir_pandemia: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DatetimeIndex]:
    """
    Preprocesa el DataFrame para entrenamiento multi-output.

    Pasos
    ─────
    1. Parse temporal + set_index + sort.
    2. [FIX-3] Filtro pandemia 2020-2021 (opcional).
    3. Conversión object → numeric en columnas X e Y.
    4. Eliminar filas donde TODAS las Y son NaN.
    5. Devolver X [n × |x_cols|] e Y [n × |y_cols|] como DataFrames.

    Returns
    ───────
    X   : DataFrame de features
    Y   : DataFrame de targets
    idx : DatetimeIndex del DataFrame resultante
    """
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_excl = n_antes - len(df)
        if n_excl:
            log.info("[FIX-3] %d registros de %s eliminados.", n_excl, ANOS_PANDEMIA)

    # Convertir object → numeric (evita errores silenciosos con strings en columnas numéricas)
    all_target_cols = x_cols + y_cols
    obj_mask = df[all_target_cols].select_dtypes(include=["object", "string"]).columns
    if len(obj_mask):
        df[obj_mask] = df[obj_mask].apply(pd.to_numeric, errors="coerce")

    # Eliminar filas donde TODAS las Y son NaN (mantiene filas con Y parcialmente nulas)
    y_cols_ok = [c for c in y_cols if c in df.columns]
    x_cols_ok = [c for c in x_cols if c in df.columns]
    df = df.dropna(subset=y_cols_ok, how="all")

    X = df[x_cols_ok].copy()
    Y = df[y_cols_ok].copy()

    return X, Y, df.index


def _dividir_cronologico(
    X: pd.DataFrame,
    Y: pd.DataFrame,
    ratio: float = 0.80,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """División temporal estricta. El (1-ratio)*100% final = Test Ciego 100% intocable."""
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], Y.iloc[:corte], Y.iloc[corte:]


# ──────────────────────────────────────────────────────────────────────────────
# 6.  CONSTRUCTORES DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def _hgb_base(max_iter: int = 300, warm_start: bool = False) -> HistGradientBoostingRegressor:
    """
    HGB con regularización penalizada (especificación docente, heredada de v10).

    Parámetros de regularización
    ────────────────────────────
    · l2_regularization=5.0  → λ (penaliza pesos grandes en hojas)
    · min_samples_leaf=50    → α-proxy (fuerza hojas densas, efecto sparsity L1)
    · max_leaf_nodes=31      → complejidad geométrica máxima por árbol
    · early_stopping=False   → controlado por max_iter + best_iter del TSS
    """
    return HistGradientBoostingRegressor(
        max_iter          = max_iter,
        max_leaf_nodes    = HGB_MAX_LEAF_NODES,
        min_samples_leaf  = HGB_MIN_SAMPLES_LEAF,
        l2_regularization = HGB_L2_REG,
        learning_rate     = HGB_LEARNING_RATE,
        early_stopping    = False,
        warm_start        = warm_start,
        random_state      = 42,
    )


def _construir_modelo_multi(
    max_iter: int = 300,
    n_jobs:   int = N_JOBS_MULTI,
) -> MultiOutputRegressor:
    """
    MultiOutputRegressor(HistGBR) — un único modelo por parroquia.

    Arquitectura
    ────────────
    · 6 regressors independientes (uno por contaminante en COLUMNAS_Y_ESPERADAS).
    · n_jobs=N_JOBS_MULTI → entrenamiento paralelo entre regressors (joblib/loky).
    · El .pkl resultante encapsula los 6 estimadores ajustados (estimators_).

    Nota de paralelismo
    ────────────────────
    MultiOutputRegressor usa joblib (loky) para paralelizar entre los 6 regressors.
    Cada regressor HGB usa internamente OpenMP (C-level threads).
    Para evitar sobresubscripción de threads:
        N_JOBS_MULTI = min(6, cpu_count // 2)
    Esto da ~ cpu_count // 2 × 2 ≈ cpu_count threads totales activos.
    """
    return MultiOutputRegressor(
        estimator = _hgb_base(max_iter=max_iter, warm_start=False),
        n_jobs    = n_jobs,
    )


# ──────────────────────────────────────────────────────────────────────────────
# 7.  CURVAS DE APRENDIZAJE WARM-START POR TARGET
# ──────────────────────────────────────────────────────────────────────────────

def _curvas_por_target(
    X_tr_sc:  np.ndarray,
    Y_tr:     np.ndarray,
    X_va_sc:  np.ndarray,
    Y_va:     np.ndarray,
    y_cols:   list[str],
    max_iter: int  = 300,
    batch:    int  = BATCH_CURVA,
    log_fn         = None,
) -> dict[str, dict]:
    """
    Genera curvas de aprendizaje Train/Val RMSE por target con warm_start individual.

    Estrategia
    ──────────
    · MultiOutputRegressor NO soporta warm_start directamente.
    · Por cada target: un HGB independiente con warm_start=True (incremental).
    · El split X_tr_sc / X_va_sc es CRONOLÓGICO (último fold TSS → mayor ventana).
    · Complejidad: O(len(y_cols) × max_iter/batch) iteraciones de fit() total.
    · Los modelos warm_start son SOLO para diagnóstico visual; el modelo de
      predicción final usa MultiOutputRegressor entrenado con best_iter derivado.

    Returns
    ───────
    dict {target: {"train": list[float], "val": list[float], "metric": "RMSE"}}
    """
    if log_fn is None:
        log_fn = log.info

    curvas: dict[str, dict] = {}

    for t_idx, target in enumerate(y_cols):
        y_tr_t = Y_tr[:, t_idx].copy()
        y_va_t = Y_va[:, t_idx].copy()

        # Omitir targets completamente nulos en este split
        if np.all(np.isnan(y_tr_t)) or np.all(np.isnan(y_va_t)):
            log_fn(f"  [Curva:{target}] Datos nulos en este split — omitido.")
            curvas[target] = {"train": [], "val": [], "metric": "RMSE"}
            continue

        # HistGB maneja NaN nativamente; para las métricas filtramos NaN en val
        y_tr_t = np.where(np.isnan(y_tr_t), np.nanmean(y_tr_t), y_tr_t)

        model = _hgb_base(max_iter=batch, warm_start=True)
        tr_curve: list[float] = []
        va_curve: list[float] = []

        for step in range(batch, max_iter + 1, batch):
            model.max_iter = step
            model.fit(X_tr_sc, y_tr_t)

            preds_tr = model.predict(X_tr_sc)
            preds_va = model.predict(X_va_sc)

            # Métricas en train (sin NaN)
            rmse_tr = float(np.sqrt(mean_squared_error(y_tr_t, preds_tr)))

            # Métricas en val (filtrando NaN de Y real)
            mask_va = ~np.isnan(y_va_t)
            rmse_va = float(np.sqrt(mean_squared_error(
                y_va_t[mask_va], preds_va[mask_va]
            ))) if mask_va.sum() > 1 else np.nan

            tr_curve.append(rmse_tr)
            va_curve.append(rmse_va)

        curvas[target] = {"train": tr_curve, "val": va_curve, "metric": "RMSE"}
        best_local = (int(np.nanargmin(va_curve)) + 1) * batch
        log_fn(
            f"  [Curva:{target}] {len(tr_curve)} pasos  |  "
            f"Val RMSE final: {va_curve[-1]:.4f}  |  best_iter local: {best_local}"
        )

    return curvas


def _best_iter_from_curvas(
    curvas: dict[str, dict],
    batch:  int = BATCH_CURVA,
) -> int:
    """
    Deriva la iteración óptima global promediando las curvas Val RMSE de todos los targets.

    Lógica
    ──────
    1. Recopilar curvas de validación de cada target (ignorar vacías).
    2. Alinear al mínimo de longitud.
    3. Promediar elemento a elemento → curva global.
    4. Tomar el índice del mínimo → best_iter en unidades de árboles.
    """
    va_curves = [
        curvas[t]["val"] for t in curvas
        if curvas[t].get("val") and any(not np.isnan(v) for v in curvas[t]["val"])
    ]
    if not va_curves:
        log.warning("[best_iter] Sin curvas de validación — usando fallback 300.")
        return 300

    min_len  = min(len(c) for c in va_curves)
    arr      = np.array([[v if not np.isnan(v) else np.nanmean(c)
                          for v in c[:min_len]]
                         for c in va_curves])
    avg_val  = arr.mean(axis=0)
    best_idx = int(np.argmin(avg_val))
    best_iter = (best_idx + 1) * batch
    log.info(
        "[best_iter] Iteración óptima global = %d árboles "
        "(Val RMSE promedio mínimo: %.4f)",
        best_iter, avg_val[best_idx],
    )
    return best_iter


# ──────────────────────────────────────────────────────────────────────────────
# 8.  PIPELINE TimeSeriesSplit — MULTI-OUTPUT
# ──────────────────────────────────────────────────────────────────────────────

def _avg_r2_scorer(estimator, X_sc: np.ndarray, Y: np.ndarray) -> float:
    """
    Scorer personalizado para permutation_importance con multi-output.

    Calcula el R² promedio entre los targets presentes (ignora columnas
    completamente nulas), compatible con la API de sklearn.
    """
    Y_pred = estimator.predict(X_sc)
    scores = []
    for t_idx in range(Y.shape[1]):
        mask = ~np.isnan(Y[:, t_idx])
        if mask.sum() < 5:
            continue
        scores.append(r2_score(Y[mask, t_idx], Y_pred[mask, t_idx]))
    return float(np.nanmean(scores)) if scores else 0.0


def _entrenar_tss_multi(
    X:        pd.DataFrame,
    Y:        pd.DataFrame,
    max_iter: int = 300,
    n_splits: int = 10,
    log_fn        = None,
) -> tuple[pd.DataFrame, dict, int]:
    """
    TimeSeriesSplit(K=5) con MultiOutputRegressor + curvas warm_start en último fold.

    Por cada fold
    ─────────────
    1. Split cronológico train/val (ventana expansiva → fold K tiene más datos).
    2. StandardScaler ajustado SOLO con X_train del fold (sin leakage temporal).
    3. MultiOutputRegressor.fit() → predicciones Y para métricas per-target.
    4. Registro: MAE, RMSE, R² por target + R²_mean agregado.

    Curvas de aprendizaje
    ─────────────────────
    · Calculadas UNA VEZ en el último fold (fold K = mayor ventana).
    · Se usan HGB individuales con warm_start por target (ver _curvas_por_target).
    · Esto evita 30 entrenamientos (5 folds × 6 targets × warm_start) sin perder
      representatividad diagnóstica.
    · best_iter = mínimo de la curva Val RMSE promediada entre los 6 targets.

    Returns
    ───────
    df_metricas : métricas por fold (per-target + agregadas)
    curvas      : dict con curvas warm_start por target (último fold)
    best_iter   : int — iteración óptima global para el modelo final
    """
    if log_fn is None:
        log_fn = log.info

    y_cols = Y.columns.tolist()
    X_arr  = X.values
    Y_arr  = Y.values

    if len(X_arr) < n_splits * 50:
        raise ValueError(
            f"Datos insuficientes: {len(X_arr)} filas para {n_splits} folds.\n"
            f"Mínimo recomendado: {n_splits * 50} registros."
        )

    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_metrics: list[dict] = []
    last_fold_sc_data: tuple | None = None  # (X_tr_sc, Y_tr, X_va_sc, Y_va) del fold K

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
        X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
        Y_tr, Y_va = Y_arr[tr_idx], Y_arr[va_idx]

        # Scaler ajustado SOLO con el train de esta ventana temporal
        scaler  = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_va_sc = scaler.transform(X_va)

        log_fn(
            f"  ── Fold {fold_idx}/{n_splits} | "
            f"Train: {len(X_tr):,}  Val: {len(X_va):,} "
            f"[{X.index[tr_idx[0]].date()} → {X.index[tr_idx[-1]].date()}] ──"
        )

        # Entrenar MultiOutputRegressor en este fold
        modelo_fold = _construir_modelo_multi(max_iter=max_iter)
        modelo_fold.fit(X_tr_sc, Y_tr)
        Y_pred = modelo_fold.predict(X_va_sc)

        row = {"Fold": fold_idx, "N_train": len(X_tr), "N_val": len(X_va)}
        r2_vals = []

        for t_idx, target in enumerate(y_cols):
            mask = ~np.isnan(Y_va[:, t_idx])
            if mask.sum() < 5:
                row[f"MAE_{target}"]  = np.nan
                row[f"RMSE_{target}"] = np.nan
                row[f"R2_{target}"]   = np.nan
                continue
            y_va_t = Y_va[mask, t_idx]
            y_pr_t = Y_pred[mask, t_idx]
            row[f"MAE_{target}"]  = mean_absolute_error(y_va_t, y_pr_t)
            row[f"RMSE_{target}"] = float(np.sqrt(mean_squared_error(y_va_t, y_pr_t)))
            row[f"R2_{target}"]   = r2_score(y_va_t, y_pr_t)
            r2_vals.append(row[f"R2_{target}"])

        row["R2_mean"]   = float(np.nanmean(r2_vals)) if r2_vals else np.nan
        row["MAE_mean"]  = float(np.nanmean([row.get(f"MAE_{t}",  np.nan) for t in y_cols]))
        row["RMSE_mean"] = float(np.nanmean([row.get(f"RMSE_{t}", np.nan) for t in y_cols]))
        fold_metrics.append(row)

        r2_summary = "  ".join(
            f"{t}:{row.get(f'R2_{t}', np.nan):.3f}" for t in y_cols
        )
        log_fn(
            f"  Fold {fold_idx}/{n_splits} → {r2_summary}  "
            f"| R²_mean={row['R2_mean']:.3f}"
        )

        # Guardar datos escalados del último fold para las curvas
        if fold_idx == n_splits:
            last_fold_sc_data = (X_tr_sc, Y_tr, X_va_sc, Y_va)

    # ── Curvas de aprendizaje warm_start — solo en el último fold ────────────
    log_fn(
        f"  Calculando curvas warm_start en Fold {n_splits} "
        f"(mayor ventana, más representativo)…"
    )
    X_tr_sc_lf, Y_tr_lf, X_va_sc_lf, Y_va_lf = last_fold_sc_data
    curvas = _curvas_por_target(
        X_tr_sc_lf, Y_tr_lf, X_va_sc_lf, Y_va_lf,
        y_cols=y_cols, max_iter=max_iter, batch=BATCH_CURVA, log_fn=log_fn,
    )
    best_iter = _best_iter_from_curvas(curvas, batch=BATCH_CURVA)

    return pd.DataFrame(fold_metrics), curvas, best_iter


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo: str, xlabel: str, ylabel: str) -> None:
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje_multi(
    curvas:    dict[str, dict],
    parroquia: str = "",
    batch:     int = BATCH_CURVA,
) -> plt.Figure:
    """
    Curvas de aprendizaje Train/Val RMSE por contaminante + curva global promediada.

    Layout
    ──────
    · Líneas finas (α=0.45) con color por target: sólida=Train, discontinua=Val.
    · Línea gruesa blanca: promedio global de todos los targets.
    · Banda sombreada gris entre Train global y Val global.
    · Línea vertical dorada: mejor iteración global.
    · Generadas en el último fold TSS (mayor ventana de entrenamiento).
    """
    fig, ax = plt.subplots(figsize=(13, 5.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    targets_ok = [t for t, v in curvas.items() if v.get("val") and len(v["val"]) >= 2]
    if not targets_ok:
        ax.text(0.5, 0.5, "Sin datos de curva disponibles.", ha="center", va="center",
                color=TEXT_PLOT, fontsize=11, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, "Curva de Aprendizaje Multi-Output", "", "")
        plt.tight_layout()
        return fig

    min_len = min(len(curvas[t]["val"]) for t in targets_ok)
    x = (np.arange(min_len) + 1) * batch

    all_tr, all_va = [], []
    for target in targets_ok:
        color = TARGET_COLORS.get(target, "#94A3B8")
        tr = np.array(curvas[target]["train"][:min_len])
        va = np.array(curvas[target]["val"][:min_len])
        # Imputar NaN para visualización (no afecta best_iter)
        tr = np.where(np.isnan(tr), np.nanmean(tr), tr)
        va = np.where(np.isnan(va), np.nanmean(va), va)
        all_tr.append(tr)
        all_va.append(va)
        ax.plot(x, tr, color=color, lw=1.2, alpha=0.40, linestyle="-",  zorder=3)
        ax.plot(x, va, color=color, lw=1.2, alpha=0.40, linestyle="--", zorder=3,
                label=target)

    # Curva global promediada (Train + Val)
    global_tr = np.array(all_tr).mean(axis=0)
    global_va = np.array(all_va).mean(axis=0)
    ax.plot(x, global_tr, color=COLOR_GLOBAL, lw=2.6, linestyle="-",
            label="μ global Train", zorder=6, alpha=0.95)
    ax.plot(x, global_va, color=COLOR_GLOBAL, lw=2.6, linestyle="--",
            label="μ global Val",   zorder=6, alpha=0.95)
    ax.fill_between(x, global_tr, global_va,
                    alpha=0.07, color=COLOR_GLOBAL, zorder=4)

    # Mejor iteración global
    best_idx  = int(np.argmin(global_va))
    best_iter = x[best_idx]
    best_val  = global_va[best_idx]
    ax.axvline(best_iter, color="#F59E0B", lw=1.8, linestyle=":", alpha=0.9, zorder=7,
               label=f"Best iter: {best_iter}  (RMSE global={best_val:.4f})")
    ax.scatter([best_iter], [best_val], color="#F59E0B", s=80, zorder=9,
               edgecolors="#FCD34D", lw=1.5)
    offset = max(batch, int((x[-1] - x[0]) * 0.02))
    ax.annotate(
        f"  ← {best_iter} árboles",
        xy=(best_iter, best_val), xytext=(best_iter + offset, best_val),
        color="#F59E0B", fontsize=8.5, va="center", zorder=9,
    )

    titulo = "Curva de Aprendizaje Multi-Output — HistGB penalizado (último fold TSS)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    _aplicar_estilo_ax(ax, titulo, "Árboles entrenados (iteraciones acumuladas)", "RMSE  (↓ mejor)")
    ax.legend(
        framealpha=0.22, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
        edgecolor=GRID_PLOT, fontsize=9, loc="upper right", ncol=2,
    )
    nota = (
        f"Líneas finas coloreadas = curva individual por target  ·  "
        f"Línea blanca = μ global ({len(targets_ok)} targets)  ·  "
        f"Batch warm-start = {batch} árboles/paso"
    )
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


def _fig_prediccion_multi(
    Y_te:      pd.DataFrame,
    Y_pred:    np.ndarray,
    parroquia: str = "",
    days:      int = 7,
) -> plt.Figure:
    """
    Grid 2×3 de subplots — Real vs Predicho por contaminante (Test Ciego).
    """
    targets = Y_te.columns.tolist()
    n_cols  = 3
    n_rows  = -(-len(targets) // n_cols)   # ceil division

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(14, n_rows * 3.6),
        facecolor=BG_PLOT, squeeze=False,
    )
    titulo_global = (
        f"[{parroquia}]  Real vs Predicho — Test Ciego (últimos {days} días)"
        if parroquia else f"Real vs Predicho — Test Ciego (últimos {days} días)"
    )
    fig.suptitle(titulo_global, fontsize=12, fontweight="bold", color=TEXT_PLOT, y=1.01)

    cutoff = Y_te.index.max() - pd.Timedelta(days=days)

    for ax_idx, target in enumerate(targets):
        row_i, col_i = divmod(ax_idx, n_cols)
        ax = axes[row_i][col_i]
        ax.set_facecolor(BG_PLOT)

        mask     = Y_te.index >= cutoff
        y_real   = Y_te.loc[mask, target]
        y_pr     = Y_pred[mask.values, ax_idx]
        color    = TARGET_COLORS.get(target, COLOR_REAL)

        ax.plot(y_real.index, y_real.values, label="Real",     color=color,      lw=1.8, alpha=0.95)
        ax.plot(y_real.index, y_pr,          label="Predicho", color=COLOR_PRED, lw=1.4,
                linestyle="--", alpha=0.85)
        ax.fill_between(y_real.index, y_real.values, y_pr, alpha=0.07, color=COLOR_PRED)

        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=max(1, days // 5)))
        plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=7.5, color=TEXT_PLOT)
        plt.setp(ax.get_yticklabels(), fontsize=7.5, color=TEXT_PLOT)
        _aplicar_estilo_ax(ax, target, "", target)
        ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
                  edgecolor=GRID_PLOT, fontsize=7.5)

    for extra in range(len(targets), n_rows * n_cols):
        ri, ci = divmod(extra, n_cols)
        axes[ri][ci].set_visible(False)

    plt.tight_layout()
    return fig


def _fig_feature_importance(
    fi_df:     pd.DataFrame,
    y_cols:    list[str],
    parroquia: str = "",
) -> plt.Figure:
    """
    Importancia por permutación — Δ R² promedio entre los 6 targets.
    Top-25 features ordenadas de mayor a menor importancia.
    """
    n = min(len(fi_df), 25)
    fi_top = fi_df.head(n)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_top["Importance"]]
    ax.barh(fi_top["Feature"][::-1], fi_top["Importance"][::-1],
            xerr=fi_top["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo = f"Feature Importance  ·  μ R² ({', '.join(y_cols)})  ·  Permutation (Test Ciego)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo, fontsize=10, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R² promedio, multi-target)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(X_df: pd.DataFrame, Y_df: pd.DataFrame, parroquia: str = "") -> plt.Figure | None:
    """Mapa de calor de correlación Pearson X×Y — columnas Y resaltadas."""
    df_full = pd.concat([X_df, Y_df], axis=1).select_dtypes(include=[np.number])
    if df_full.shape[1] < 2:
        return None
    corr = df_full.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.52), max(7, n * 0.48)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    sns.heatmap(
        corr, mask=mask, cmap=sns.diverging_palette(230, 20, as_cmap=True),
        vmin=-1, vmax=1, center=0,
        annot=True, fmt=".2f", annot_kws={"size": 6.5, "color": TEXT_PLOT},
        linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
        cbar_kws={"shrink": 0.55},
    )
    cols_list = list(corr.columns)
    for target in Y_df.columns:
        if target in cols_list:
            i = cols_list.index(target)
            ax.add_patch(plt.Rectangle(
                (i, 0), 1, n, fill=False,
                edgecolor=TARGET_COLORS.get(target, "#F97316"), lw=2.0, clip_on=False,
            ))
    titulo = "Correlación Pearson — X × Y  (columnas Y resaltadas)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=7.5)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


def _fig_r2_por_target(
    m_te_dict: dict[str, dict],
    y_cols:    list[str],
    parroquia: str = "",
) -> plt.Figure:
    """
    Bar chart horizontal — R² por contaminante en Test Ciego.
    Línea vertical en R²=0.70 como umbral de calidad.
    """
    r2_vals = [m_te_dict.get(t, {}).get("R2", 0.0) for t in y_cols]
    colors  = [TARGET_COLORS.get(t, "#94A3B8") for t in y_cols]

    fig, ax = plt.subplots(figsize=(9, max(3.5, len(y_cols) * 0.65)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    bars = ax.barh(y_cols[::-1], r2_vals[::-1], color=colors[::-1], alpha=0.85, height=0.55)
    for bar, val in zip(bars, r2_vals[::-1]):
        ax.text(
            max(val + 0.015, 0.02), bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", ha="left",
            color=TEXT_PLOT, fontsize=9, fontweight="bold",
        )

    ax.axvline(0.70, color="#F59E0B", lw=1.5, linestyle=":", alpha=0.75,
               label="Umbral recomendado R² = 0.70")
    ax.set_xlim(0, 1.10)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo = "R² por Contaminante — Test Ciego 100%"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    _aplicar_estilo_ax(ax, titulo, "R²  (↑ mejor)", "")
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9)
    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PREDICCIÓN AQI — Tabla de niveles para PM25
# ──────────────────────────────────────────────────────────────────────────────

def _nivel_aqi_pm25(val: float) -> str:
    if val <= 12:    return "🟢 Buena"
    if val <= 35.4:  return "🟡 Moderada"
    if val <= 55.4:  return "🟠 Insalubre (grupos sensibles)"
    if val <= 150.4: return "🔴 Insalubre"
    if val <= 250.4: return "🟣 Muy insalubre"
    return "🔴⚫ Peligrosa"


_AQI_UMBRALES: dict[str, list[tuple[float, str]]] = {
    "PM10": [(54, "🟢 Buena"), (154, "🟡 Moderada"), (254, "🟠 Insalubre (sensibles)"),
             (354, "🔴 Insalubre"), (float("inf"), "🟣 Muy insalubre")],
    "O3":   [(54, "🟢 Buena"), (70, "🟡 Moderada"), (85, "🟠 Insalubre (sensibles)"),
             (105, "🔴 Insalubre"), (float("inf"), "🟣 Muy insalubre")],
    "CO":   [(4.4, "🟢 Buena"), (9.4, "🟡 Moderada"), (12.4, "🟠 Insalubre (sensibles)"),
             (15.4, "🔴 Insalubre"), (float("inf"), "🟣 Muy insalubre")],
    "NO2":  [(53, "🟢 Buena"), (100, "🟡 Moderada"), (360, "🟠 Insalubre (sensibles)"),
             (649, "🔴 Insalubre"), (float("inf"), "🟣 Muy insalubre")],
    "SO2":  [(35, "🟢 Buena"), (75, "🟡 Moderada"), (185, "🟠 Insalubre (sensibles)"),
             (304, "🔴 Insalubre"), (float("inf"), "🟣 Muy insalubre")],
}


def _nivel_aqi(target: str, val: float) -> str:
    if target == "PM25":
        return _nivel_aqi_pm25(val)
    umbrales = _AQI_UMBRALES.get(target, [])
    for umbral, label in umbrales:
        if val <= umbral:
            return label
    return "—"


# ──────────────────────────────────────────────────────────────────────────────
# 11.  PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    nombre_modelo:    str,
    max_iter:         int,
    plot_days:        int,
    train_ratio:      float,
    excluir_pandemia: bool,
):
    """
    Pipeline completo Multi-Output — un único .pkl por parroquia.

    Pasos
    ─────
    1.  Resolución del archivo CSV de entrada.
    2.  Detección automática de columna temporal.
    3.  [SDM] Detección dinámica del esquema X/Y con validación de seguridad.
    4.  Preprocesamiento temporal (parse, sort, filtro pandemia, conversión).
    5.  División cronológica estricta: train_ratio% train / resto test ciego.
    6.  TimeSeriesSplit(K=5) con MultiOutputRegressor → métricas per-fold.
        + Curvas warm_start en último fold → best_iter global.
    7.  Modelo final: StandardScaler + MultiOutputRegressor(best_iter).
        Entrenado sobre el 100% del train (sin exponer test).
    8.  Evaluación per-target en test ciego (MAE, RMSE, R²).
    9.  Feature Importance: Permutation Δ R² promedio (multi-output scorer).
    10. Construcción de Markdown de métricas + tablas.
    11. Persistencia: único .pkl por parroquia con todo el estado necesario.
    12. Actualización de SESION para la pestaña de consulta.
    13. Generación de 6 figuras: métricas R², pred multi-output, curvas,
        feature importance, heatmap, R² por target.
    """
    logs: list[str] = []

    def info(m):  log.info(m);    logs.append(f"✅ {m}")
    def warn(m):  log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):   log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])

    VACIO = (None, None, None, None, None, None)   # 6 figuras

    try:
        # ── 11.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        nombre_modelo = nombre_modelo.strip() or "surrogate_multi_hgb"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 11.2  Cargar CSV ──────────────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ Sin columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp detectado: '{ts_col}'")

        # ── 11.3  [SDM] Detección dinámica de esquema ─────────────────────────
        info("[SDM] Analizando esquema X/Y automáticamente…")
        try:
            x_cols, y_cols, y_faltantes = _detectar_esquema(df, ts_col)
        except (ValueError, RuntimeError) as sdm_err:
            err(str(sdm_err))
            return f"### ❌ Error [SDM]\n```\n{sdm_err}\n```", *VACIO, _estado()

        info(f"[SDM] ✅ X: {len(x_cols)} features | Y: {len(y_cols)} targets {y_cols}")
        info("[SDM] ✅ [SDM-3] Regla de seguridad: ninguna Y filtrada hacia X.")
        if y_faltantes:
            warn(f"[SDM] Columnas Y faltantes en el CSV: {y_faltantes}")

        # ── 11.4  Preprocesamiento ────────────────────────────────────────────
        X, Y, idx = _preprocesar_multi(df, x_cols, y_cols, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: {ANOS_PANDEMIA}")

        n_nulos_x = int(X.isna().sum().sum())
        n_nulos_y = int(Y.isna().sum().sum())
        if n_nulos_x:
            warn(f"{n_nulos_x:,} NaN en X — HistGB los maneja nativamente.")
        if n_nulos_y:
            warn(f"{n_nulos_y:,} NaN en Y — filas con todos Y=NaN eliminadas.")

        # ── 11.5  División cronológica ────────────────────────────────────────
        X_tr, X_te, Y_tr, Y_te = _dividir_cronologico(X, Y, train_ratio)
        info(
            f"Split cronológico {int(train_ratio*100)}/{int((1-train_ratio)*100)}: "
            f"Train={len(X_tr):,} | Test ciego={len(X_te):,}"
        )
        info("El 20% test ciego NO se expone durante TSS ni curvas de aprendizaje.")

        # ── 11.6  TimeSeriesSplit + curvas ────────────────────────────────────
        info(
            f"TimeSeriesSplit (K=5) · MultiOutputRegressor · max_iter={max_iter} · "
            f"n_jobs={N_JOBS_MULTI}  (paralelo entre {len(y_cols)} targets)"
        )
        info(
            f"HGB: l2={HGB_L2_REG}  min_samples_leaf={HGB_MIN_SAMPLES_LEAF}  "
            f"max_leaf_nodes={HGB_MAX_LEAF_NODES}  lr={HGB_LEARNING_RATE}"
        )

        tss_logs: list[str] = []
        def tss_log(m): log.info(m); tss_logs.append(m)

        df_tss, curvas, best_iter = _entrenar_tss_multi(
            X_tr, Y_tr, max_iter=max_iter, n_splits=10, log_fn=tss_log,
        )
        for l in tss_logs:
            logs.append(l)

        info(f"TSS completado — best_iter global: {best_iter} árboles")

        # ── 11.7  Modelo final sobre el 80% completo ──────────────────────────
        final_max_iter = min(best_iter + 2 * BATCH_CURVA, max_iter)
        info(
            f"Entrenando modelo final (MultiOutputRegressor): "
            f"max_iter={final_max_iter}  (best_iter={best_iter} + buffer={2*BATCH_CURVA})"
        )

        scaler_final = StandardScaler()
        X_tr_sc      = scaler_final.fit_transform(X_tr)
        X_te_sc      = scaler_final.transform(X_te)

        modelo_final = _construir_modelo_multi(max_iter=final_max_iter)
        modelo_final.fit(X_tr_sc, Y_tr.values)
        info("Modelo final entrenado. Test ciego nunca fue expuesto al entrenamiento.")

        Y_pred_tr = modelo_final.predict(X_tr_sc)
        Y_pred_te = modelo_final.predict(X_te_sc)

        # ── 11.8  Métricas per-target ─────────────────────────────────────────
        m_tr_dict: dict[str, dict] = {}
        m_te_dict: dict[str, dict] = {}

        for t_idx, target in enumerate(y_cols):
            for Y_real_arr, Y_pred_arr, dest in [
                (Y_tr.values, Y_pred_tr, m_tr_dict),
                (Y_te.values, Y_pred_te, m_te_dict),
            ]:
                mask = ~np.isnan(Y_real_arr[:, t_idx])
                if mask.sum() < 5:
                    dest[target] = {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan}
                    continue
                yt = Y_real_arr[mask, t_idx]
                yp = Y_pred_arr[mask, t_idx]
                dest[target] = dict(
                    MAE  = mean_absolute_error(yt, yp),
                    RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                    R2   = r2_score(yt, yp),
                )

        r2_tr_mean = float(np.nanmean([m_tr_dict[t]["R2"] for t in y_cols]))
        r2_te_mean = float(np.nanmean([m_te_dict[t]["R2"] for t in y_cols]))
        gap = r2_tr_mean - r2_te_mean
        if gap > 0.15:
            warn(f"Posible overfitting residual: ΔR²(train-test) = {gap:.3f}")

        # ── 11.9  Feature Importance (multi-output scorer) ────────────────────
        info("Calculando Permutation Importance en test ciego (avg R² multi-target)…")
        perm = permutation_importance(
            modelo_final, X_te_sc, Y_te.values,
            n_repeats=8, random_state=42,
            scoring=_avg_r2_scorer,
        )
        fi_df = (
            pd.DataFrame({
                "Feature":    x_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )
        info("Feature Importance lista.")

        # ── 11.10  Markdown de métricas ───────────────────────────────────────
        hw_label = (
            f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}  "
            f"_(HistGB usa CPU nativo de scikit-learn)_"
        )
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        # Tabla TSS — encabezado dinámico por los targets presentes
        header_tss = " | ".join(f"R²({t})" for t in y_cols)
        sep_tss    = "|:---:" * len(y_cols)
        filas_tss  = []
        for _, row in df_tss.iterrows():
            r2_cells = " | ".join(
                f"`{row.get(f'R2_{t}', np.nan):.3f}`" for t in y_cols
            )
            r2_m  = row.get("R2_mean", np.nan)
            ico   = "🟢" if r2_m >= 0.7 else ("🟡" if r2_m >= 0.5 else "🔴")
            filas_tss.append(
                f"| **{int(row['Fold'])}** "
                f"| `{int(row['N_train']):,}` | `{int(row['N_val']):,}` "
                f"| {r2_cells} | {ico} `{r2_m:.3f}` |"
            )
        # Fila μ±σ
        r2_g_mean = df_tss["R2_mean"].mean()
        r2_g_std  = df_tss["R2_mean"].std()
        r2_t_mean_cells = " | ".join(
            f"`{df_tss[f'R2_{t}'].mean():.3f}`" if f"R2_{t}" in df_tss.columns else "`—`"
            for t in y_cols
        )
        ico_g = "🟢" if r2_g_mean >= 0.7 else ("🟡" if r2_g_mean >= 0.5 else "🔴")
        filas_tss.append(
            f"| **μ±σ** | — | — | {r2_t_mean_cells} "
            f"| {ico_g} `{r2_g_mean:.3f} ± {r2_g_std:.3f}` |"
        )
        tabla_tss = "\n".join(filas_tss)

        # Tabla test ciego per-target
        filas_te = []
        for target in y_cols:
            m  = m_te_dict.get(target, {})
            mt = m_tr_dict.get(target, {})
            r2 = m.get("R2", np.nan)
            ico = "🟢" if r2 >= 0.7 else ("🟡" if r2 >= 0.5 else "🔴")
            filas_te.append(
                f"| **{target}** "
                f"| `{mt.get('MAE',  np.nan):.4f}` "
                f"| `{mt.get('RMSE', np.nan):.4f}` "
                f"| `{mt.get('R2',   np.nan):.4f}` "
                f"| `{m.get('MAE',   np.nan):.4f}` "
                f"| `{m.get('RMSE',  np.nan):.4f}` "
                f"| {ico} `{r2:.4f}` |"
            )
        # Fila agregada global
        mae_tr_g  = float(np.nanmean([m_tr_dict.get(t, {}).get("MAE",  np.nan) for t in y_cols]))
        rmse_tr_g = float(np.nanmean([m_tr_dict.get(t, {}).get("RMSE", np.nan) for t in y_cols]))
        mae_te_g  = float(np.nanmean([m_te_dict.get(t, {}).get("MAE",  np.nan) for t in y_cols]))
        rmse_te_g = float(np.nanmean([m_te_dict.get(t, {}).get("RMSE", np.nan) for t in y_cols]))
        ico_ag = "🟢" if r2_te_mean >= 0.7 else ("🟡" if r2_te_mean >= 0.5 else "🔴")
        filas_te.append(
            f"| **μ global** "
            f"| `{mae_tr_g:.4f}` | `{rmse_tr_g:.4f}` | `{r2_tr_mean:.4f}` "
            f"| `{mae_te_g:.4f}` | `{rmse_te_g:.4f}` "
            f"| {ico_ag} `{r2_te_mean:.4f}` |"
        )
        tabla_te = "\n".join(filas_te)

        overfitting_badge = (
            f"⚠️ **Overfitting residual** — ΔR²(train-test) = `{gap:.3f}`"
            if gap > 0.15 else
            "✅ Sin señales de overfitting (l2=5.0 + min_samples_leaf=50 activos)."
        )

        sdm_nota_faltantes = (
            f"\n> ⚠️ **Columnas Y faltantes** en el CSV: `{y_faltantes}` — excluidas del modelo."
            if y_faltantes else ""
        )

        metricas_md = f"""
## 📊 Surrogate Model v11.0 — *{parroquia}*

**Arquitectura:** `MultiOutputRegressor(HistGBR)` — **{len(y_cols)} salidas simultáneas**  ·  1 único `.pkl` por parroquia

---

### 🗂️ Detección Dinámica de Esquema [SDM]

| Campo | Detalle |
|-------|---------|
| **Y — Targets** | `{y_cols}` |
| **X — Features** | `{len(x_cols)}` columnas (meteorología · lags · cíclicas) |
| **[SDM-3] Leakage Y→X** | ✅ Sin contaminación detectada |
| **Columnas Y faltantes** | `{y_faltantes if y_faltantes else "ninguna"}` |
{sdm_nota_faltantes}

---

### Validación Cronológica TimeSeriesSplit (K=5) — Ventanas Expansivas

| Fold | N Train | N Val | {header_tss} | R²\_mean |
|:----:|:-------:|:-----:|{sep_tss}|:--------:|
{tabla_tss}

> **Validación temporal honesta**: cada fold valida sobre observaciones estrictamente
> **posteriores** al bloque de entrenamiento. Curvas de aprendizaje con `warm_start`
> calculadas en el último fold (mayor ventana → más representativas).

---

### Modelo Final — Test Ciego {int((1-train_ratio)*100)}% (último segmento, 100% intocable)

> `max_iter` final = **{final_max_iter}** árboles  (best\_iter={best_iter} + buffer={2*BATCH_CURVA})

| Target | 🟦 MAE Train | 🟦 RMSE Train | 🟦 R² Train | 🟧 MAE Test | 🟧 RMSE Test | 🟧 R² Test |
|:------:|:------------:|:-------------:|:-----------:|:-----------:|:------------:|:-----------:|
{tabla_te}

{overfitting_badge}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Hardware | {hw_label} |
| n\_jobs MultiOutput | `{N_JOBS_MULTI}` (paralelo entre {len(y_cols)} regressors) |
| max\_leaf\_nodes | `{HGB_MAX_LEAF_NODES}` |
| l2\_regularization | `{HGB_L2_REG}` (λ) |
| min\_samples\_leaf | `{HGB_MIN_SAMPLES_LEAF}` (α-proxy) |
| learning\_rate | `{HGB_LEARNING_RATE}` |
| Filtro pandemia | {pandemia_label} |
"""

        # ── 11.11  Persistencia — único .pkl por parroquia ────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                # Modelo e infraestructura de inferencia
                "modelo":         modelo_final,   # MultiOutputRegressor(6×HGB)
                "scaler":         scaler_final,   # StandardScaler ajustado en X_train
                "x_cols":         x_cols,
                "y_cols":         y_cols,
                "y_faltantes":    y_faltantes,
                "parroquia":      parroquia,
                # Diagnóstico y trazabilidad
                "tss_resumen":    df_tss,
                "curvas":         curvas,
                "best_iter":      best_iter,
                "final_max_iter": final_max_iter,
                "m_te":           m_te_dict,
                "m_tr":           m_tr_dict,
                "fi_df":          fi_df,
            }, fh)
        info(f"Modelo guardado (único .pkl): {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        df_tss.to_csv(OUTPUT_DIR / f"{tag}_tss_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia":     parroquia,
            "Archivo":       ruta_csv,
            "Targets":       str(y_cols),
            "N_features":    len(x_cols),
            "best_iter":     best_iter,
            "R2_train_mean": r2_tr_mean,
            "R2_test_mean":  r2_te_mean,
            **{f"R2_test_{t}": m_te_dict.get(t, {}).get("R2", np.nan) for t in y_cols},
            "GPU":           GPU_DISPONIBLE,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 11.12  Actualizar sesión ──────────────────────────────────────────
        SESION.update({
            "modelo":      modelo_final,
            "scaler":      scaler_final,
            "feat_cols":   x_cols,
            "target_cols": y_cols,
            "parroquia":   parroquia,
            "feat_stats": {
                col: {
                    "min":  float(X[col].min()),
                    "max":  float(X[col].max()),
                    "mean": float(X[col].mean()),
                }
                for col in x_cols
            },
        })
        info("Sesión actualizada → pestaña Consulta lista para predicciones.")

        # ── 11.13  Figuras ────────────────────────────────────────────────────
        fig_lc   = _fig_curva_aprendizaje_multi(curvas, parroquia, batch=BATCH_CURVA)
        fig_pred = _fig_prediccion_multi(Y_te, Y_pred_te, parroquia, plot_days)
        fig_fi   = _fig_feature_importance(fi_df, y_cols, parroquia)
        fig_hm   = _fig_heatmap(X, Y, parroquia)
        fig_r2   = _fig_r2_por_target(m_te_dict, y_cols, parroquia)
        info("6 figuras generadas.")

        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, fig_r2, _estado()

    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado()
        )


# ──────────────────────────────────────────────────────────────────────────────
# 12.  PREDICCIÓN DESDE SESIÓN (pestaña Consulta)
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    """
    Predicción simultánea de los 6 contaminantes usando el modelo en sesión.
    Devuelve una tabla Markdown con estimación + nivel AQI por target.
    """
    if SESION["modelo"] is None:
        return "### ⚠️ Entrena primero un modelo para habilitar esta sección."
    try:
        import json
        vals        = json.loads(valores_json)
        feat_cols   = SESION["feat_cols"]
        target_cols = SESION["target_cols"]
        parroquia   = SESION["parroquia"]

        # Construir fila de entrada con fallback a la media del feature
        row    = {
            col: float(vals.get(col, SESION["feat_stats"].get(col, {}).get("mean", 0.0)))
            for col in feat_cols
        }
        X_in   = pd.DataFrame([row])
        X_sc   = SESION["scaler"].transform(X_in)
        preds  = SESION["modelo"].predict(X_sc)[0]   # array (n_targets,)

        filas = []
        for t_idx, target in enumerate(target_cols):
            pred  = float(preds[t_idx])
            nivel = _nivel_aqi(target, pred)
            filas.append(f"| **{target}** | `{pred:.3f} µg/m³` | {nivel} |")

        return (
            f"## 🔮 Predicción Simultánea — *{parroquia}*\n\n"
            f"| Contaminante | Estimación | Nivel AQI |\n"
            f"|:------------:|:----------:|:---------:|\n"
            + "\n".join(filas)
            + f"\n\n> *Predicción generada por `MultiOutputRegressor` — "
              f"{len(feat_cols)} features de entrada.*"
        )
    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 13.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}
body, .gradio-container { background:var(--bg) !important; color:var(--text) !important;
    font-family:'Inter','Segoe UI',sans-serif !important; }
.gr-group, .gr-box { background:var(--card) !important;
    border:1px solid var(--border) !important; border-radius:var(--r) !important; padding:16px !important; }
input, textarea, select { background:var(--input) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:6px !important; }
label, .gr-label { color:var(--muted) !important; font-size:.8rem !important;
    font-weight:700 !important; text-transform:uppercase; letter-spacing:.05em !important; }
button.primary { background:linear-gradient(135deg,var(--accent),#7C3AED) !important;
    color:#fff !important; border:none !important; border-radius:8px !important;
    font-weight:800 !important; font-size:1rem !important; padding:12px 28px !important;
    box-shadow:0 4px 20px rgba(99,102,241,.45); transition:all .15s; }
button.primary:hover { transform:translateY(-2px); box-shadow:0 6px 28px rgba(99,102,241,.6); }
button.secondary { background:var(--card) !important; color:var(--text) !important;
    border:1px solid var(--border) !important; border-radius:8px !important; }
.gr-markdown { color:var(--text) !important; }
.gr-markdown table { border-collapse:collapse; width:100%; }
.gr-markdown th { background:#1E293B; color:var(--accentH); padding:8px 14px; border:1px solid var(--border); }
.gr-markdown td { color:var(--text); padding:7px 14px; border:1px solid var(--border); }
.gr-markdown tr:nth-child(even) td { background:#19253a; }
.gr-file { border:2px dashed var(--accent) !important; border-radius:var(--r) !important; }
.hero { text-align:center; padding:24px 0 6px; }
.hero h1 { font-size:2rem; font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent; }
.hero p { color:var(--muted); font-size:.88rem; }
.gpu-badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.75rem;
    font-weight:700; margin-top:4px; }
.gpu-on  { background:rgba(34,197,94,.18);  color:#4ADE80; border:1px solid #22C55E; }
.gpu-off { background:rgba(99,102,241,.15); color:#A5B4FC; border:1px solid #6366F1; }
.logs-box { background:#0B1527 !important; border:1px solid #1D3557 !important;
    border-radius:8px; padding:10px 14px; font-family:'JetBrains Mono',monospace;
    font-size:.76rem; color:#7DD3FC; max-height:140px; overflow-y:auto; line-height:1.6; }
.reg-badge { background:rgba(99,102,241,.12); border:1px solid #4338CA;
    border-radius:8px; padding:8px 14px; font-size:.78rem; color:#C7D2FE; margin-top:6px; }
.sdm-badge { background:rgba(34,197,94,.10); border:1px solid #22C55E;
    border-radius:8px; padding:8px 14px; font-size:.78rem; color:#86EFAC; margin-top:6px; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 14.  UI — INTERFAZ GRADIO
# ──────────────────────────────────────────────────────────────────────────────

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 {GPU_MSG}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU · HistGB nativo scikit-learn</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v11",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Multi-Output · Detección Dinámica de Esquema [SDM] · HistGB Penalizado · TSS · v11.0</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ══════════════════════════════════════════
            # PANEL IZQUIERDO — CONTROLES
            # ══════════════════════════════════════════
            with gr.Column(scale=1, min_width=340):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs detectados en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")

                    # Badge SDM — auto-detección
                    gr.HTML(
                        '<div class="sdm-badge">'
                        '📐 <b>[SDM]</b> Schema auto-detectado — '
                        'Y = {PM25, PM10, O3, CO, NO2, SO2} · '
                        'X = resto de numéricas (meteo + lags + cíclicas)'
                        '</div>'
                    )
                    gr.HTML("<div style='height:4px'/>")

                    # Badge regularización
                    gr.HTML(
                        f'<div class="reg-badge">'
                        f'<b>HistGB</b> · λ(L2)={HGB_L2_REG} · α-proxy(min_samples)={HGB_MIN_SAMPLES_LEAF}'
                        f' · max_leaf_nodes={HGB_MAX_LEAF_NODES} · n_jobs={N_JOBS_MULTI}'
                        f'</div>'
                    )
                    gr.HTML("<div style='height:6px'/>")

                    nombre_modelo_input = gr.Textbox(
                        label="Nombre del modelo (.pkl)",
                        value="surrogate_multi_hgb",
                        placeholder="surrogate_multi_hgb_carapungo",
                    )

                    max_iter_slider = gr.Slider(
                        label="Máximo de árboles (max_iter)",
                        minimum=50, maximum=600, step=10, value=300,
                        info=f"best_iter se deriva automáticamente del mínimo de la curva "
                             f"Val RMSE global (promedio de los {len(COLUMNAS_Y_ESPERADAS)} targets). "
                             f"Batch warm-start: {BATCH_CURVA} árboles/paso.",
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                        info="Elimina años atípicos para entrenar con condiciones normales.",
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Test split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80,
                            info="El (1-ratio)% final = Test Ciego 100% intocable.",
                            scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento Multi-Output", variant="primary", size="lg")
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(value="_Esperando ejecución…_", elem_classes=["logs-box"])

            # ══════════════════════════════════════════
            # PANEL DERECHO — TABS
            # ══════════════════════════════════════════
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value=(
                                "*Tras entrenar verás:*\n"
                                "- Tabla de validación TSS por fold y por contaminante.\n"
                                "- Tabla de métricas en Test Ciego (MAE, RMSE, R² per-target).\n"
                                "- Resumen del esquema X/Y detectado automáticamente [SDM]."
                            )
                        )

                    with gr.Tab("📈 Real vs Predicho"):
                        gr.Markdown(
                            "Grid **2×3** de series temporales — Real vs Predicho "
                            "por contaminante en el **Test Ciego**."
                        )
                        fig_pred_output = gr.Plot(
                            label="Real vs Predicho (6 contaminantes · Test Ciego)"
                        )

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Evolución del RMSE** por iteración acumulada.\n\n"
                            "- **Líneas finas** coloreadas = curva individual por contaminante.\n"
                            "- **Línea blanca gruesa** = μ global (promedio de los 6 targets).\n"
                            "- **Línea vertical dorada** = mejor iteración global.\n"
                            "- Generada con **warm_start individual por target** en el "
                            "último fold TSS (mayor ventana de entrenamiento).\n\n"
                            "> Sin early stopping aleatorio: el split de validación es siempre "
                            "**cronológico** (ventana posterior al train)."
                        )
                        fig_lc_output = gr.Plot(label="Curva de Aprendizaje Multi-Output")

                    with gr.Tab("🎯 R² por Target"):
                        gr.Markdown(
                            "**R² en Test Ciego** para cada uno de los 6 contaminantes.\n\n"
                            "Línea punteada dorada = umbral recomendado R² = 0.70."
                        )
                        fig_r2_output = gr.Plot(label="R² por Contaminante — Test Ciego")

                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown(
                            "**Importancia por permutación** — Δ R² promediado entre los "
                            "6 targets, calculado sobre el **Test Ciego**.\n\n"
                            "Un valor negativo indica que permutar esa feature empeora el modelo "
                            "(es decir, la feature es informativa para la predicción conjunta)."
                        )
                        fig_fi_output = gr.Plot(label="Permutation Importance (avg R² · Test Ciego)")

                    with gr.Tab("📊 Dependencias"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables X e Y. "
                            "Las columnas Y aparecen resaltadas con el color de su contaminante."
                        )
                        fig_hm_output = gr.Plot(label="Correlación de Pearson — X × Y")

                    with gr.Tab("🔮 Consulta AQI"):
                        gr.Markdown(
                            "Ingresa valores meteorológicos y de lags para obtener la "
                            "**estimación simultánea de los 6 contaminantes** con el modelo "
                            "en memoria. Los valores por defecto son las medias del dataset."
                        )
                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada (X)")
                            # 30 slots de input (se muestran dinámicamente los que correspondan)
                            slider_componentes = [
                                gr.Number(
                                    label=f"feature_{i}", value=0.0,
                                    visible=False, interactive=True,
                                )
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir los 6 Contaminantes", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v11.0 · MultiOutputRegressor(HistGB) · SDM · TSS · Warm-Start Honesto
        </div>
        """)

        # ── Funciones auxiliares de UI ─────────────────────────────────────────

        def _refresh_pred_ui():
            """Recarga la UI de consulta con los features y estadísticas del modelo en sesión."""
            feat_cols   = SESION.get("feat_cols", [])
            feat_stats  = SESION.get("feat_stats", {})
            target_cols = SESION.get("target_cols", [])
            parroquia   = SESION.get("parroquia", "")
            sesion_msg  = (
                f"✅ Modelo listo: **{', '.join(target_cols)}** · *{parroquia}* · "
                f"{len(feat_cols)} features."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col, value=round(st["mean"], 4),
                        visible=True, interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            import json
            feat_cols = SESION.get("feat_cols", [])
            d = {
                feat_cols[i]: float(vals[i])
                for i in range(min(len(feat_cols), len(vals)))
                if vals[i] is not None
            }
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────
        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_dropdown, csv_upload, nombre_modelo_input,
                max_iter_slider, plot_days_slider,
                train_ratio_slider, excluir_pandemia_chk,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_fi_output,
                fig_hm_output,   fig_lc_output,   fig_r2_output,
                estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 15.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 68)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v11.0")
    print("  🗂️  Multi-Output · SDM · HistGB Penalizado · TSS · Warm-Start")
    print("═" * 68)
    print(f"  Hardware     : {GPU_MSG}")
    print(f"  Targets Y    : {COLUMNAS_Y_ESPERADAS}")
    print(f"  l2_reg       : {HGB_L2_REG}  |  min_samples_leaf : {HGB_MIN_SAMPLES_LEAF}")
    print(f"  max_leaf     : {HGB_MAX_LEAF_NODES}  |  batch warm-start : {BATCH_CURVA}")
    print(f"  n_jobs_multi : {N_JOBS_MULTI}  (MultiOutputRegressor)")
    print("  Puerto       : automático (server_port=None)")
    print("═" * 68)

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}")
        print("🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  El túnel público falló: {tunnel_err}")
        print("🔄  Relanzando sin túnel — acceso por IP directa:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
<>:1298: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_9167/3050022174.py:1298: SyntaxWarning: invalid escape sequence '\_'
  """
/tmp/ipykernel_9167/3050022174.py:1298: SyntaxWarning: invalid escape sequence '\_'
  """
/tmp/ipykernel_9167/3050022174.py:1298: SyntaxWarning: invalid escape sequence


════════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v11.0
  🗂️  Multi-Output · SDM · HistGB Penalizado · TSS · Warm-Start
════════════════════════════════════════════════════════════════════
  Hardware     : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  Targets Y    : ['PM25', 'PM10', 'O3', 'CO', 'NO2', 'SO2']
  l2_reg       : 5.0  |  min_samples_leaf : 50
  max_leaf     : 31  |  batch warm-start : 20
  n_jobs_multi : 6  (MultiOutputRegressor)
  Puerto       : automático (server_port=None)
════════════════════════════════════════════════════════════════════


version 2 con sakilearn

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Autor   : MLOps Engineer / Arquitecto de Datos
  Versión : 11.1  (Bugfix NotFittedError · Smart Schema · Confirmación UI)
  ─── CAMBIOS v11.1 ──────────────────────────────────────────────────────────
  [FIX-A]  NotFittedError en entrenar_tss_multioutput
             · Causa raíz: cuando mask_tr.sum() < 10 (ej. PM10 todo-NaN en
               un fold), el estimador[i] nunca hace .fit() pero sigue en la
               lista y luego se llama .predict() sobre él → NotFittedError.
             · Solución: lista fitted_flags[i] booleana por target.
               El estimador sólo se guarda en estimadores_ok[i] si el fit()
               completó sin excepción (patrón atómico try/except).
               Antes de predict(), check_is_fitted() + guardia fitted_flags[i].
             · Debug: si check_is_fitted falla, imprime tipo y estado del obj.

  [FIX-B]  Falsos positivos en advertencias de similitud X↔Y
             · "hora_cos" contenía "co" → advertencia incorrecta.
             · Nueva regla de clasificación en 3 niveles:
               NIVEL 1 — Prefijo exacto + sufijo ("PM25_lag_1h") → X legítimo,
                         se propone automáticamente, requiere confirmación UI.
               NIVEL 2 — Match en medio del nombre ("hora_cos" / "CO") →
                         silencioso: es un falso positivo geométrico.
               NIVEL 3 — Coincidencia exacta sin sufijo → SchemaDetectionError
                         (sigue igual: data leakage real).

  [NEW-4]  Confirmación de Esquema en la UI (gr.State + gr.Group)
             · Después de "Detectar Esquema", si hay features del Nivel 1,
               se muestra un panel de confirmación con tabla interactiva.
             · El usuario puede ver y verificar qué lags/derivados se asignan
               a X antes de entrenar.
             · El entrenamiento sólo puede iniciarse tras confirmar el esquema
               o si no hay ambigüedades que confirmar.
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted   # [FIX-A]

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo":      None,
    "scaler":      None,
    "feat_cols":   [],
    "target_cols": [],
    "parroquia":   "",
    "feat_stats":  {},
    # Estado del esquema confirmado
    "esquema_confirmado": False,
    "x_cols_propuestos":  [],
    "y_cols_propuestos":  [],
    "lags_detectados":    [],   # [(feat, contaminante_base), …]
}

ANOS_PANDEMIA = [2020, 2021]

CONTAMINANTES_Y: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

COLUMNAS_CONTROL: set[str] = {
    "Fecha", "fecha", "Parroquia", "parroquia",
    "Date", "date", "Timestamp", "timestamp",
    "Time", "time", "DateTime", "datetime",
}

FEATURES_X_BASE: list[str] = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

HGB_L2_REG           = 5.0
HGB_MIN_SAMPLES_LEAF = 50
HGB_MAX_LEAF_NODES   = 31
HGB_LEARNING_RATE    = 0.05
BATCH_CURVA          = 20

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"

COLORES_TARGETS = {
    "PM25": "#3B82F6", "PM10": "#F97316",
    "O3":   "#22C55E", "CO":   "#A855F7",
    "NO2":  "#EF4444", "SO2":  "#F59E0B",
}


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada: {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    return False, "No se detectó GPU — CPU nativo (HistGB funciona correctamente)"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


# ──────────────────────────────────────────────────────────────────────────────
# 3.  UTILIDADES DE ARCHIVOS
# ──────────────────────────────────────────────────────────────────────────────

def listar_csvs(directorio: str = ".") -> list[str]:
    csvs = []
    for raiz, _, archivos in os.walk(directorio):
        for f in archivos:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.relpath(os.path.join(raiz, f), directorio))
    csvs.sort()
    return csvs if csvs else ["(No se encontraron archivos CSV)"]


def refrescar_dropdown() -> gr.Dropdown:
    opciones = listar_csvs()
    return gr.Dropdown(choices=opciones, value=opciones[0] if opciones else None)


def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  ★ MÓDULO DE DETECCIÓN DINÁMICA DE ESQUEMA — v2 (FIX-B) ★
# ──────────────────────────────────────────────────────────────────────────────

class SchemaDetectionError(ValueError):
    """Error crítico en la detección de esquema X/Y."""


def _clasificar_similitud(xc: str, yc: str) -> str:
    """
    Clasifica la relación entre una feature xc y un contaminante yc.

    Retorna uno de:
      "prefijo"  — xc comienza con yc y tiene sufijo   → lag/derivado legítimo
                   Ejemplo: "PM25_lag_1h" empieza con "PM25"
      "silencio" — yc aparece en medio de xc           → falso positivo
                   Ejemplo: "hora_cos" contiene "co" pero no empieza con "CO"
      "exacto"   — xc == yc                            → data leakage (nunca
                   debería llegar aquí, pero por seguridad se cubre)
    """
    xc_up = xc.upper()
    yc_up = yc.upper()

    if xc_up == yc_up:
        return "exacto"

    # Prefijo exacto: xc empieza con yc y el siguiente carácter no es letra/dígito
    # que pudiera confundir ("O3_lag" comienza con "O3"; "O3_raw" también)
    if xc_up.startswith(yc_up):
        siguiente = xc_up[len(yc_up):]
        # El separador debe ser '_', '-', '.', o dígito (ej. "NO2lag" sin guión)
        if siguiente and (siguiente[0] in ("_", "-", ".") or siguiente[0].isdigit()):
            return "prefijo"

    # En cualquier otro caso (match en medio del nombre) → falso positivo
    return "silencio"


def detectar_esquema_xy(
    df: pd.DataFrame,
    timestamp_col: str | None,
    log_fn=None,
) -> tuple[list[str], list[str], list[tuple[str, str]]]:
    """
    Detección Dinámica de Esquema X/Y — versión 2.

    Niveles de clasificación de similitud X↔Y
    ------------------------------------------
    NIVEL 1  (tipo "prefijo") — feature comienza con nombre de contaminante
             + tiene sufijo → lag/derivado → se asigna a X automáticamente
             y se reporta para confirmación por el usuario.
             Ejemplo: "PM25_lag_1h", "NO2_lag_24h", "PM25_3h"

    NIVEL 2  (tipo "silencio") — el nombre del contaminante aparece en medio
             del nombre de la feature → falso positivo geométrico → se ignora
             sin ninguna advertencia.
             Ejemplo: "hora_cos" contiene "co", "mes_cos" contiene "co"

    NIVEL 3  (tipo "exacto") → SchemaDetectionError (data leakage real)

    Returns
    -------
    x_cols   : list[str]             — features X confirmadas
    y_cols   : list[str]             — targets Y en orden canónico
    lags_x   : list[tuple[str,str]]  — features del Nivel 1 (para UI de confirmación)
                                       [(feature, contaminante_base), …]
    """
    if log_fn is None:
        log_fn = log.info

    columnas_df = set(df.columns.tolist())
    log_fn("─── Detección Dinámica de Esquema X/Y (v2) ─────────────────────")

    # ── Paso 1: Identificar Y ─────────────────────────────────────────────────
    y_cols_faltantes = [c for c in CONTAMINANTES_Y if c not in columnas_df]
    if y_cols_faltantes:
        raise SchemaDetectionError(
            f"[ESQUEMA] Faltan {len(y_cols_faltantes)} columnas de Y: "
            f"{y_cols_faltantes}\n"
            f"El CSV debe contener: {CONTAMINANTES_Y}"
        )

    y_cols = [c for c in CONTAMINANTES_Y if c in columnas_df]  # orden canónico
    log_fn(f"[ESQUEMA] ✅ Y detectada ({len(y_cols)} targets): {y_cols}")

    # ── Paso 2: Identificar X ─────────────────────────────────────────────────
    excluir_de_x: set[str] = (
        set(y_cols) | COLUMNAS_CONTROL | ({timestamp_col} if timestamp_col else set())
    )
    cols_numericas = set(df.select_dtypes(include=[np.number]).columns.tolist())

    x_cols_base  = [c for c in FEATURES_X_BASE if c in cols_numericas and c not in excluir_de_x]
    x_cols_extra = [
        c for c in df.columns
        if c in cols_numericas and c not in excluir_de_x and c not in set(FEATURES_X_BASE)
    ]
    x_cols = x_cols_base + x_cols_extra

    if not x_cols:
        raise SchemaDetectionError(
            "[ESQUEMA] No se encontraron features numéricas para X. "
            "Verifica que el CSV contenga variables meteorológicas o lags."
        )

    log_fn(
        f"[ESQUEMA] ✅ X detectada ({len(x_cols)} features): "
        f"{x_cols[:8]}{'…' if len(x_cols) > 8 else ''}"
    )

    # ── Paso 3: Regla de Seguridad (Nivel 3 — exacto) ─────────────────────────
    y_set = set(y_cols)
    fuga_exacta = [c for c in x_cols if c in y_set]
    if fuga_exacta:
        raise SchemaDetectionError(
            f"[ESQUEMA] 🚨 VIOLACIÓN: las variables de Y están en X: {fuga_exacta}\n"
            f"Esto causaría data leakage. Revisa el CSV."
        )
    log_fn("[ESQUEMA] ✅ Regla de seguridad (Nivel 3): sin fugas Y→X exactas")

    # ── Paso 4: Clasificación de similitudes (Niveles 1 y 2) ─────────────────
    lags_detectados: list[tuple[str, str]] = []  # Nivel 1 → para UI de confirmación

    for xc in x_cols:
        for yc in y_cols:
            nivel = _clasificar_similitud(xc, yc)
            if nivel == "prefijo":
                lags_detectados.append((xc, yc))
                # No rompe la ejecución: este target ya fue asignado a X correctamente
            # nivel == "silencio" → no hace nada (falso positivo ignorado)
            # nivel == "exacto"   → ya capturado en fuga_exacta arriba

    if lags_detectados:
        log_fn(
            f"[ESQUEMA] ℹ️  {len(lags_detectados)} features de Nivel-1 detectadas "
            f"(lags/derivados con nombre de contaminante como prefijo) → asignadas a X."
        )
        for xc, yc in lags_detectados:
            log_fn(f"           · '{xc}' (prefijo de '{yc}') → X ✔")
        log_fn(
            "[ESQUEMA] Estas features requieren CONFIRMACIÓN en la UI "
            "antes de iniciar el entrenamiento."
        )
    else:
        log_fn("[ESQUEMA] ✅ Sin ambigüedades de similitud — no requiere confirmación.")

    log_fn(
        f"[ESQUEMA] Columnas de control excluidas: "
        f"{sorted(excluir_de_x & columnas_df)}"
    )
    log_fn("─────────────────────────────────────────────────────────────────")

    return x_cols, y_cols, lags_detectados


def ejecutar_deteccion_esquema(ruta: str) -> tuple[
    str,               # mensaje Markdown para la UI
    list[list],        # filas de la tabla de confirmación
    bool,              # hay ambigüedades (True = mostrar panel de confirmación)
    list[str],         # x_cols propuestos
    list[str],         # y_cols propuestos
    list[tuple],       # lags detectados
]:
    """
    Función llamada por el botón "Detectar Esquema".
    Retorna todo lo necesario para renderizar el panel de confirmación.
    """
    if not ruta or ruta.startswith("(No"):
        return "⚠️ Selecciona un archivo válido.", [], False, [], [], []

    try:
        df_head = pd.read_csv(ruta, comment="#", nrows=5, low_memory=False)
        ts = _detectar_timestamp(df_head)

        x_cols, y_cols, lags = detectar_esquema_xy(df_head, ts)

        hay_ambiguedades = len(lags) > 0

        # Filas para gr.Dataframe de confirmación
        filas_tabla = []
        for xc, yc in lags:
            filas_tabla.append([xc, yc, "X (lag/derivado)", "✅ Correcto"])

        # Mensaje resumen
        msg_partes = [
            f"✅ **Esquema detectado**",
            f"**Y** ({len(y_cols)} targets): `{y_cols}`",
            f"**X**: `{len(x_cols)}` features",
            f"**Timestamp**: `{ts}`",
        ]
        if hay_ambiguedades:
            msg_partes.append(
                f"⚠️ **{len(lags)} features de Nivel-1** (lags con nombre de contaminante) "
                f"→ revisa la tabla y confirma antes de entrenar."
            )
        else:
            msg_partes.append("✅ Sin ambigüedades — puedes entrenar directamente.")

        msg = "  |  ".join(msg_partes)

        return msg, filas_tabla, hay_ambiguedades, x_cols, y_cols, lags

    except SchemaDetectionError as e:
        return f"❌ **Error de esquema**: {e}", [], False, [], [], []
    except Exception as e:
        return f"❌ **Error inesperado**: {e}", [], False, [], [], []


# ──────────────────────────────────────────────────────────────────────────────
# 5.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def preprocesar_dataframe(
    df: pd.DataFrame,
    timestamp_col: str,
    excluir_pandemia: bool = True,
    log_fn=None,
) -> pd.DataFrame:
    if log_fn is None:
        log_fn = log.info

    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()

    if excluir_pandemia:
        n_antes = len(df)
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
        n_excl  = n_antes - len(df)
        if n_excl:
            log_fn(f"[PRE] {n_excl:,} registros de {ANOS_PANDEMIA} eliminados.")

    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")

    y_presentes = [c for c in CONTAMINANTES_Y if c in df.columns]
    mask_ok     = df[y_presentes].notna().any(axis=1)
    n_antes     = len(df)
    df = df[mask_ok]
    if n_antes - len(df):
        log_fn(f"[PRE] {n_antes - len(df):,} filas sin ningún contaminante → eliminadas.")

    log_fn(f"[PRE] DataFrame limpio: {len(df):,} filas × {df.shape[1]} columnas")
    return df


def _dividir_cronologico(X, Y, ratio=0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], Y.iloc[:corte], Y.iloc[corte:]


# ──────────────────────────────────────────────────────────────────────────────
# 6.  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def construir_modelo_multioutput(max_iter: int = 300) -> MultiOutputRegressor:
    """
    MultiOutputRegressor(HistGradientBoostingRegressor, n_jobs=-1).

    · Un estimador HGB independiente por contaminante.
    · n_jobs=-1 → los 6 estimadores se entrenan en paralelo.
    · Un único .pkl por parroquia.
    """
    estimador_base = HistGradientBoostingRegressor(
        max_iter          = max_iter,
        max_leaf_nodes    = HGB_MAX_LEAF_NODES,
        min_samples_leaf  = HGB_MIN_SAMPLES_LEAF,
        l2_regularization = HGB_L2_REG,
        learning_rate     = HGB_LEARNING_RATE,
        early_stopping    = False,
        warm_start        = False,
        random_state      = 42,
    )
    return MultiOutputRegressor(estimator=estimador_base, n_jobs=-1)


def _construir_hgb_warmstart(max_iter: int = 20) -> HistGradientBoostingRegressor:
    """Estimador individual con warm_start=True — solo para curvas de aprendizaje."""
    return HistGradientBoostingRegressor(
        max_iter          = max_iter,
        max_leaf_nodes    = HGB_MAX_LEAF_NODES,
        min_samples_leaf  = HGB_MIN_SAMPLES_LEAF,
        l2_regularization = HGB_L2_REG,
        learning_rate     = HGB_LEARNING_RATE,
        early_stopping    = False,
        warm_start        = True,
        random_state      = 42,
    )


# ──────────────────────────────────────────────────────────────────────────────
# 7.  ★ CURVA WARM-START MULTI-OUTPUT — FIX-A (atomicidad + fitted_flags) ★
# ──────────────────────────────────────────────────────────────────────────────

def _curva_warmstart_multioutput(
    X_tr:     np.ndarray,
    Y_tr:     np.ndarray,
    X_val:    np.ndarray,
    Y_val:    np.ndarray,
    y_cols:   list[str],
    max_iter: int = 300,
    batch:    int = BATCH_CURVA,
    log_fn        = None,
) -> tuple[dict, dict, list, list[bool]]:
    """
    Curva de aprendizaje multi-output con warm_start incremental.

    FIX-A — Atomicidad + fitted_flags
    -----------------------------------
    · fitted_flags[i] = True  → el estimador[i] fue entrenado con éxito al
                                 menos una vez y puede llamarse .predict().
    · fitted_flags[i] = False → el target i no tiene datos suficientes en
                                 este fold (mask_tr.sum() < 10).
                                 NO se llama .predict() sobre él.
    · El .fit() se envuelve en try/except: si falla, fitted_flags[i]=False
      y se registra el error sin detener el fold completo.
    · Verificación explícita con check_is_fitted() antes de cada predict().

    Returns
    -------
    train_curves  : dict {col: [rmse, …]}
    val_curves    : dict {col: [rmse, …]}
    estimadores   : list[HGB]   — sólo índices donde fitted_flags[i]=True
                                  tienen un estimador válido
    fitted_flags  : list[bool]  — [True/False] por target
    """
    if log_fn is None:
        log_fn = lambda m: None

    n_targets = len(y_cols)
    estimadores  = [_construir_hgb_warmstart(max_iter=batch) for _ in range(n_targets)]
    fitted_flags = [False] * n_targets   # ← [FIX-A] estado de entrenamiento por target

    train_curves: dict[str, list] = {c: [] for c in y_cols}
    val_curves:   dict[str, list] = {c: [] for c in y_cols}

    for step in range(batch, max_iter + 1, batch):
        for i, (col, est) in enumerate(zip(y_cols, estimadores)):
            mask_tr = ~np.isnan(Y_tr[:, i])
            mask_va = ~np.isnan(Y_val[:, i])

            if mask_tr.sum() < 10:
                # No hay datos suficientes → NaN en curva, fitted_flag permanece False
                train_curves[col].append(np.nan)
                val_curves[col].append(np.nan)
                log_fn(
                    f"    [SKIP] '{col}' fold: solo {mask_tr.sum()} muestras válidas "
                    f"en train → estimador[{i}] no entrenado (fitted_flags[{i}]=False)"
                )
                continue

            # ── FIX-A: Patrón atómico — fit() protegido ──────────────────────
            est.max_iter = step
            try:
                est.fit(X_tr[mask_tr], Y_tr[mask_tr, i])
                fitted_flags[i] = True   # ← sólo se marca True si fit() exitoso
            except Exception as fit_err:
                log_fn(
                    f"    [ERROR-FIT] '{col}' step={step}: {fit_err} "
                    f"| tipo obj: {type(est).__name__} | fitted_flags[{i}]=False"
                )
                train_curves[col].append(np.nan)
                val_curves[col].append(np.nan)
                continue

            # ── FIX-A: Verificación con check_is_fitted antes de predict() ───
            try:
                check_is_fitted(est)
            except Exception as cif_err:
                log_fn(
                    f"    [DEBUG-NotFitted] '{col}' step={step}: {cif_err}\n"
                    f"      tipo={type(est).__name__}  "
                    f"  hasattr(estimators_)={hasattr(est, 'estimators_')}"
                )
                fitted_flags[i] = False
                train_curves[col].append(np.nan)
                val_curves[col].append(np.nan)
                continue

            rmse_tr = float(np.sqrt(mean_squared_error(
                Y_tr[mask_tr, i], est.predict(X_tr[mask_tr])
            )))
            rmse_va = (
                float(np.sqrt(mean_squared_error(
                    Y_val[mask_va, i], est.predict(X_val[mask_va])
                ))) if mask_va.sum() >= 2 else np.nan
            )
            train_curves[col].append(rmse_tr)
            val_curves[col].append(rmse_va)

        # Log de progreso cada 5 batches
        if step % (batch * 5) == 0:
            partes = []
            for c_idx, c in enumerate(y_cols):
                if fitted_flags[c_idx] and train_curves[c]:
                    last_tr = train_curves[c][-1]
                    last_va = val_curves[c][-1]
                    if not np.isnan(last_tr):
                        partes.append(f"{c}: tr={last_tr:.3f} va={last_va:.3f}")
            if partes:
                log_fn(f"    iter {step:4d}/{max_iter} → {' | '.join(partes)}")

    return train_curves, val_curves, estimadores, fitted_flags


# ──────────────────────────────────────────────────────────────────────────────
# 8.  ★ PIPELINE TimeSeriesSplit — FIX-A (guardia fitted_flags en predict) ★
# ──────────────────────────────────────────────────────────────────────────────

def entrenar_tss_multioutput(
    df_train: pd.DataFrame,
    x_cols:   list[str],
    y_cols:   list[str],
    max_iter: int = 300,
    n_splits: int = 10,
    log_fn        = None,
) -> tuple[pd.DataFrame, dict, int]:
    """
    TimeSeriesSplit multi-output con curvas de aprendizaje.

    FIX-A aplicado aquí
    -------------------
    · Recibe fitted_flags de _curva_warmstart_multioutput.
    · La métrica del fold para el target i sólo se calcula si
      fitted_flags[i] is True Y mask_va.sum() >= 2.
    · Si fitted_flags[i] is False → métricas NaN para ese target en ese fold.
    · Esto elimina el NotFittedError del Fold 3 con PM10 todo-NaN.
    """
    if log_fn is None:
        log_fn = log.info

    X_arr = df_train[x_cols].values
    Y_arr = df_train[y_cols].values.astype(float)

    if len(X_arr) < n_splits * 50:
        raise ValueError(
            f"Datos insuficientes: {len(X_arr)} filas para {n_splits} folds "
            f"(mínimo: {n_splits * 50})."
        )

    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_metrics: list[dict] = []
    curvas_dict:  dict = {col: {"train": [], "val": [], "metric": "RMSE"} for col in y_cols}

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
        X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
        Y_tr, Y_va = Y_arr[tr_idx], Y_arr[va_idx]

        scaler  = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_va_sc = scaler.transform(X_va)

        log_fn(f"  ── Fold {fold_idx}/{n_splits} | Train: {len(X_tr):,}  Val: {len(X_va):,} ──")

        # _curva_warmstart_multioutput ahora devuelve fitted_flags
        tr_curves, va_curves, estimadores_fold, fitted_flags = _curva_warmstart_multioutput(
            X_tr_sc, Y_tr, X_va_sc, Y_va,
            y_cols=y_cols, max_iter=max_iter, batch=BATCH_CURVA, log_fn=log_fn,
        )

        for col in y_cols:
            curvas_dict[col]["train"].append(tr_curves[col])
            curvas_dict[col]["val"].append(va_curves[col])

        # ── FIX-A: Métricas del fold con guardia fitted_flags ─────────────────
        fold_row = {"Fold": fold_idx, "N_train": len(X_tr), "N_val": len(X_va)}
        for i, col in enumerate(y_cols):
            mask_va = ~np.isnan(Y_va[:, i])

            # Guardia 1: el estimador debe estar entrenado
            if not fitted_flags[i]:
                log_fn(
                    f"  [SKIP-PREDICT] '{col}' Fold {fold_idx}: "
                    f"fitted_flags[{i}]=False → métricas NaN (sin datos suficientes)"
                )
                fold_row[f"{col}_MAE"]  = np.nan
                fold_row[f"{col}_RMSE"] = np.nan
                fold_row[f"{col}_R2"]   = np.nan
                continue

            # Guardia 2: debe haber muestras de validación válidas
            if mask_va.sum() < 2:
                fold_row[f"{col}_MAE"]  = np.nan
                fold_row[f"{col}_RMSE"] = np.nan
                fold_row[f"{col}_R2"]   = np.nan
                continue

            # Verificación final con check_is_fitted (debug explícito)
            try:
                check_is_fitted(estimadores_fold[i])
            except Exception as cif_err:
                log_fn(
                    f"  [DEBUG-NotFitted] Fold {fold_idx} '{col}': {cif_err}\n"
                    f"    tipo={type(estimadores_fold[i]).__name__}"
                )
                fold_row[f"{col}_MAE"]  = np.nan
                fold_row[f"{col}_RMSE"] = np.nan
                fold_row[f"{col}_R2"]   = np.nan
                continue

            y_pred_col = estimadores_fold[i].predict(X_va_sc[mask_va])
            fold_row[f"{col}_MAE"]  = mean_absolute_error(Y_va[mask_va, i], y_pred_col)
            fold_row[f"{col}_RMSE"] = float(np.sqrt(mean_squared_error(Y_va[mask_va, i], y_pred_col)))
            fold_row[f"{col}_R2"]   = r2_score(Y_va[mask_va, i], y_pred_col)

        fold_metrics.append(fold_row)
        resumen_fold = " | ".join(
            f"{c}: R²={fold_row.get(f'{c}_R2', np.nan):.3f}"
            for c in y_cols
        )
        log_fn(f"  Fold {fold_idx}/{n_splits} → {resumen_fold}")

    # ── Derivar best_iter ─────────────────────────────────────────────────────
    best_iters_por_target = []
    for col in y_cols:
        va_lists = [c for c in curvas_dict[col]["val"] if c and not all(np.isnan(c))]
        if not va_lists:
            continue
        min_len  = min(len(c) for c in va_lists)
        va_arr   = np.array([c[:min_len] for c in va_lists])
        avg_val  = np.nanmean(va_arr, axis=0)
        best_idx = int(np.nanargmin(avg_val))
        best_iters_por_target.append((best_idx + 1) * BATCH_CURVA)

    best_iter = int(np.median(best_iters_por_target)) if best_iters_por_target else max_iter
    log_fn(f"  Best iters por target: {best_iters_por_target} → mediana = {best_iter}")

    return pd.DataFrame(fold_metrics), curvas_dict, best_iter


# ──────────────────────────────────────────────────────────────────────────────
# 9.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_curva_aprendizaje_multi(curvas_dict, parroquia="", batch_size=BATCH_CURVA):
    y_cols = list(curvas_dict.keys())
    n_cols = len(y_cols)
    if n_cols == 0:
        fig, ax = plt.subplots(figsize=(10, 4), facecolor=BG_PLOT)
        ax.text(0.5, 0.5, "Sin datos.", ha="center", va="center",
                color=TEXT_PLOT, transform=ax.transAxes)
        return fig

    cols_grid = 3
    rows_grid = (n_cols + cols_grid - 1) // cols_grid
    fig, axes = plt.subplots(
        rows_grid, cols_grid,
        figsize=(14, 4.5 * rows_grid),
        facecolor=BG_PLOT, squeeze=False,
    )
    fig.patch.set_facecolor(BG_PLOT)

    for ax_idx, col in enumerate(y_cols):
        row, col_idx = divmod(ax_idx, cols_grid)
        ax = axes[row][col_idx]
        ax.set_facecolor(BG_PLOT)
        color = COLORES_TARGETS.get(col, COLOR_REAL)
        data  = curvas_dict.get(col, {})

        tr_lists = data.get("train", [])
        va_lists = data.get("val",   [])

        if not tr_lists:
            ax.text(0.5, 0.5, "Sin datos", ha="center", va="center",
                    color=TEXT_PLOT, transform=ax.transAxes, fontsize=9)
            _aplicar_estilo_ax(ax, col, "", "RMSE")
            continue

        min_len_all = min((len(c) for c in tr_lists + va_lists if c), default=0)
        if min_len_all < 2:
            ax.text(0.5, 0.5, "Historial corto o target sin datos en este fold",
                    ha="center", va="center", color=TEXT_PLOT,
                    transform=ax.transAxes, fontsize=8)
            _aplicar_estilo_ax(ax, col, "", "RMSE")
            continue

        x = (np.arange(min_len_all) + 1) * batch_size

        tr_arr  = np.array([c[:min_len_all] for c in tr_lists])
        tr_mean = np.nanmean(tr_arr, axis=0)
        tr_std  = np.nanstd(tr_arr, axis=0)
        ax.plot(x, tr_mean, color=color, lw=2, label="Train", zorder=5)
        ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color=color)

        if va_lists:
            va_arr  = np.array([c[:min_len_all] for c in va_lists])
            va_mean = np.nanmean(va_arr, axis=0)
            va_std  = np.nanstd(va_arr, axis=0)
            ax.plot(x, va_mean, color=color, lw=2, linestyle="--", label="Val", zorder=5)
            ax.fill_between(x, va_mean - va_std, va_mean + va_std, alpha=0.10, color=color)

            if not np.all(np.isnan(va_mean)):
                best_idx  = int(np.nanargmin(va_mean))
                best_it   = x[best_idx]
                ax.axvline(best_it, color="#F59E0B", lw=1.5, linestyle=":", alpha=0.85)
                ax.scatter([best_it], [va_mean[best_idx]], color="#F59E0B", s=60, zorder=8)

        ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
                  edgecolor=GRID_PLOT, fontsize=8, loc="upper right")
        _aplicar_estilo_ax(ax, col, "Árboles", "RMSE ↓")

    for ax_idx in range(n_cols, rows_grid * cols_grid):
        row, col_idx = divmod(ax_idx, cols_grid)
        axes[row][col_idx].set_visible(False)

    titulo = "Curvas de Aprendizaje Multi-Output — HistGB TSS (K=5)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    fig.suptitle(titulo, color=TEXT_PLOT, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    return fig


def _fig_prediccion_multi(Y_test, Y_pred, y_cols, days, parroquia=""):
    n = len(y_cols)
    fig, axes = plt.subplots(n, 1, figsize=(13, 3.5 * n), facecolor=BG_PLOT, squeeze=False)
    fig.patch.set_facecolor(BG_PLOT)

    for i, col in enumerate(y_cols):
        ax    = axes[i][0]
        ax.set_facecolor(BG_PLOT)
        color = COLORES_TARGETS.get(col, COLOR_REAL)

        idx   = Y_test.index
        corte = idx.max() - pd.Timedelta(days=days)
        mask  = idx >= corte

        real     = Y_test[col].values[mask]
        pred     = Y_pred[:, i][mask]
        idx_plot = idx[mask]

        ax.plot(idx_plot, real, label="Real",     color=color,   lw=1.8, alpha=0.95)
        ax.plot(idx_plot, pred, label="Predicho", color="#CBD5E1", lw=1.5,
                linestyle="--", alpha=0.90)
        ax.fill_between(idx_plot, real, pred, alpha=0.06, color=color)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        plt.setp(ax.get_xticklabels(), rotation=28, ha="right", color=TEXT_PLOT, fontsize=8)
        plt.setp(ax.get_yticklabels(), color=TEXT_PLOT, fontsize=8)
        for sp in ax.spines.values():
            sp.set_edgecolor(GRID_PLOT)
        ax.set_ylabel(col, color=TEXT_PLOT, fontsize=10)
        ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
                  edgecolor=GRID_PLOT, fontsize=9, loc="upper right")
        ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
        ax.set_title(f"{col} — Real vs Predicho", color=TEXT_PLOT,
                     fontsize=10, fontweight="bold", pad=8)

    titulo = f"[{parroquia}]  Test Ciego — últimos {days} días" if parroquia else f"Test Ciego — últimos {days} días"
    fig.suptitle(titulo, color=TEXT_PLOT, fontsize=13, fontweight="bold", y=1.002)
    plt.tight_layout()
    return fig


def _fig_r2_barras(df_metricas_test, parroquia=""):
    fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    y_cols_plot = df_metricas_test["Target"].tolist()
    r2_vals     = df_metricas_test["R2"].tolist()
    colores_bar = [COLORES_TARGETS.get(c, COLOR_POS) for c in y_cols_plot]
    bars = ax.barh(y_cols_plot, r2_vals, color=colores_bar, alpha=0.85, height=0.55)
    ax.axvline(0,   color="#475569", linewidth=0.9)
    ax.axvline(0.7, color="#22C55E", linewidth=1.0, linestyle="--",
               alpha=0.5, label="R²=0.70 (umbral bueno)")
    ax.set_xlim(-0.2, 1.05)
    for bar, val in zip(bars, r2_vals):
        ax.text(
            min(val + 0.02, 1.0), bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", ha="left",
            color=TEXT_PLOT, fontsize=10, fontweight="bold",
        )
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo = f"R² por Contaminante — Test Ciego (20%)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("R²  (↑ mejor)", color=TEXT_PLOT)
    ax.tick_params(colors=TEXT_PLOT)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, y_cols, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.58), max(7, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7, "color": TEXT_PLOT},
                linewidths=0.3, linecolor=GRID_PLOT, square=True, ax=ax,
                cbar_kws={"shrink": 0.6})
    cols_list = list(corr.columns)
    for yc in y_cols:
        if yc in cols_list:
            idx = cols_list.index(yc)
            ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                       edgecolor="#F97316", lw=1.5, clip_on=False))
    titulo_hm = "Correlación de Pearson — Variables de Calidad del Aire"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=7)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=7)
    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_local:        str,
    csv_upload,
    nombre_modelo:    str,
    max_iter:         int,
    plot_days:        int,
    train_ratio:      float,
    excluir_pandemia: bool,
    # Esquema pre-confirmado desde el estado de la UI
    x_cols_estado:    list,
    y_cols_estado:    list,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])

    VACIO = (None, None, None, None)

    try:
        # ── 10.1  Resolver archivo ────────────────────────────────────────────
        if csv_upload is not None:
            ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name
            info(f"Archivo subido: {Path(ruta_csv).name}")
        elif csv_local and not csv_local.startswith("(No"):
            ruta_csv = csv_local
            info(f"Archivo local: {ruta_csv}")
        else:
            err("Sin archivo de entrada.")
            return "### ❌ No se seleccionó ningún archivo.", *VACIO, _estado()

        if not Path(ruta_csv).exists():
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        nombre_modelo = nombre_modelo.strip() or "surrogate_multioutput"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()

        # ── 10.2  Carga raw ───────────────────────────────────────────────────
        df_raw = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

        ts_col = _detectar_timestamp(df_raw)
        if ts_col is None:
            return "### ❌ Sin columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp detectado: '{ts_col}'")

        # ── 10.3  Preprocesamiento ────────────────────────────────────────────
        df_clean = preprocesar_dataframe(df_raw, ts_col, excluir_pandemia, log_fn=info)

        # ── 10.4  Esquema X/Y ────────────────────────────────────────────────
        # Si el usuario ya confirmó desde la UI, usar ese esquema.
        # Si no, ejecutar detección ahora (caso sin ambigüedades).
        if x_cols_estado and y_cols_estado:
            # Filtrar por columnas que existan en df_clean
            x_cols = [c for c in x_cols_estado if c in df_clean.columns]
            y_cols = [c for c in y_cols_estado if c in df_clean.columns]
            info(f"Esquema confirmado por usuario: X={len(x_cols)} features | Y={y_cols}")
        else:
            info("Ejecutando detección de esquema X/Y…")
            try:
                x_cols, y_cols, lags = detectar_esquema_xy(df_clean, None, log_fn=info)
            except SchemaDetectionError as e:
                err(str(e))
                return f"### ❌ Error de Esquema\n```\n{e}\n```", *VACIO, _estado()
            info(f"Esquema detectado: X={len(x_cols)} features | Y={y_cols}")

        n_nulos = int(df_clean[x_cols].isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en X — manejados nativamente por HistGB.")

        # ── 10.5  Split X/Y → división cronológica ────────────────────────────
        X = df_clean[x_cols]
        Y = df_clean[y_cols]
        X_tr, X_te, Y_tr, Y_te = _dividir_cronologico(X, Y, train_ratio)
        info(
            f"Split {int(train_ratio*100)}/{int((1-train_ratio)*100)}: "
            f"Train={len(X_tr):,} | Test ciego={len(X_te):,}"
        )

        # ── 10.6  TimeSeriesSplit K=5 ─────────────────────────────────────────
        info(f"TimeSeriesSplit (K=5) · max_iter={max_iter} · batch={BATCH_CURVA}")
        info(f"l2={HGB_L2_REG} · min_samples_leaf={HGB_MIN_SAMPLES_LEAF} · max_leaf_nodes={HGB_MAX_LEAF_NODES}")

        df_train_all = pd.concat([X_tr, Y_tr], axis=1)
        tss_logs = []
        def tss_log(m): log.info(m); tss_logs.append(m)

        kf_resumen, kf_curvas, best_iter = entrenar_tss_multioutput(
            df_train_all, x_cols, y_cols,
            max_iter=max_iter, n_splits=10, log_fn=tss_log,
        )
        for l in tss_logs:
            logs.append(l)

        info(f"TSS completado — best_iter: {best_iter}")

        # ── 10.7  Modelo final MultiOutputRegressor ────────────────────────────
        final_max_iter = min(best_iter + 2 * BATCH_CURVA, max_iter)
        info(f"Entrenando MultiOutputRegressor final: {final_max_iter} iter × {len(y_cols)} targets")

        scaler_final = StandardScaler()
        X_tr_sc      = scaler_final.fit_transform(X_tr)
        X_te_sc      = scaler_final.transform(X_te)

        modelo_final = construir_modelo_multioutput(max_iter=final_max_iter)
        modelo_final.fit(X_tr_sc, Y_tr.values)
        info("Modelo final entrenado.")

        Y_pred_tr = modelo_final.predict(X_tr_sc)
        Y_pred_te = modelo_final.predict(X_te_sc)

        # ── 10.8  Métricas test ciego ──────────────────────────────────────────
        metricas_test_rows = []
        for i, col in enumerate(y_cols):
            mask = ~np.isnan(Y_te.values[:, i])
            if mask.sum() < 2:
                metricas_test_rows.append({"Target": col, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan})
                continue
            yt = Y_te.values[mask, i]
            yp = Y_pred_te[mask, i]
            metricas_test_rows.append({
                "Target": col,
                "MAE":  float(mean_absolute_error(yt, yp)),
                "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
                "R2":   float(r2_score(yt, yp)),
            })
        df_metricas_test = pd.DataFrame(metricas_test_rows)

        # ── 10.9  Markdown de métricas ────────────────────────────────────────
        hw_label = (
            f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG} "
            f"_(n_jobs=-1 paraleliza los {len(y_cols)} estimadores)_"
        )
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        filas_test = []
        for _, row in df_metricas_test.iterrows():
            r2 = row["R2"]
            ico = ("🟢" if r2 >= 0.7 else ("🟡" if r2 >= 0.5 else "🔴")) if not np.isnan(r2) else "⬜"
            r2_str = f"{r2:.4f}" if not np.isnan(r2) else "sin datos"
            filas_test.append(
                f"| **{row['Target']}** "
                f"| `{row['MAE']:.4f}` "
                f"| `{row['RMSE']:.4f}` "
                f"| {ico} `{r2_str}` |"
            )

        r2_cols_tss = [c for c in kf_resumen.columns if c.endswith("_R2")]
        filas_tss = []
        for _, row in kf_resumen.iterrows():
            r2_vals_fold = [row[c] for c in r2_cols_tss if not np.isnan(row.get(c, np.nan))]
            r2_mean_fold = np.mean(r2_vals_fold) if r2_vals_fold else np.nan
            ico = ("🟢" if r2_mean_fold >= 0.7 else ("🟡" if r2_mean_fold >= 0.5 else "🔴")) if not np.isnan(r2_mean_fold) else "⬜"
            filas_tss.append(
                f"| **{int(row['Fold'])}** | `{int(row['N_train']):,}` | `{int(row['N_val']):,}` "
                f"| {ico} `{r2_mean_fold:.3f}` |"
            )

        metricas_md = f"""
## 📊 Surrogate Model v11.1 — *{parroquia}*

### `MultiOutputRegressor(HistGradientBoostingRegressor)` — 1 PKL/Parroquia

---

### Esquema Confirmado

| Rol | Columnas |
|-----|----------|
| **Y (Targets)** | `{y_cols}` |
| **X (Features)** | `{len(x_cols)} columnas` |

✅ Regla de Seguridad (Y∩X = ∅) verificada.

---

### Validación TSS (K=5) — R² promedio por fold

| Fold | N Train | N Val | R² Prom. |
|:----:|:-------:|:-----:|:--------:|
{"".join(filas_tss)}

---

### Test Ciego — {int((1-train_ratio)*100)}% final

| Contaminante | MAE | RMSE | R² |
|:------------:|:---:|:----:|:--:|
{"".join(filas_test)}

> `max_iter` final = **{final_max_iter}** (best_iter={best_iter} + buffer {2*BATCH_CURVA})

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| PKL | `{nombre_modelo}_{parroquia.replace(' ', '_')}.pkl` |
| Hardware | {hw_label} |
| l2_regularization | `{HGB_L2_REG}` (λ) |
| min_samples_leaf | `{HGB_MIN_SAMPLES_LEAF}` (α-proxy) |
| max_leaf_nodes | `{HGB_MAX_LEAF_NODES}` |
| Pandemia | {pandemia_label} |
"""

        # ── 10.10  Persistencia ───────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo":         modelo_final,
                "scaler":         scaler_final,
                "x_cols":         x_cols,
                "y_cols":         y_cols,
                "parroquia":      parroquia,
                "best_iter":      best_iter,
                "final_max_iter": final_max_iter,
                "hgb_params": {
                    "l2_regularization": HGB_L2_REG,
                    "min_samples_leaf":  HGB_MIN_SAMPLES_LEAF,
                    "max_leaf_nodes":    HGB_MAX_LEAF_NODES,
                    "learning_rate":     HGB_LEARNING_RATE,
                },
                "tss_resumen":    kf_resumen,
                "kf_curvas":      kf_curvas,
                "metricas_test":  df_metricas_test,
            }, fh)
        info(f"PKL guardado: {pkl_path}")

        df_metricas_test.to_csv(OUTPUT_DIR / f"{tag}_metricas_test.csv", index=False)
        kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_tss_resumen.csv",  index=False)

        SESION.update({
            "modelo":      modelo_final,
            "scaler":      scaler_final,
            "feat_cols":   x_cols,
            "target_cols": y_cols,
            "parroquia":   parroquia,
            "feat_stats":  {
                col: {"min": float(X[col].min()), "max": float(X[col].max()), "mean": float(X[col].mean())}
                for col in x_cols
            },
        })
        info("Sesión actualizada → pestaña Predicción lista.")

        # ── 10.11  Figuras ────────────────────────────────────────────────────
        fig_lc   = _fig_curva_aprendizaje_multi(kf_curvas, parroquia, batch_size=BATCH_CURVA)
        fig_pred = _fig_prediccion_multi(Y_te, Y_pred_te, y_cols, plot_days, parroquia)
        fig_r2   = _fig_r2_barras(df_metricas_test, parroquia)
        fig_hm   = _fig_heatmap(pd.concat([X, Y], axis=1), y_cols, parroquia)
        info("Figuras generadas.")

        return metricas_md, fig_pred, fig_r2, fig_hm, fig_lc, _estado()

    except Exception as e:
        err(str(e))
        return (f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado())


# ──────────────────────────────────────────────────────────────────────────────
# 11.  PREDICCIÓN DESDE SESIÓN
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return "### ⚠️ Entrena primero un modelo."
    try:
        import json
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        y_cols    = SESION["target_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        X_sc      = SESION["scaler"].transform(X_input)
        Y_pred    = SESION["modelo"].predict(X_sc)[0]

        filas = []
        for i, col in enumerate(y_cols):
            val = float(Y_pred[i])
            if col == "PM25":
                if val <= 12:    cal = "🟢 Buena"
                elif val <= 35:  cal = "🟡 Moderada"
                elif val <= 55:  cal = "🟠 Insalubre GS"
                elif val <= 150: cal = "🔴 Insalubre"
                else:            cal = "🟣 Muy insalubre"
                filas.append(f"| **{col}** | `{val:.3f} µg/m³` | {cal} |")
            else:
                filas.append(f"| **{col}** | `{val:.3f} µg/m³` | — |")

        return (
            f"## 🔮 Predicción Multi-Output — *{SESION['parroquia']}*\n\n"
            f"| Contaminante | Estimado | Calidad |\n|:--:|:--:|:--:|\n"
            + "\n".join(filas)
            + f"\n\n_Features usadas: {len(feat_cols)}_"
        )
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 12.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:#0F172A; --card:#1E293B; --input:#0D1525; --border:#334155;
    --accent:#6366F1; --accentH:#818CF8; --text:#F1F5F9; --muted:#94A3B8;
    --ok:#22C55E; --warn:#F59E0B; --err:#EF4444; --r:10px;
}
body,.gradio-container{background:var(--bg)!important;color:var(--text)!important;
    font-family:'Inter','Segoe UI',sans-serif!important;}
.gr-group,.gr-box{background:var(--card)!important;border:1px solid var(--border)!important;
    border-radius:var(--r)!important;padding:16px!important;}
input,textarea,select{background:var(--input)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:6px!important;}
label,.gr-label{color:var(--muted)!important;font-size:.8rem!important;font-weight:700!important;
    text-transform:uppercase;letter-spacing:.05em!important;}
button.primary{background:linear-gradient(135deg,var(--accent),#7C3AED)!important;
    color:#fff!important;border:none!important;border-radius:8px!important;
    font-weight:800!important;font-size:1rem!important;padding:12px 28px!important;
    box-shadow:0 4px 20px rgba(99,102,241,.45);transition:all .15s;}
button.primary:hover{transform:translateY(-2px);box-shadow:0 6px 28px rgba(99,102,241,.6);}
button.secondary{background:var(--card)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:8px!important;}
.gr-markdown{color:var(--text)!important;}
.gr-markdown table{border-collapse:collapse;width:100%;}
.gr-markdown th{background:#1E293B;color:var(--accentH);padding:8px 14px;border:1px solid var(--border);}
.gr-markdown td{color:var(--text);padding:7px 14px;border:1px solid var(--border);}
.gr-markdown tr:nth-child(even) td{background:#19253a;}
.gr-file{border:2px dashed var(--accent)!important;border-radius:var(--r)!important;}
.hero{text-align:center;padding:24px 0 6px;}
.hero h1{font-size:2rem;font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text;-webkit-text-fill-color:transparent;}
.hero p{color:var(--muted);font-size:.88rem;}
.gpu-badge{display:inline-block;padding:4px 12px;border-radius:20px;
    font-size:.75rem;font-weight:700;margin-top:4px;}
.gpu-on{background:rgba(34,197,94,.18);color:#4ADE80;border:1px solid #22C55E;}
.gpu-off{background:rgba(99,102,241,.15);color:#A5B4FC;border:1px solid #6366F1;}
.logs-box{background:#0B1527!important;border:1px solid #1D3557!important;
    border-radius:8px;padding:10px 14px;font-family:'JetBrains Mono',monospace;
    font-size:.76rem;color:#7DD3FC;max-height:160px;overflow-y:auto;line-height:1.6;}
.reg-badge{background:rgba(99,102,241,.12);border:1px solid #4338CA;
    border-radius:8px;padding:8px 14px;font-size:.78rem;color:#C7D2FE;margin-top:6px;}
.confirm-box{background:rgba(245,158,11,.08);border:2px solid #F59E0B;
    border-radius:10px;padding:14px;margin-top:8px;}
.confirm-ok{background:rgba(34,197,94,.10);border:1px solid #22C55E;
    border-radius:8px;padding:8px 14px;font-size:.8rem;color:#86EFAC;margin-top:6px;}
"""


# ──────────────────────────────────────────────────────────────────────────────
# 13.  ★ UI con panel de confirmación de esquema (NEW-4) ★
# ──────────────────────────────────────────────────────────────────────────────

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 {GPU_MSG}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU · n_jobs=-1</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo", secondary_hue="sky", neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model v11.1",
    ) as app:

        # ── Estado de esquema confirmado ──────────────────────────────────────
        estado_esquema = gr.State({
            "x_cols": [], "y_cols": [], "lags": [], "confirmado": False
        })

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>MultiOutputRegressor · Detección Dinámica de Esquema · 1 PKL/Parroquia · v11.1</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ── PANEL IZQUIERDO ───────────────────────────────────────────────
            with gr.Column(scale=1, min_width=370):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    with gr.Row():
                        csv_dropdown = gr.Dropdown(
                            label="CSVs en el servidor",
                            choices=listar_csvs(), value=None,
                            interactive=True, scale=5,
                        )
                        btn_refresh = gr.Button("🔄", scale=1, min_width=46, variant="secondary")
                    csv_upload = gr.File(
                        label="O arrastra / sube un CSV externo",
                        file_types=[".csv"], type="filepath",
                    )
                    btn_detectar = gr.Button(
                        "🔍 Detectar Esquema X/Y automáticamente",
                        variant="secondary", size="sm",
                    )
                    info_esquema = gr.Markdown(
                        "_Pulsa **Detectar Esquema** para validar las matrices X e Y._"
                    )

                # ── Panel de confirmación de ambigüedades (NEW-4) ─────────────
                with gr.Group(visible=False) as panel_confirmacion:
                    gr.HTML('<div class="confirm-box">')
                    gr.Markdown(
                        "### ⚠️ Confirmación de Variables Nivel-1\n\n"
                        "Las siguientes features tienen el **nombre de un contaminante "
                        "como prefijo** (ej. `PM25_lag_1h`). El módulo las ha clasificado "
                        "automáticamente como **variables X** (lags/derivados), lo cual "
                        "es correcto para la mayoría de los casos.\n\n"
                        "**Revisa la tabla y confirma** que la asignación es correcta "
                        "antes de entrenar. Si alguna feature no debería estar en X, "
                        "edita tu CSV y vuelve a detectar."
                    )
                    tabla_confirmacion = gr.Dataframe(
                        headers=["Feature (X)", "Contaminante base (Y)", "Asignación", "Estado"],
                        label="Variables de Nivel-1 detectadas",
                        interactive=False,
                        wrap=True,
                    )
                    with gr.Row():
                        btn_confirmar = gr.Button(
                            "✅ Confirmar — estas variables son lags/derivados correctos",
                            variant="primary", size="sm",
                        )
                        btn_rechazar = gr.Button(
                            "❌ Cancelar — necesito revisar el CSV",
                            variant="secondary", size="sm",
                        )
                    gr.HTML('</div>')

                estado_confirmacion = gr.Markdown(
                    visible=False,
                    value="",
                )

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    gr.HTML(
                        f'<div class="reg-badge">'
                        f'<b>MultiOutputRegressor</b>(HistGB) · n_jobs=-1 · '
                        f'λ(L2)={HGB_L2_REG} · α={HGB_MIN_SAMPLES_LEAF} · '
                        f'max_leaf={HGB_MAX_LEAF_NODES}'
                        f'</div>'
                    )
                    nombre_modelo_input = gr.Textbox(
                        label="Nombre base del modelo (.pkl)",
                        value="surrogate_multioutput",
                    )
                    max_iter_slider = gr.Slider(
                        label="Máximo de árboles por estimador",
                        minimum=50, maximum=500, step=10, value=300,
                    )
                    excluir_pandemia_chk = gr.Checkbox(
                        label="🚫 Excluir datos de pandemia (2020–2021)",
                        value=True,
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Holdout split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar", minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button(
                    "🚀  Entrenar MultiOutput — 1 PKL/Parroquia",
                    variant="primary", size="lg",
                )
                gr.Markdown("##### 🖥️ Log de ejecución")
                estado_output = gr.Markdown(value="_Esperando…_", elem_classes=["logs-box"])

            # ── PANEL DERECHO ─────────────────────────────────────────────────
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():
                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Entrena para ver métricas multi-output.*"
                        )
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot()
                    with gr.Tab("📉 Curvas de Aprendizaje"):
                        fig_lc_output = gr.Plot()
                    with gr.Tab("📊 R² por Contaminante"):
                        fig_r2_output = gr.Plot()
                    with gr.Tab("🔥 Correlaciones"):
                        fig_hm_output = gr.Plot()
                    with gr.Tab("🔮 Predicción Instantánea"):
                        sesion_info = gr.Markdown("_⚠️ Entrena primero._")
                        with gr.Group():
                            slider_componentes = [
                                gr.Number(label=f"feature_{i}", value=0,
                                          visible=False, interactive=True)
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir 6 Contaminantes", variant="primary")
                        resultado_pred = gr.Markdown("_Resultado aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v11.1 · FIX-A NotFittedError · Smart Schema · Confirmación UI
        </div>
        """)

        # ── Funciones auxiliares ───────────────────────────────────────────────

        def _get_ruta(csv_local, csv_up):
            if csv_up is not None:
                return csv_up if isinstance(csv_up, str) else csv_up.name
            if csv_local and not csv_local.startswith("(No"):
                return csv_local
            return None

        def _detectar_y_mostrar(csv_local, csv_up, esquema_actual):
            ruta = _get_ruta(csv_local, csv_up)
            if not ruta:
                return (
                    "⚠️ Selecciona un archivo CSV primero.",
                    [], gr.Group(visible=False),
                    gr.Markdown(visible=False, value=""),
                    esquema_actual,
                )

            msg, filas_tabla, hay_ambig, x_cols, y_cols, lags = ejecutar_deteccion_esquema(ruta)

            nuevo_esquema = {
                "x_cols": x_cols,
                "y_cols": y_cols,
                "lags":   [(xc, yc) for xc, yc in lags],
                # Si no hay ambigüedades, confirmar automáticamente
                "confirmado": not hay_ambig and bool(x_cols),
            }

            if not hay_ambig and x_cols:
                msg_conf = gr.Markdown(
                    value='<div class="confirm-ok">✅ Esquema validado automáticamente — sin ambigüedades. Puedes entrenar.</div>',
                    visible=True,
                )
            else:
                msg_conf = gr.Markdown(visible=False, value="")

            return (
                msg,
                filas_tabla,
                gr.Group(visible=hay_ambig),
                msg_conf,
                nuevo_esquema,
            )

        def _confirmar_esquema(esquema_actual):
            """El usuario confirma que los lags detectados son correctos."""
            esquema_actual["confirmado"] = True
            n_lags = len(esquema_actual.get("lags", []))
            return (
                gr.Markdown(
                    value=(
                        f'<div class="confirm-ok">'
                        f'✅ Esquema confirmado por el usuario. '
                        f'{n_lags} feature(s) de Nivel-1 asignadas a X como lags/derivados. '
                        f'Puedes iniciar el entrenamiento.</div>'
                    ),
                    visible=True,
                ),
                gr.Group(visible=False),
                esquema_actual,
            )

        def _rechazar_esquema(esquema_actual):
            """El usuario cancela — resetea el estado."""
            nuevo = {"x_cols": [], "y_cols": [], "lags": [], "confirmado": False}
            return (
                gr.Markdown(
                    value='<div style="color:#EF4444">❌ Confirmación cancelada. Revisa el CSV y vuelve a detectar el esquema.</div>',
                    visible=True,
                ),
                gr.Group(visible=False),
                nuevo,
            )

        def _entrenar_wrapper(
            csv_local, csv_up, nombre_modelo, max_iter, plot_days,
            train_ratio, excluir_pandemia, esquema,
        ):
            if not esquema.get("confirmado", False):
                vacio = (None, None, None, None)
                return (
                    "### ⚠️ Detecta y confirma el esquema X/Y antes de entrenar.",
                    *vacio,
                    "⚠️ Esquema no confirmado.",
                )
            ruta = _get_ruta(csv_local, csv_up)
            if not ruta:
                vacio = (None, None, None, None)
                return "### ❌ Sin archivo.", *vacio, "Sin archivo."

            return entrenar(
                ruta, None, nombre_modelo, max_iter, plot_days, train_ratio,
                excluir_pandemia,
                esquema.get("x_cols", []),
                esquema.get("y_cols", []),
            )

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            y_cols     = SESION.get("target_cols", [])
            parroquia  = SESION.get("parroquia", "")
            sesion_msg = (
                f"✅ *{parroquia}* · {len(feat_cols)} features · Predice: `{y_cols}`"
                if feat_cols else "_⚠️ Entrena primero._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"mean": 0})
                    updates.append(gr.Number(label=col, value=round(st["mean"], 3),
                                             visible=True, interactive=True))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            import json
            feat_cols = SESION.get("feat_cols", [])
            d = {feat_cols[i]: float(vals[i])
                 for i in range(min(len(feat_cols), len(vals)))
                 if vals[i] is not None}
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────

        btn_refresh.click(fn=refrescar_dropdown, inputs=[], outputs=[csv_dropdown])

        btn_detectar.click(
            fn=_detectar_y_mostrar,
            inputs=[csv_dropdown, csv_upload, estado_esquema],
            outputs=[info_esquema, tabla_confirmacion,
                     panel_confirmacion, estado_confirmacion, estado_esquema],
        )

        btn_confirmar.click(
            fn=_confirmar_esquema,
            inputs=[estado_esquema],
            outputs=[estado_confirmacion, panel_confirmacion, estado_esquema],
        )

        btn_rechazar.click(
            fn=_rechazar_esquema,
            inputs=[estado_esquema],
            outputs=[estado_confirmacion, panel_confirmacion, estado_esquema],
        )

        btn_train.click(
            fn=_entrenar_wrapper,
            inputs=[
                csv_dropdown, csv_upload, nombre_modelo_input,
                max_iter_slider, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, estado_esquema,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_r2_output,
                fig_hm_output, fig_lc_output, estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 14.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback():
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local: http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 70)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v11.1")
    print("  🔧  FIX-A: NotFittedError (fitted_flags + check_is_fitted)")
    print("  🔧  FIX-B: Smart Schema (3 niveles de similitud, sin falsos positivos)")
    print("  ✨  NEW-4: Panel de Confirmación de Esquema en la UI")
    print("═" * 70)
    print(f"  Hardware    : {GPU_MSG}")
    print(f"  Arquitectura: MultiOutputRegressor(HistGB, n_jobs=-1)")
    print(f"  Y targets   : {CONTAMINANTES_Y}")
    print(f"  l2_reg      : {HGB_L2_REG}  |  min_samples_leaf: {HGB_MIN_SAMPLES_LEAF}")
    print(f"  max_leaf    : {HGB_MAX_LEAF_NODES}  |  batch warm-start: {BATCH_CURVA}")
    print("═" * 70)

    try:
        app.launch(
            server_name="0.0.0.0", server_port=None,
            share=True, max_threads=40,
            debug=True, show_error=True,
            prevent_thread_lock=False, quiet=False,
        )
    except OSError:
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40)
    except Exception:
        _imprimir_ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40)